# Short notebook 1: hydropower commitment prediction
**Group:** 167  
**Authors:** Hallvard Grundt — 115526; Raviksan Navaratnarajah — 109485; Bøye Tvenge Einang — 139410.

**Selected submission #3; historical public ROC-AUC: 0.99657.**

## Reproduction instructions
Set `RAW_DATA` to the supplied project folder and run all cells in a fresh kernel. The notebook reads the ten supplied files and rebuilds all features and models offline. It creates temporary Python modules and checkpoints; no saved predictions, model weights or repository checkout are required.

Use Python 3.11.15, the pinned requirements, at most four CPU threads, 32 GB RAM and **one NVIDIA Tesla T4 (16 GB)**. The GPU model groups run sequentially. The original models used separate Python environments, so the combined environment still needs a complete rerun. CPU XGBoost training changes the recipe.

**Verification status:** An earlier saved-component replay matched the historical CSV. This does not verify retraining from raw inputs. Neither complete notebook has been run from raw inputs on the required GPU. Total runtime and peak RAM remain unmeasured. The final cell writes a receipt, checks the schema and resource limits, and requires an exact historical CSV hash match.

## 1. Raw inputs and a fresh working directory


In [ ]:
from pathlib import Path
import os, sys, json, shutil, time, hashlib, tempfile, gc, resource
# Set this to the supplied sintef_project directory (contains data/, mapping and template).
RAW_DATA = Path("sintef_project").resolve()
THREADS = 4
# The historical XGBoost members used one Tesla T4 at a time; CPU retraining is NOT equivalent.
XGB_DEVICE = "cuda:0"
for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[variable] = str(THREADS)
assert RAW_DATA.is_dir(), f"Set RAW_DATA to the supplied project directory: {RAW_DATA}"
WORK = Path(tempfile.mkdtemp(prefix="tdt4173_reproduction_", dir=Path.cwd()))
PKG = WORK / "sintef_project"
PKG.mkdir()
(PKG / "__init__.py").write_text("")
STARTED = time.monotonic()
RAW_FILES = [
    "data/kernel/Historical_day_ahead_price_2015_2025.csv",
    "data/kernel/Historical_inflow_1958_2025.csv",
    "data/kernel/Historical_volume_2015_2024.csv",
    "data/kernel/Synthetic_water_value_2015_2024.csv",
    "data/kernel/Unit_commitment_decisions.csv",
    "data/extended/Constraint_min_flow.csv", "data/extended/Constraint_min_volume.csv",
    "data/extended/Tokke_Vinje_topology.yaml", "prediction_mapping.csv", "sample_submission.csv",
]
raw_hashes = {}
def file_sha256(path):
    with open(path, "rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

def peak_memory_bytes():
    peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return int(peak if sys.platform == "darwin" else peak*1024)

for relative in RAW_FILES:
    source = RAW_DATA / relative
    assert source.is_file(), source
    target = PKG / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(source, target)
    raw_hashes[relative] = file_sha256(source)
(WORK / "raw_hashes.json").write_text(json.dumps(raw_hashes, indent=2))
sys.path.insert(0, str(WORK))
# Prevent accidental reuse of modules from a preceding notebook in the same kernel.
for name in list(sys.modules):
    if name == "sintef_project" or name.startswith("sintef_project."):
        del sys.modules[name]
print("Fresh offline working directory:", WORK)


In [ ]:
STARTUP_SEEDS = [4173, 1]
HISTORICAL_SHA = 'f7b0b52fc54e93e1db6171fdf85a7955486c3439897911291eef98c70a4f716a'
OUTPUT_NAME = "submission_0.99657.csv"
CATALOGUE_NUMBER = 3
USE_LEGACY = False


## 2. Self-contained feature and hydraulic solver code
The following cells store readable implementation code in the fresh working directory. These are generated intermediate files, not dependencies distributed alongside the notebook.

### Features

In [ ]:
module_source = r'''
"""Case-hour features using only the documented hydropower optimizer inputs.

Every case has 168 rows. Cases are ordered as the training CSV followed by the
prediction mapping; hour increases fastest. Initial storage is read at case
start and terminal water value at case start plus seven days. Neither future
storage nor any commitment labels enter the features.
"""

from pathlib import Path
import re

import numpy as np
import pandas as pd


HORIZON = 168


def _read_timeseries(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path)
    frame.index = pd.to_datetime(frame.pop("date"), utc=True)
    if frame.index.has_duplicates:
        raise ValueError(f"Duplicate timestamps in {path}")
    return frame.astype(np.float32)


def _lookup(frame: pd.DataFrame, dates: pd.DatetimeIndex, name: str) -> np.ndarray:
    positions = frame.index.get_indexer(dates)
    if (positions < 0).any():
        raise ValueError(f"Missing {name} timestamps: {dates[positions < 0][:5]}")
    values = frame.to_numpy()[positions]
    if not np.isfinite(values).all():
        raise ValueError(f"Nonfinite values in {name}")
    return values


def _capacities(path: Path) -> dict[str, float]:
    """Read simple reservoir max_vol entries without requiring a YAML package."""
    result = {}
    current = None
    in_reservoir = False
    for line in path.read_text().splitlines():
        if line == "  reservoir:":
            in_reservoir = True
            continue
        if in_reservoir and re.match(r"^  \S", line):
            break
        if in_reservoir and re.match(r"^    \S", line):
            current = line.strip().removesuffix(":")
        if in_reservoir and line.startswith("      max_vol:"):
            result[current] = float(line.split(":", 1)[1])
    if len(result) != 17:
        raise ValueError("Expected 17 reservoir capacities in topology")
    return result


def build_features(root: Path) -> tuple[pd.DataFrame, dict[str, np.ndarray], pd.DataFrame, pd.DataFrame, dict]:
    """Return features, per-unit labels, case frames, and reproducibility metadata.

    ``root`` may be the repository root or its ``sintef_project`` directory.
    X contains all training cases then all test cases, 168 rows per case.
    Each label vector covers training cases only in the same case-hour order.
    The case frames contain Run No and UTC starttime. The metadata includes
    ordered units, submission label columns, and detailed feature definitions.
    """
    root = Path(root)
    if (root / "sintef_project").is_dir():
        root = root / "sintef_project"
    kernel = root / "data" / "kernel"
    extended = root / "data" / "extended"
    training = pd.read_csv(kernel / "Unit_commitment_decisions.csv")
    train = training[["Run No", "starttime"]].copy()
    mapping = pd.read_csv(root / "prediction_mapping.csv")
    train["starttime"] = pd.to_datetime(train["starttime"], utc=True)
    mapping["starttime"] = pd.to_datetime(mapping["starttime"], utc=True)
    cases = pd.concat([train, mapping], ignore_index=True)
    if cases["Run No"].duplicated().any() or cases["starttime"].duplicated().any():
        raise ValueError("Case IDs and start dates must be unique")
    label_columns = pd.read_csv(root / "sample_submission.csv", nrows=0).columns.tolist()[1:]
    if training.columns.tolist()[3:] != label_columns:
        raise ValueError("Training and submission labels have different order")
    units = [column.removeprefix("result_committed_").removesuffix("_t0")
             for column in label_columns if column.endswith("_t0")]
    labels = {}
    for unit in units:
        columns = [f"result_committed_{unit}_t{hour}" for hour in range(HORIZON)]
        y = training[columns].to_numpy(dtype=np.uint8)
        if not np.isin(training[columns].to_numpy(), [0, 1]).all():
            raise ValueError(f"Invalid binary labels for {unit}")
        labels[unit] = y.reshape(-1)
    del training

    starts = pd.DatetimeIndex(cases["starttime"])
    terminal_dates = starts + pd.Timedelta(days=7)
    n_cases = len(cases)
    hours = np.arange(HORIZON)
    timestamps = pd.DatetimeIndex(
        np.repeat(starts.as_unit("ns").asi8, HORIZON)
        + np.tile(hours.astype(np.int64) * 3_600_000_000_000, n_cases), tz="UTC"
    )
    price_frame = _read_timeseries(kernel / "Historical_day_ahead_price_2015_2025.csv")
    price = _lookup(price_frame, timestamps, "price").reshape(n_cases, HORIZON)
    volume_frame = _read_timeseries(kernel / "Historical_volume_2015_2024.csv")
    water_frame = _read_timeseries(kernel / "Synthetic_water_value_2015_2024.csv")
    volume = _lookup(volume_frame, starts, "initial volume")
    water = _lookup(water_frame, terminal_dates, "terminal water value")
    inflow_frame = _read_timeseries(kernel / "Historical_inflow_1958_2025.csv")
    inflow = _lookup(inflow_frame, timestamps, "inflow").reshape(n_cases, HORIZON, -1)
    capacities = _capacities(extended / "Tokke_Vinje_topology.yaml")
    features = {}

    def add(name: str, value: np.ndarray) -> None:
        array = np.asarray(value, dtype=np.float32)
        if array.shape == (n_cases,):
            array = np.repeat(array, HORIZON)
        elif array.shape == (HORIZON,):
            array = np.tile(array, n_cases)
        else:
            array = array.reshape(-1)
        if array.size != n_cases * HORIZON:
            raise ValueError(f"Wrong feature size: {name}: {array.shape}")
        features[name] = array

    add("horizon_hour", hours)
    add("horizon_day", hours // 24)
    add("hour_of_day", hours % 24)
    add("hour_sin", np.sin(2 * np.pi * hours / 24))
    add("hour_cos", np.cos(2 * np.pi * hours / 24))
    add("weekday", timestamps.dayofweek.to_numpy())
    add("month", timestamps.month.to_numpy())
    add("day_of_year", timestamps.dayofyear.to_numpy())
    add("season_sin", np.sin(2 * np.pi * timestamps.dayofyear.to_numpy() / 365.25))
    add("season_cos", np.cos(2 * np.pi * timestamps.dayofyear.to_numpy() / 365.25))
    add("price", price)
    for offset in (-24, -12, -6, -3, -1, 1, 3, 6, 12, 24):
        add(f"price_offset_{offset:+d}", price[:, np.clip(hours + offset, 0, HORIZON - 1)])
    price_mean = price.mean(axis=1)
    price_std = price.std(axis=1)
    add("price_week_mean", price_mean)
    add("price_week_std", price_std)
    add("price_week_min", price.min(axis=1))
    add("price_week_max", price.max(axis=1))
    for quantile in (0.1, 0.25, 0.5, 0.75, 0.9):
        add(f"price_week_q{int(100 * quantile)}", np.quantile(price, quantile, axis=1))
    add("price_minus_week_mean", price - price_mean[:, None])
    add("price_week_zscore", (price - price_mean[:, None]) / (price_std[:, None] + 1))
    add("price_week_rank", pd.DataFrame(price).rank(axis=1, pct=True).to_numpy())
    daily_price = price.reshape(n_cases, 7, 24)
    for stat in ("mean", "min", "max", "std"):
        add(f"price_day_{stat}", np.repeat(getattr(daily_price, stat)(axis=2), 24, axis=1))
    add("price_remaining_mean", np.cumsum(price[:, ::-1], axis=1)[:, ::-1] / (HORIZON - hours))
    add("price_remaining_max", np.maximum.accumulate(price[:, ::-1], axis=1)[:, ::-1])

    for idx, reservoir in enumerate(volume_frame.columns):
        initial = volume[:, idx]
        capacity = capacities[reservoir]
        water_idx = water_frame.columns.get_loc(reservoir)
        terminal = water[:, water_idx]
        add(f"volume_{reservoir}", initial)
        add(f"fill_{reservoir}", initial / capacity)
        add(f"water_value_{reservoir}", terminal)
        add(f"price_minus_water_{reservoir}", price - terminal[:, None])
        add(f"price_over_water_{reservoir}", price / (terminal[:, None] + 1))
        inflow_idx = inflow_frame.columns.get_loc(reservoir)
        inflow_volume = inflow[:, :, inflow_idx].sum(axis=1) * 0.0036
        add(f"headroom_after_inflow_{reservoir}", (capacity - initial - inflow_volume) / capacity)
    for idx, reservoir in enumerate(inflow_frame.columns):
        flow = inflow[:, :, idx]
        add(f"inflow_{reservoir}", flow)
        add(f"inflow_week_mean_{reservoir}", flow.mean(axis=1))
        add(f"inflow_elapsed_mm3_{reservoir}", np.cumsum(flow, axis=1) * 0.0036)
        add(f"inflow_remaining_mm3_{reservoir}", np.cumsum(flow[:, ::-1], axis=1)[:, ::-1] * 0.0036)
    for filename, prefix in (("Constraint_min_volume.csv", "min_volume"),
                             ("Constraint_min_flow.csv", "min_flow")):
        frame = _read_timeseries(extended / filename)
        values = _lookup(frame, timestamps, prefix).reshape(n_cases, HORIZON, -1)
        for idx, name in enumerate(frame.columns):
            add(f"{prefix}_{name}", values[:, :, idx])
            add(f"{prefix}_week_max_{name}", values[:, :, idx].max(axis=1))
    X = pd.DataFrame(features, copy=False)
    for name in X:
        if not np.isfinite(X[name].to_numpy()).all():
            raise ValueError(f"Nonfinite feature {name}")
    metadata = {
        "units": units,
        "label_columns": label_columns,
        "feature_names": list(X.columns),
        "n_train_cases": len(train),
        "n_test_cases": len(mapping),
        "horizon_hours": HORIZON,
        "row_order": "training CSV cases, then prediction mapping cases; hour increases fastest",
        "feature_spec": {
            "volume": "Initial storage at case start; fill = volume/max_vol from topology",
            "water_value": "Terminal water value at start+7 days; margins price-water; ratio price/(water+1)",
            "inflow": "Hourly flow, 168h mean, elapsed incl. current and remaining incl. current volumes; m3/s times 0.0036 -> million m3 per hour",
            "headroom": "(max_vol-initial_volume-sum(horizon_inflow)*0.0036)/max_vol; no generation/outflow approximation",
            "price": "Current, +/-1/3/6/12/24h clamped inside case, horizon stats/rank, daily stats, remaining mean/max; zscore divisor std+1",
            "constraints": "Current hourly and maximum across 168-hour horizon",
            "calendar": "UTC; annual sine/cosine period 365.25; daily sine/cosine period24",
            "excluded": "Commitment labels, calculation time, future historical reservoir volumes, calendar year, row ID",
        },
    }
    return X, labels, train, mapping, metadata
'''
(PKG / "features.py").write_text(module_source.lstrip("\n"), encoding="utf-8")


### Physics

In [ ]:
module_source = r'''
"""Optional hydraulic and switching-cost features, without optimization labels.

The supplied EUR/MWh water values are treated as local energy values, consistent
with SHOP's energy_value_input semantics. They are not subtracted across plants.
These transparent approximations are features, not a replacement SHOP solver.
"""

from pathlib import Path

import numpy as np
import pandas as pd
import yaml


# Reservoirs connected without an intervening generator. The equivalent-value
# groups also match the exact shared water-value series in the supplied dataset.
POOLS = {
    "Hogga": ["Bandak"],
    "Tokke": ["Vinjevatn"],
    "Lio": ["Byrtevatn"],
    "Byrte": ["Botnedalsvatn"],
    "Vinje": ["Vaamarvatn", "Totak", "Venemo", "Hyljelihyl"],
    "Haukeli": ["Vatjern", "Langeidvatn"],
    "Songa": ["Songavatn", "Bitdalsvatn"],
    "Kjela": ["Foersvatn", "Bordalsvatn", "Langesae"],
    "Vesle Kjela": ["Kjelavatn", "Staavatn"],
}


def switching_advantage(profit: np.ndarray, start_cost: float, stop_cost: float) -> np.ndarray:
    """Return optimal objective(on at t) minus objective(off at t), for each t.

    Max-sum dynamic programming includes hourly operating profit and asymmetric
    switch costs. Initial and final states are free because no initial commitment
    is supplied. Water budget, cascade coupling and ramp constraints are omitted.
    Both future and past profits belong to the same known scheduling horizon.
    """
    profit = np.asarray(profit, dtype=np.float64)
    if profit.ndim != 2 or not np.isfinite(profit).all():
        raise ValueError("Profit must be a finite case-by-hour matrix")
    n_cases, horizon = profit.shape
    forward0 = np.zeros((n_cases, horizon), dtype=np.float64)
    forward1 = np.empty_like(forward0)
    forward1[:, 0] = profit[:, 0]
    for hour in range(1, horizon):
        forward0[:, hour] = np.maximum(forward0[:, hour - 1], forward1[:, hour - 1] - stop_cost)
        forward1[:, hour] = profit[:, hour] + np.maximum(forward1[:, hour - 1], forward0[:, hour - 1] - start_cost)
    backward0 = np.zeros(n_cases, dtype=np.float64)
    backward1 = np.zeros(n_cases, dtype=np.float64)
    advantage = np.empty_like(forward0)
    for hour in range(horizon - 1, -1, -1):
        advantage[:, hour] = forward1[:, hour] + backward1 - forward0[:, hour] - backward0
        previous0 = np.maximum(backward0, profit[:, hour] + backward1 - start_cost)
        previous1 = np.maximum(backward0 - stop_cost, profit[:, hour] + backward1)
        backward0, backward1 = previous0, previous1
    return advantage.astype(np.float32)


def add_physics_features(x: pd.DataFrame, root: Path) -> tuple[pd.DataFrame, dict]:
    """Enrich complete case-hour features with physical and economic proxies.

    Returns a new frame and a JSON-serializable specification. Does not mutate x.
    Expects the 168-hour case order emitted by features.build_features.
    """
    root = Path(root)
    if (root / "sintef_project").is_dir():
        root = root / "sintef_project"
    topology_path = root / "data" / "extended" / "Tokke_Vinje_topology.yaml"
    topology = yaml.safe_load(topology_path.read_text())
    model = topology["model"]
    if len(x) % 168 or not np.array_equal(x["horizon_hour"].to_numpy(), np.tile(np.arange(168), len(x) // 168)):
        raise ValueError("Expected complete 168-hour cases in case-major order")
    n_cases = len(x) // 168
    at_start = x.iloc[::168]
    price = x["price"].to_numpy().reshape(n_cases, 168)
    remaining_hours = 168 - np.arange(168)
    additions = {}
    plant_specs = {}
    incoming = {}
    downstream = {}
    for edge in topology["connections"]:
        incoming.setdefault(edge["to"], []).append(edge["from"])
        if edge["from_type"] == "plant" and edge["to_type"] == "reservoir":
            downstream[edge["from"]] = edge["to"]

    def upstream_reservoirs(plant: str) -> list[str]:
        visited = set()
        queue = list(incoming.get(plant, []))
        while queue:
            node = queue.pop()
            if node in visited:
                continue
            visited.add(node)
            queue.extend(incoming.get(node, []))
        return sorted(visited.intersection(model["reservoir"]))

    def head(reservoir: str) -> np.ndarray:
        curve = model["reservoir"][reservoir]["vol_head"]
        return np.interp(at_start[f"volume_{reservoir}"], curve["x"], curve["y"])

    def add(name: str, values: np.ndarray) -> None:
        values = np.asarray(values, dtype=np.float32)
        if values.shape == (n_cases,):
            values = np.repeat(values, 168)
        values = values.reshape(-1)
        if len(values) != len(x) or not np.isfinite(values).all():
            raise ValueError(f"Invalid physical feature: {name}")
        additions[name] = values

    for plant, pool in POOLS.items():
        upstream = pool[0]
        plant_data = model["plant"][plant]
        units = [u for u in model["generator"] if u.rsplit("_G", 1)[0] == plant]
        representative = model["generator"][units[0]]
        max_power = sum(model["generator"][u]["p_max"] for u in units)
        outlet = np.full(n_cases, plant_data["outlet_line"], dtype=np.float64)
        if plant in downstream:
            outlet = np.maximum(outlet, head(downstream[plant]))
        upstream_head = head(upstream)
        if plant == "Songa":
            upstream_head = (head("Songavatn") + head("Bitdalsvatn")) / 2
        gross_head = np.maximum(upstream_head - outlet, 1)
        curves = sorted(representative["turb_eff_curves"], key=lambda curve: curve["ref"])
        peak_turbine_eff = np.interp(gross_head, [c["ref"] for c in curves], [max(c["y"]) / 100 for c in curves])
        generator_eff = max(representative["gen_eff_curve"]["y"]) / 100
        # 1000 kg/m3 *9.81 m/s2 *1e6 m3 /3.6e9 J/MWh = 2.725.
        energy_equivalent = 2.725 * gross_head * peak_turbine_eff * generator_eff
        flow_capacity = max_power / (energy_equivalent * 0.0036)
        local_water = at_start[f"water_value_{upstream}"].to_numpy()
        margin = price - local_water[:, None]
        capacity = sum(model["reservoir"][r]["max_vol"] for r in pool)
        initial = sum(at_start[f"volume_{r}"].to_numpy() for r in pool)
        minimum = sum(at_start[f"min_volume_week_max_{r}"].to_numpy() if f"min_volume_week_max_{r}" in x else np.zeros(n_cases) for r in pool)
        remaining_inflow = sum(x[f"inflow_remaining_mm3_{r}"].to_numpy().reshape(n_cases, 168) for r in pool)
        available = np.maximum(initial - minimum, 0)[:, None] + remaining_inflow
        catchment = upstream_reservoirs(plant)
        catchment_flow = sum(x[f"inflow_{r}"].to_numpy().reshape(n_cases, 168) for r in catchment)
        add(f"physics_head_{plant}", gross_head)
        add(f"physics_pool_fill_{plant}", initial / capacity)
        add(f"physics_supply_fraction_{plant}", available * energy_equivalent[:, None] / (remaining_hours * max_power))
        add(f"physics_catchment_inflow_fraction_{plant}", catchment_flow / flow_capacity[:, None])
        weekly_inflow = remaining_inflow[:, 0]
        add(f"physics_dispatch_to_avoid_spill_{plant}", np.maximum(initial + weekly_inflow - capacity, 0) * energy_equivalent / (168 * max_power))
        add(f"physics_profitable_remaining_fraction_{plant}", np.cumsum((margin > 0)[:, ::-1], axis=1)[:, ::-1] / remaining_hours)
        for operating_point, power in (("minimum", representative["p_min"]), ("maximum", representative["p_max"])):
            advantage = switching_advantage(margin * power, representative["startcost_const"], representative["stopcost_const"])
            add(f"physics_switch_margin_{operating_point}_{plant}", advantage / power)
        plant_specs[plant] = {
            "pool": pool, "head_reference_reservoir": upstream,
            "downstream_reservoir": downstream.get(plant), "catchment": catchment,
            "representative_unit": units[0], "plant_pmax_mw": max_power,
            "switch_pmin_mw": representative["p_min"], "switch_pmax_mw": representative["p_max"],
            "start_cost": representative["startcost_const"], "stop_cost": representative["stopcost_const"],
        }
    spec = {
        "sources": ["sintef_project/Dataset_definitions_and_explanation.pdf",
                    "https://docs.shop.sintef.energy/examples/water_values/water_values.html",
                    "https://docs.shop.sintef.energy/examples/water_values/individual_water_values.html"],
        "water_value_semantics": "Local PDF states EUR/MWh. Interpret as local energy_value_input consistent with official SHOP docs; case-generation code unavailable. No upstream-downstream EUR/MWh subtraction.",
        "head": "Initial volume interpolated on YAML vol_head; upstream minus max(plant outlet_line, downstream reservoir head). Songa averages its two supply heads.",
        "energy_equivalent": "Approximation 2.725*head*peak turbine efficiency interpolated across head reference curves*peak generator efficiency, MWh/Mm3. Ignores hydraulic losses and joint-unit dispatch.",
        "supply_fraction": "(max(initial pool volume-max weekly min-volume,0)+remaining natural pool inflow)*energy_equivalent/(remaining_hours*plant_pmax); omits upstream generation and past dispatch",
        "catchment_inflow_fraction": "Sum natural hourly inflow over directed upstream reservoir graph divided by approximate full-plant discharge capacity; omits storage buffering and travel times",
        "spill": "max(initial pool volume+weekly natural pool inflow-pool max volume,0)*energy_equivalent/(168*plant_pmax)",
        "switching": "Max-sum binary-state DP at representative pmin and pmax; profit=(price-local terminal WV)*power; transition costs from YAML; free start/end states; feature=(best objective forcing on-best forcing off)/power. No water budgets or cascade constraints.",
        "columns": list(additions), "plants": plant_specs,
    }
    return pd.concat([x, pd.DataFrame(additions, index=x.index, copy=False)], axis=1, copy=False), spec
'''
(PKG / "physics.py").write_text(module_source.lstrip("\n"), encoding="utf-8")


### Context Features

In [ ]:
module_source = r'''
"""Case-local price context adapted from the user's canonical feature design.

This transform runs on complete 168-hour cases before monetary normalization
and before training-row sampling. It reads no labels or external files.
"""
from pathlib import Path

import numpy as np
import pandas as pd

HORIZON = 168


def add_context_features(x: pd.DataFrame, root: Path) -> tuple[pd.DataFrame, dict]:
    """Add remaining-horizon midpoint rank and neighboring price means.

    Each previous/next mean excludes the current hour and uses at most 6, 12,
    or 24 available case-local hours. At an empty boundary, use the current
    price. The rank includes the current hour, gives ties their midpoint,
    and is named without ``price_`` so monetary normalization preserves it.
    ``root`` is accepted for the shared transform interface but is not read.
    """
    del root
    if "price" not in x or "horizon_hour" not in x:
        raise ValueError("Context requires raw price and horizon_hour before normalization")
    if not len(x) or len(x) % HORIZON:
        raise ValueError("Context requires complete 168-hour cases before row sampling")
    hours = np.arange(HORIZON)
    if not np.array_equal(x["horizon_hour"].to_numpy(), np.tile(hours, len(x) // HORIZON)):
        raise ValueError("Expected case-major hours 0 through 167")
    price = x["price"].to_numpy(dtype=np.float64).reshape(-1, HORIZON)
    if not np.isfinite(price).all():
        raise ValueError("Prices must be finite")

    remaining_rank = np.empty(price.shape, dtype=np.float32)
    for hour in range(HORIZON):
        suffix = price[:, hour:]
        current = price[:, hour:hour + 1]
        remaining_rank[:, hour] = (
            (suffix < current).sum(axis=1) + .5 * (suffix == current).sum(axis=1)
        ) / (HORIZON - hour)
    additions = {"context_rank_remaining": remaining_rank.ravel()}
    prefix = np.concatenate([np.zeros_like(price[:, :1]), price.cumsum(axis=1)], axis=1)
    for window in (6, 12, 24):
        left = np.maximum(hours - window, 0)
        right = np.minimum(hours + window + 1, HORIZON)
        for direction, sums, counts in (
            ("previous", prefix[:, hours] - prefix[:, left], hours - left),
            ("next", prefix[:, right] - prefix[:, hours + 1], right - hours - 1),
        ):
            mean = np.divide(sums, counts[None, :], out=price.copy(), where=counts[None, :] > 0)
            additions[f"price_{direction}_mean_{window}h"] = mean.astype(np.float32).ravel()
            additions[f"price_minus_{direction}_mean_{window}h"] = (price - mean).astype(np.float32).ravel()
    overlap = set(x.columns).intersection(additions)
    if overlap:
        raise ValueError(f"Context features already exist: {sorted(overlap)}")
    spec = {
        "source": "User main repository src/temporal_features.py and src/price_features.py; independent adapter preserving formulas",
        "rank": "context_rank_remaining=(number strictly lower + 0.5*number equal)/(168-hour), over current hour through167; flat suffix and final hour rank0.5",
        "rank_normalization": "Dimensionless context_rank_remaining intentionally lacks price_ prefix and must not be divided by monetary scale",
        "neighbor_mean": "For6/12/24h, previous=[max(0,h-window),h), next=(h,min(167,h+window)]; excludecurrent; divide by actual available count; empty neighborhoods use current price",
        "contrasts": "Current price minus each neighbor mean; all12 price_* context columns are monetary and should be normalized with the rest of the case prices",
        "fit_scope": "No fitting, labels, filesystem reads or cross-case aggregation; compute complete168-hour features before selecting training rows",
        "added_columns": list(additions),
    }
    result = pd.concat([x, pd.DataFrame(additions, index=x.index, copy=False)], axis=1)
    return result, spec
'''
(PKG / "context_features.py").write_text(module_source.lstrip("\n"), encoding="utf-8")


### Normalized

In [ ]:
module_source = r'''
"""Optional case-relative economics for transfer across electricity-price regimes.

Replace absolute monetary predictors with case-normalized versions rather than
duplicating them. Fixed startup costs mean the complete scheduling problem is
not invariant to monetary scaling; inverse price scale retains that information.
"""

from pathlib import Path

import numpy as np
import pandas as pd


def add_normalized_features(x: pd.DataFrame, root: Path) -> tuple[pd.DataFrame, dict]:
    """Normalize monetary columns by each case's mean absolute hourly price + 1.

    ``root`` is accepted for the common feature-transform interface; no files,
    targets, or fitted statistics are read. All 168 prices are documented inputs
    to a case's optimization. The input frame remains unchanged.
    """
    del root
    if len(x) == 0 or len(x) % 168:
        raise ValueError("Expected nonempty complete 168-hour cases")
    expected_hours = np.tile(np.arange(168), len(x) // 168)
    if not np.array_equal(x["horizon_hour"].to_numpy(), expected_hours):
        raise ValueError("Expected case-major ordering with hours 0 through 167")
    price = x["price"].to_numpy(dtype=np.float64).reshape(-1, 168)
    if not np.isfinite(price).all():
        raise ValueError("Price must be finite")
    case_scale = np.abs(price).mean(axis=1) + 1.0
    scale = np.repeat(case_scale, 168)
    preserved_relative = {"price_week_rank", "price_week_zscore"}
    monetary = []
    for column in x:
        is_relative = column in preserved_relative or column.startswith("price_over_water_")
        is_monetary = column == "price" or column.startswith(("price_", "water_value_", "physics_switch_margin_"))
        if is_monetary and not is_relative:
            monetary.append(column)
    additions = {}
    for column in monetary:
        values = x[column].to_numpy(dtype=np.float64)
        if not np.isfinite(values).all():
            raise ValueError(f"Nonfinite monetary feature: {column}")
        name = f"norm_{column}"
        if name in x:
            raise ValueError(f"Normalized column already exists: {name}")
        additions[name] = (values / scale).astype(np.float32)
    if "monetary_scale_inverse" in x:
        raise ValueError("Monetary normalization has already been applied")
    additions["monetary_scale_inverse"] = (1.0 / scale).astype(np.float32)
    retained = [column for column in x if column not in monetary]
    result = pd.concat([x[retained], pd.DataFrame(additions, index=x.index, copy=False)], axis=1)
    spec = {
        "scale_formula": "s(case) = 1 + mean(abs(price(case,h))) for h=0..167, in EUR/MWh",
        "normalized_formula": "norm_<original> = original / s(case)",
        "inverse_scale": "monetary_scale_inverse = 1/s(case); no raw scale feature",
        "replaced_columns": monetary,
        "normalized_columns": [f"norm_{column}" for column in monetary],
        "retained_columns": retained,
        "preserved_relative_features": [column for column in retained if column in preserved_relative or column.startswith("price_over_water_")],
        "row_order": "Complete 168-hour cases; each scale applies only to its own case",
        "fit_scope": "No fitting; per-case known optimizer inputs only; no label or filesystem access",
        "switching_costs": "Normalize previously computed physics switching margins after their fixed-cost DP. This is not exact economic scale invariance: startup/stop costs stay fixed; inverse scale allows the learner to recover their relative importance.",
        "stabilization": "The +1 EUR/MWh denominator offset prevents near-zero price instability. Existing price_over_water uses water+1 and price_week_zscore uses std+1; retain both as requested relative features, despite nonexact scale invariance.",
        "hypothesis": "Relative prices, water values and economic margins generalize across historical electricity-price scales more easily than unnormalized currency values",
    }
    return result, spec
'''
(PKG / "normalized.py").write_text(module_source.lstrip("\n"), encoding="utf-8")


### Lp Features

In [ ]:
module_source = r'''
"""A documented, approximate hydraulic LP for features, never ground-truth labels.

Nine storage pools, separate turbine/bypass routes, fixed initial-head concave
generation curves, hourly water balances, and terminal water values are used.
This does not reproduce SHOP's iterative nonlinear or integer optimization.
"""

from dataclasses import dataclass
import hashlib
import json
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from scipy.optimize import linprog
from scipy.sparse import coo_matrix
import yaml

from sintef_project.features import _read_timeseries, _lookup
from sintef_project.physics import POOLS


UNIT_ORDER = ["Hogga_G1", "Tokke_G1", "Tokke_G2", "Tokke_G3", "Tokke_G4", "Lio_G1", "Byrte_G1", "Vinje_G1", "Vinje_G2", "Vinje_G3", "Haukeli_G1", "Songa_G1", "Kjela_G1", "Vesle Kjela_G1"]
POOL_ORDER = list(POOLS)
DOWNSTREAM = {"Hogga": None, "Tokke": "Hogga", "Lio": "Hogga", "Byrte": "Lio", "Vinje": "Tokke", "Haukeli": "Tokke", "Songa": "Vinje", "Kjela": "Vinje", "Vesle Kjela": "Kjela"}
# (source pool, destination pool or sea, mandatory-flow column, reaches Vest)
BYPASSES = [("Hogga", None, "b_Bandak", False), ("Tokke", "Hogga", "b_Vinjevatn", True),
            ("Lio", "Hogga", "b_Byrtevatn", True), ("Byrte", "Hogga", None, True),
            ("Vinje", "Tokke", "b_Hyljelihyl", False), ("Vinje", "Hogga", None, True),
            ("Haukeli", "Tokke", None, False), ("Songa", "Vinje", None, False),
            ("Kjela", "Vinje", None, False), ("Vesle Kjela", "Vinje", "b_Kjelavatn", False)]


@dataclass
class Network:
    """Topology needed by the generic linear water-balance model."""

    pools: list[str]
    units: list[str]
    unit_up: np.ndarray
    unit_down: np.ndarray
    bypass_up: np.ndarray
    bypass_down: np.ndarray
    spill_down: np.ndarray
    vest_units: np.ndarray
    vest_bypasses: np.ndarray
    vest_spills: np.ndarray


class HydraulicLP:
    """Reuse sparse constraint structure across independently solved cases."""

    def __init__(self, network: Network, horizon: int = 168, segments: int = 4):
        self.network, self.horizon, self.segments = network, horizon, segments
        r, g, b, t, k = len(network.pools), len(network.units), len(network.bypass_up), horizon, segments
        cursor = 0

        def allocate(shape):
            nonlocal cursor
            result = np.arange(cursor, cursor + np.prod(shape)).reshape(shape)
            cursor += int(np.prod(shape))
            return result

        self.q = allocate((g, k, t))
        self.v = allocate((r, t))
        self.bypass = allocate((b, t))
        self.spill = allocate((r, t))
        self.low_slack = allocate((r, t))
        self.high_slack = allocate((r, t))
        self.bypass_slack = allocate((b, t))
        self.vest_slack = allocate((t,))
        self.n_variables = cursor
        rows, cols, vals = [], [], []

        def eq(row, col, value):
            rows.append(row); cols.append(int(col)); vals.append(value)

        for pool in range(r):
            for hour in range(t):
                row = pool * t + hour
                eq(row, self.v[pool, hour], 1)
                if hour:
                    eq(row, self.v[pool, hour - 1], -1)
                eq(row, self.spill[pool, hour], 1)
                if network.spill_down[pool] >= 0:
                    eq(network.spill_down[pool] * t + hour, self.spill[pool, hour], -1)
        for unit in range(g):
            for segment in range(k):
                for hour in range(t):
                    eq(network.unit_up[unit] * t + hour, self.q[unit, segment, hour], 1)
                    if network.unit_down[unit] >= 0:
                        eq(network.unit_down[unit] * t + hour, self.q[unit, segment, hour], -1)
        for edge in range(b):
            for hour in range(t):
                eq(network.bypass_up[edge] * t + hour, self.bypass[edge, hour], 1)
                if network.bypass_down[edge] >= 0:
                    eq(network.bypass_down[edge] * t + hour, self.bypass[edge, hour], -1)
        self.a_eq = coo_matrix((vals, (rows, cols)), shape=(r * t, cursor)).tocsc()
        rows, cols, vals = [], [], []
        next_row = 0

        def inequality(terms):
            nonlocal next_row
            for col, coefficient in terms:
                rows.append(next_row); cols.append(int(col)); vals.append(coefficient)
            next_row += 1

        for pool in range(r):
            for hour in range(t):
                inequality([(self.v[pool, hour], -1), (self.low_slack[pool, hour], -1)])
        for pool in range(r):
            for hour in range(t):
                inequality([(self.v[pool, hour], 1), (self.high_slack[pool, hour], -1)])
        for edge in range(b):
            for hour in range(t):
                inequality([(self.bypass[edge, hour], -1), (self.bypass_slack[edge, hour], -1)])
        for hour in range(t):
            terms = [(self.vest_slack[hour], -1)]
            terms += [(self.q[unit, segment, hour], -1) for unit in np.flatnonzero(network.vest_units) for segment in range(k)]
            terms += [(self.bypass[edge, hour], -1) for edge in np.flatnonzero(network.vest_bypasses)]
            terms += [(self.spill[pool, hour], -1) for pool in np.flatnonzero(network.vest_spills)]
            inequality(terms)
        self.a_ub = coo_matrix((vals, (rows, cols)), shape=(next_row, cursor)).tocsc()

    def solve(self, case: dict, time_limit: float = 20.0) -> dict:
        """Solve one case, returning physical flows, shadow prices and diagnostics."""
        network, t = self.network, self.horizon
        r, g, b = len(network.pools), len(network.units), len(network.bypass_up)
        scale = 1 + np.abs(case["price"]).mean()
        penalty = case.get("slack_penalty", 1e7 + 100 * np.max(np.abs(case["terminal_value"])))
        energy = np.repeat(case["energy"][:, :, None], t, axis=2)
        # Nonpositive revenue invalidates incremental-segment loading. At negative
        # prices use a single average conversion for all segments, preserving a
        # linear physical production/discharge relation rather than allowing free
        # water disposal through low-efficiency out-of-order segments.
        negative = case["price"] < 0
        average_energy = (case["energy"] * case["widths"]).sum(axis=1) / case["widths"].sum(axis=1)
        energy[:, :, negative] = average_energy[:, None, None]
        c = np.zeros(self.n_variables)
        c[self.q] = -energy * case["price"][None, None, :]
        c[self.v[:, -1]] = -case["terminal_value"]
        c[self.bypass] = case.get("bypass_cost", 10 / 0.0036)
        c[self.spill] = case.get("spill_cost", 1000 / 0.0036)
        for indices in (self.low_slack, self.high_slack, self.bypass_slack, self.vest_slack):
            c[indices] = penalty
        bounds = np.column_stack([np.zeros(self.n_variables), np.full(self.n_variables, np.inf)])
        bounds[self.q.ravel(), 1] = np.repeat(case["widths"][:, :, None], t, axis=2).ravel()
        bounds[self.bypass.ravel(), 1] = case.get("bypass_capacity", 1000 * 0.0036)
        rhs = case["natural"].copy()
        rhs[:, 0] += case["initial"]
        b_ub = np.concatenate([-case["minimum"].ravel(), np.repeat(case["capacity"], t),
                               -case["bypass_minimum"].ravel(), case["vest_natural"] - case["vest_minimum"]])
        options = {"threads": 1, "time_limit": time_limit, "dual_feasibility_tolerance": 1e-7, "primal_feasibility_tolerance": 1e-7}
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message="Unrecognized options detected.*")
            result = linprog(c / scale, A_ub=self.a_ub, b_ub=b_ub, A_eq=self.a_eq,
                             b_eq=rhs.ravel(), bounds=bounds, method="highs", options=options)
        if not result.success:
            raise RuntimeError(f"LP failed status={result.status}: {result.message}")
        solution = result.x
        flows = solution[self.q].sum(axis=1)
        power = (solution[self.q] * energy).sum(axis=1)
        water_shadow = -result.eqlin.marginals.reshape(r, t) * scale
        residual = float(np.max(np.abs(self.a_eq @ solution - rhs.ravel())))
        if residual > 1e-5:
            raise RuntimeError(f"Water balance residual {residual}")
        sea = flows[network.unit_down < 0].sum() + solution[self.bypass][network.bypass_down < 0].sum() + solution[self.spill][network.spill_down < 0].sum()
        total_balance_error = float(solution[self.v][:, -1].sum() + sea - case["initial"].sum() - case["natural"].sum())
        return {"flow": flows, "power": power, "storage": solution[self.v], "bypass": solution[self.bypass],
                "spill": solution[self.spill], "water_shadow": water_shadow, "minimum_slack": solution[self.low_slack],
                "maximum_slack": solution[self.high_slack], "bypass_slack": solution[self.bypass_slack],
                "vest_slack": solution[self.vest_slack], "objective": float(-result.fun * scale),
                "balance_residual": residual, "total_balance_error": total_balance_error,
                "iterations": int(result.nit), "negative_price_hours": int(negative.sum()), "status": int(result.status)}


def unit_curve(generator: dict, plant: dict, gross_head: float, segments: int = 4) -> dict:
    """Build a four-segment concave initial-head approximation for one unit."""
    curves = sorted(generator["turb_eff_curves"], key=lambda curve: curve["ref"])
    q_upper = max(max(curve["x"]) for curve in curves)
    q_grid = np.unique(np.concatenate([np.linspace(0, q_upper, 257), *[np.array(curve["x"]) for curve in curves]]))
    penstock = plant["penstock_loss"][generator["penstock"] - 1]
    loss = sum(float(value) for value in plant["main_loss"]) + float(penstock)
    effective_head = np.maximum(gross_head - loss * q_grid ** 2, 0.01)
    efficiencies = np.stack([np.interp(q_grid, curve["x"], curve["y"]) for curve in curves]) / 100
    turbine_efficiency = np.array([np.interp(h, [curve["ref"] for curve in curves], efficiencies[:, idx]) for idx, h in enumerate(effective_head)])
    p = 0.00981 * q_grid * effective_head * turbine_efficiency
    for _ in range(8):
        p = 0.00981 * q_grid * effective_head * turbine_efficiency * np.interp(p, generator["gen_eff_curve"]["x"], generator["gen_eff_curve"]["y"]) / 100
    maximum_index = int(np.argmax(p))
    q_grid, p = q_grid[:maximum_index + 1], p[:maximum_index + 1]
    if np.max(p) >= generator["p_max"]:
        qmax = float(np.interp(generator["p_max"], p, q_grid))
        mask = q_grid < qmax
        q_grid, p = np.append(q_grid[mask], qmax), np.append(p[mask], generator["p_max"])
    power_qmin = float(np.interp(generator["p_min"], p, q_grid))
    qmin = power_qmin
    for _ in range(8):
        limit_from_curve = float(np.interp(max(gross_head - loss * qmin ** 2, 0.01),
                                          [curve["ref"] for curve in curves],
                                          [min(curve["x"]) for curve in curves]))
        qmin = max(power_qmin, limit_from_curve)
    # Upper concave hull including the origin gives decreasing segment slopes.
    hull = []
    for q, power in zip(q_grid, p):
        while len(hull) >= 2:
            a, previous = hull[-2:]
            if (previous[1] - a[1]) / (previous[0] - a[0]) <= (power - previous[1]) / (q - previous[0]) + 1e-12:
                hull.pop()
            else:
                break
        hull.append((q, power))
    hull = np.array(hull)
    best_q = hull[1, 0]
    if best_q >= hull[-1, 0] - 1e-9:
        knots = np.linspace(0, hull[-1, 0], segments + 1)
    else:
        knots = np.concatenate([[0], np.linspace(best_q, hull[-1, 0], segments)])
    power = np.interp(knots, hull[:, 0], hull[:, 1])
    widths = np.diff(knots) * 0.0036
    energy = np.diff(power) / widths
    if not (widths > 0).all() or not np.isfinite(energy).all() or (np.diff(energy) > 1e-6).any():
        raise ValueError("Invalid concave generation curve")
    return {"widths": widths, "energy": energy, "qmin": qmin * 0.0036, "qmax": knots[-1] * 0.0036,
            "pmax": float(power[-1]), "best_energy": float(energy[0])}


class HydroCaseData:
    """Read only documented inputs and run/date metadata; no commitment labels."""

    def __init__(self, root: Path):
        root = Path(root)
        if (root / "sintef_project").is_dir():
            root = root / "sintef_project"
        self.root = root
        self.topology = yaml.safe_load((root / "data/extended/Tokke_Vinje_topology.yaml").read_text())
        self.model = self.topology["model"]
        train = pd.read_csv(root / "data/kernel/Unit_commitment_decisions.csv", usecols=["Run No", "starttime"])
        self.n_train = len(train)
        self.cases = pd.concat([train, pd.read_csv(root / "prediction_mapping.csv")], ignore_index=True)
        self.cases["starttime"] = pd.to_datetime(self.cases["starttime"], utc=True)
        self.volume = _read_timeseries(root / "data/kernel/Historical_volume_2015_2024.csv")
        self.water = _read_timeseries(root / "data/kernel/Synthetic_water_value_2015_2024.csv")
        self.price = _read_timeseries(root / "data/kernel/Historical_day_ahead_price_2015_2025.csv")
        self.inflow = _read_timeseries(root / "data/kernel/Historical_inflow_1958_2025.csv")
        self.minimum = _read_timeseries(root / "data/extended/Constraint_min_volume.csv")
        self.flow_minimum = _read_timeseries(root / "data/extended/Constraint_min_flow.csv")
        self.network = Network(POOL_ORDER, UNIT_ORDER,
            np.array([POOL_ORDER.index(unit.rsplit("_G", 1)[0]) for unit in UNIT_ORDER]),
            np.array([POOL_ORDER.index(DOWNSTREAM[unit.rsplit("_G", 1)[0]]) if DOWNSTREAM[unit.rsplit("_G", 1)[0]] else -1 for unit in UNIT_ORDER]),
            np.array([POOL_ORDER.index(edge[0]) for edge in BYPASSES]),
            np.array([POOL_ORDER.index(edge[1]) if edge[1] else -1 for edge in BYPASSES]),
            np.array([-1, 0, 0, 0, 0, 1, 4, 4, 4]),
            np.array([unit == "Lio_G1" for unit in UNIT_ORDER]),
            np.array([edge[3] for edge in BYPASSES]),
            np.array([False, True, True, True, True, False, False, False, False]))
        self.plant_downstream_reservoir = {edge["from"]: edge["to"] for edge in self.topology["connections"] if edge["from_type"] == "plant" and edge["to_type"] == "reservoir"}

    def case(self, index: int) -> dict:
        """Assemble a single legal case with units of Mm3, hours, MWh and EUR."""
        start = self.cases.iloc[index].starttime
        times = pd.date_range(start, periods=168, freq="h")
        volumes = self.volume.loc[start]
        water = self.water.loc[start + pd.Timedelta(days=7)]
        price = _lookup(self.price, times, "LP price")[:, 0].astype(np.float64)
        inflow = pd.DataFrame(_lookup(self.inflow, times, "LP inflow"), columns=self.inflow.columns)
        minimum = pd.DataFrame(_lookup(self.minimum, times, "LP min volume"), columns=self.minimum.columns)
        flow_minimum = pd.DataFrame(_lookup(self.flow_minimum, times, "LP min flow"), columns=self.flow_minimum.columns)
        head = {reservoir: float(np.interp(volumes[reservoir], data["vol_head"]["x"], data["vol_head"]["y"])) for reservoir, data in self.model["reservoir"].items()}
        curves = []
        for unit in UNIT_ORDER:
            plant_name = unit.rsplit("_G", 1)[0]
            plant = self.model["plant"][plant_name]
            pool = POOLS[plant_name]
            source_head = np.mean([head[reservoir] for reservoir in pool]) if plant_name == "Songa" else head[pool[0]]
            outlet = max(plant["outlet_line"], head.get(self.plant_downstream_reservoir.get(plant_name), -np.inf))
            curves.append(unit_curve(self.model["generator"][unit], plant, max(source_head - outlet, 1)))
        best_energy = np.array([max(curves[j]["best_energy"] for j, unit in enumerate(UNIT_ORDER) if unit.rsplit("_G", 1)[0] == pool) for pool in POOL_ORDER])
        global_values = {}

        def terminal(pool):
            if pool is None:
                return 0.0
            if pool not in global_values:
                global_values[pool] = float(water[POOLS[pool][0]]) * best_energy[POOL_ORDER.index(pool)] + terminal(DOWNSTREAM[pool])
            return global_values[pool]

        natural = np.stack([inflow[pool].sum(axis=1).to_numpy() * 0.0036 for pool in POOLS.values()])
        vest_natural = inflow["r_Vest_Vassdraget"].to_numpy() * 0.0036
        natural[POOL_ORDER.index("Hogga")] += vest_natural
        return {"price": price, "initial": np.array([sum(volumes[r] for r in pool) for pool in POOLS.values()]),
                "capacity": np.array([sum(self.model["reservoir"][r]["max_vol"] for r in pool) for pool in POOLS.values()]),
                "minimum": np.stack([sum((minimum[r].to_numpy() for r in pool if r in minimum), np.zeros(168)) for pool in POOLS.values()]),
                "natural": natural, "terminal_value": np.array([terminal(pool) for pool in POOL_ORDER]),
                "widths": np.stack([curve["widths"] for curve in curves]), "energy": np.stack([curve["energy"] for curve in curves]),
                "qmin": np.array([curve["qmin"] for curve in curves]), "qmax": np.array([curve["qmax"] for curve in curves]),
                "pmax": np.array([curve["pmax"] for curve in curves]),
                "bypass_minimum": np.stack([flow_minimum[edge[2]].to_numpy() * 0.0036 if edge[2] else np.zeros(168) for edge in BYPASSES]),
                "vest_natural": vest_natural, "vest_minimum": flow_minimum["r_Vest_Vassdraget"].to_numpy() * 0.0036}


def result_features(case: dict, result: dict, network: Network) -> pd.DataFrame:
    """Convert an LP solution to interpretable continuous unit-hour predictors."""
    features = {}
    scale = 1 + np.abs(case["price"]).mean()
    for j, unit in enumerate(network.units):
        upstream, downstream = network.unit_up[j], network.unit_down[j]
        local_shadow = result["water_shadow"][upstream] - (result["water_shadow"][downstream] if downstream >= 0 else 0)
        opportunity_cost = local_shadow / case["energy"][j, 0]
        features[f"lp_qmin_ratio_{unit}"] = result["flow"][j] / max(case["qmin"][j], 1e-8)
        features[f"lp_power_fraction_{unit}"] = result["power"][j] / max(case["pmax"][j], 1e-8)
        features[f"lp_economic_margin_{unit}"] = (case["price"] - opportunity_cost) / scale
    for pool, name in enumerate(network.pools):
        units = network.unit_up == pool
        features[f"lp_plant_power_fraction_{name}"] = result["power"][units].sum(axis=0) / case["pmax"][units].sum()
        features[f"lp_storage_fill_{name}"] = result["storage"][pool] / case["capacity"][pool]
        features[f"lp_storage_min_slack_{name}"] = result["minimum_slack"][pool] / case["capacity"][pool]
        features[f"lp_storage_max_slack_{name}"] = result["maximum_slack"][pool] / case["capacity"][pool]
    frame = pd.DataFrame(features, dtype=np.float32)
    if not np.isfinite(frame.to_numpy()).all():
        raise ValueError("Nonfinite LP features")
    return frame


def math_specification() -> dict:
    """Return the frozen approximation and provenance for cache manifests."""
    return {
        "version": "pool_lp_v1", "pools": POOLS, "units": UNIT_ORDER, "turbine_downstream": DOWNSTREAM, "bypass_routes": BYPASSES,
        "balance": "v[t]-v[t-1]+all_outflows-all_upstream_transfers=natural_inflow[t], Mm3; t0 RHS includes initialvolume. Storage stays nonnegative; no water-creation slack.",
        "objective": "Maximize hourly price*production+sum(global terminalWV*final storage)-bypass/spill costs-softconstraint penalties; divide entire objective by1+mean(abs(price)) for conditioning; rescale duals afterward.",
        "PQ": "Fixed initial head. Interpolate head from initial individual reservoir volume; outlet=max(outletline,downstream head); Songa mean of its2 heads. Unit-alone quadratic main+ownpenstock loss.257-point discharge grid plus efficiency knots; iterative generator-efficiency interpolation; enforce Pmax and physical curve Qmax; upperconcave hull;4segments beginning at best-efficiency point.",
        "negative_prices": "All incremental segments use the same average full-discharge energy conversion at negative-price hours, avoiding out-of-order low-efficiency disposal. This is a deliberate linear approximation.",
        "terminal": "Local CSV EUR/MWh at start+7d times maximum unit best-point MWh/Mm3 in plant plus downstream global terminalvalue recursively. No raw EUR/MWh upstream/downstream subtraction.",
        "constraints": "Hourly pool minimum is the sum of supplied individual minimum volumes; summed pool maximum storage. Separate minimum bypass flows. Vest minimum includes own natural inflow,Lio discharge and designatedbypass/spill routes. All minimum/maximum violations have penalized slack; physical water conservation exact.",
        "penalties": "bypass10/0.0036 EUR/Mm3,spill1000/0.0036 EUR/Mm3, bypassmax1000m3/s; soft slackpenalty=1e7+100*max(abs(globalterminalWV)) EUR/Mm3 perhour. YAMLcurrency treated consistent with priceEUR, exact case-generationcurrencycommands unavailable.",
        "omissions": "No integer commitment,startstop,minruntime,ramping,head trajectory iteration,shared-penstock unit interactions,tunnel flow limits,individual reservoir storage withinpools,convex river flow-cost tables,head-dependent spill curves,or exact SHOPcommands. Pooling is approximate.",
        "dual_sign": "With stated equality orientation and minimization solver, economic water value=-eqlin.marginals*objective_scale. Unit opportunitycost=(lambda_up-lambda_down)/best_energy.",
        "solver": "scipy.optimize.linprog methodhighs,threads1,time_limit20s,primal/dual feasibility1e-7; no randomness or learnedparameters",
        "source_links": ["https://docs.shop.sintef.energy/examples/pq_curves/pq_curves.html", "https://docs.shop.sintef.energy/examples/water_values/water_values.html", "https://docs.shop.sintef.energy/objects/generator/generator.html", "https://docs.scipy.org/doc/scipy/reference/optimize.linprog-highs.html"],
        "interpretation": "Features only. LPdischarge and the0.5Qmin heuristic are not claimed to match unknown competition solver configuration.",
    }


def add_lp_features(x: pd.DataFrame, root: Path, cache_dir: Path | None = None) -> tuple[pd.DataFrame, dict]:
    """Append a complete verified cache in exact training-plus-test case order.

    Every exported LP predictor is dimensionless: q/Qmin, power/Pmax, economic
    margin divided by mean absolute case price+1, storage/capacity, and slack/
    capacity. No raw currency-valued predictor needs further normalization.
    """
    root = Path(root).resolve()
    repository = root.parent if root.name == "sintef_project" else root
    data_root = repository / "sintef_project"
    cache_dir = repository / "artifacts/lp-full-001" if cache_dir is None else Path(cache_dir)
    if not cache_dir.is_absolute():
        cache_dir = repository / cache_dir
    manifest_path = cache_dir / "manifest.json"
    manifest = json.loads(manifest_path.read_text())
    if manifest["status"] != "complete":
        raise ValueError("LP cache is not complete")

    def digest(path: Path) -> str:
        return hashlib.sha256(path.read_bytes()).hexdigest()

    for category in ("data", "source"):
        for relative, expected in manifest[category].items():
            if digest(repository / relative) != expected:
                raise ValueError(f"LP cache {category} checksum mismatch: {relative}")
    features_path = cache_dir / "features.npz"
    if digest(features_path) != manifest["artifacts"]["features.npz"]:
        raise ValueError("LP cache feature artifact checksum mismatch")
    train = pd.read_csv(data_root / "data/kernel/Unit_commitment_decisions.csv", usecols=["Run No", "starttime"])
    mapping = pd.read_csv(data_root / "prediction_mapping.csv")
    cases = pd.concat([train, mapping], ignore_index=True)
    expected_ids = cases["Run No"].to_numpy()
    expected_dates = pd.to_datetime(cases["starttime"], utc=True).astype(str).tolist()
    expected_hours = np.tile(np.arange(168), len(cases))
    if len(x) != len(expected_hours) or not np.array_equal(x["horizon_hour"].to_numpy(), expected_hours):
        raise ValueError("Input features do not contain the complete case-hour order")
    cache = np.load(features_path, allow_pickle=False)
    if not np.array_equal(cache["case_ids"], expected_ids) or manifest["selected_case_dates"] != expected_dates:
        raise ValueError("LP cache case order or dates mismatch")
    columns = cache["columns"].tolist()
    values = cache["features"]
    if columns != manifest["feature_names"] or values.shape != (len(cases), 168, len(columns)):
        raise ValueError("LP cache shape or column order mismatch")
    if cache["units"].tolist() != UNIT_ORDER or not np.isfinite(values).all():
        raise ValueError("Invalid LP cache units or feature values")
    if set(columns).intersection(x.columns):
        raise ValueError("LP features have already been added")
    frame = pd.DataFrame(values.reshape(len(x), len(columns)), columns=columns, index=x.index, copy=False)
    spec = {"cache_directory": str(cache_dir), "manifest_sha256": digest(manifest_path),
            "features_sha256": digest(features_path), "source_verified": manifest["source"], "data_verified": manifest["data"],
            "shape": list(frame.shape), "columns": columns, "mathematics": manifest["math"],
            "monetary_schema": "All78 exported features are dimensionless. lp_economic_margin=(price-local dual opportunity cost)/(1+mean(abs(case price))); retain lp_* during later monetary normalization. Raw water dual EUR/Mm3 values exist only in diagnostic case archives."}
    return pd.concat([x, frame], axis=1), spec
'''
(PKG / "lp_features.py").write_text(module_source.lstrip("\n"), encoding="utf-8")


### Lp Head Update

In [ ]:
module_source = r'''
"""Experimental single hydraulic head update; frozen LP v1/v2 stay unchanged."""

import warnings

import numpy as np
from scipy.optimize import linprog

from sintef_project.lp_features import HydraulicLP, POOL_ORDER, UNIT_ORDER, unit_curve


def single_reservoir_heads(data, index: int, result: dict) -> dict:
    """Estimate hourly mean head from exact single-reservoir LP storage states.

    Interpolate level at beginning/end storage, then average the two levels.
    This is a midpoint approximation, not an assertion of SHOP's discretization.
    No observed post-start volume or pooled-storage allocation is used.
    """
    start = data.cases.iloc[index].starttime
    volume = data.volume.loc[start]
    levels = {}
    initial_levels = {}
    for reservoir, pool in [("Bandak", "Hogga"), ("Vinjevatn", "Tokke")]:
        curve = data.model["reservoir"][reservoir]["vol_head"]
        endpoint = result["storage"][POOL_ORDER.index(pool)]
        boundary = np.r_[float(volume[reservoir]), endpoint]
        level = np.interp(boundary, curve["x"], curve["y"])
        levels[reservoir] = (level[:-1] + level[1:]) / 2
        initial_levels[reservoir] = level[0]
    hogga_outlet = float(data.model["plant"]["Hogga"]["outlet_line"])
    tokke_outlet = float(data.model["plant"]["Tokke"]["outlet_line"])
    return {"Hogga": levels["Bandak"] - hogga_outlet,
            "Tokke": levels["Vinjevatn"] - np.maximum(tokke_outlet, levels["Bandak"]),
            "initial_Hogga": initial_levels["Bandak"] - hogga_outlet,
            "initial_Tokke": initial_levels["Vinjevatn"] - max(tokke_outlet, initial_levels["Bandak"])}


def shared_loss_correction(data, result: dict, plant_name: str) -> dict[str, np.ndarray]:
    """Estimate omitted shared head loss at previous-iteration unit discharges.

    Return additional loss relative to each unit operating alone. Nonnegative
    generation flows share the plant main tunnel and their assigned penstock.
    This estimates a missing mechanism; it does not change a generation curve.
    """
    indices = [i for i, unit in enumerate(UNIT_ORDER) if unit.rsplit("_G", 1)[0] == plant_name]
    q = result["flow"] / .0036
    plant = data.model["plant"][plant_name]
    main_loss = sum(float(value) for value in plant["main_loss"])
    total_q = q[indices].sum(axis=0)
    corrections = {}
    for i in indices:
        unit = UNIT_ORDER[i]
        penstock = data.model["generator"][unit]["penstock"]
        group = [j for j in indices if data.model["generator"][UNIT_ORDER[j]]["penstock"] == penstock]
        group_q = q[group].sum(axis=0)
        coefficient = float(plant["penstock_loss"][penstock - 1])
        corrections[unit] = main_loss * (total_q ** 2 - q[i] ** 2) + coefficient * (group_q ** 2 - q[i] ** 2)
    return corrections


def hourly_segments(values: np.ndarray, units: int, segments: int, horizon: int) -> np.ndarray:
    """Validate/broadcast constant or hourly segment coefficients to G×K×T."""
    values = np.asarray(values, dtype=np.float64)
    if values.shape == (units, segments):
        values = np.broadcast_to(values[:, :, None], (units, segments, horizon)).copy()
    elif values.shape == (units, segments, horizon):
        values = values.copy()
    else:
        raise ValueError(f"Expected {(units, segments)} or {(units, segments, horizon)}, got {values.shape}")
    if not np.isfinite(values).all() or (values <= 0).any():
        raise ValueError("Segment coefficients must be finite and positive")
    return values


def update_case_head(data, index: int, case: dict, previous: dict,
                     plants: tuple[str, ...] = ("Hogga",)) -> tuple[dict, dict]:
    """Rebuild selected unit PQ curves at prior-LP hourly mean heads exactly once."""
    if not set(plants).issubset({"Hogga", "Tokke"}):
        raise ValueError("Only exact single-reservoir Hogga/Tokke head updates are supported")
    heads = single_reservoir_heads(data, index, previous)
    result = dict(case)
    g, k = case["energy"].shape
    horizon = len(case["price"])
    result["energy"] = hourly_segments(case["energy"], g, k, horizon)
    result["widths"] = hourly_segments(case["widths"], g, k, horizon)
    for name in ["qmin", "qmax", "pmax"]:
        result[name] = np.repeat(case[name][:, None], horizon, axis=1)
    for unit_index, unit in enumerate(UNIT_ORDER):
        plant_name = unit.rsplit("_G", 1)[0]
        if plant_name not in plants:
            continue
        for hour in range(horizon):
            curve = unit_curve(data.model["generator"][unit], data.model["plant"][plant_name],
                               float(heads[plant_name][hour]), segments=k)
            result["energy"][unit_index, :, hour] = curve["energy"]
            result["widths"][unit_index, :, hour] = curve["widths"]
            for name in ["qmin", "qmax", "pmax"]:
                result[name][unit_index, hour] = curve[name]
    # Keep the initial-head terminal conversion: it represents the supplied
    # terminal energy value definition, not the iterated production curve.
    return result, heads


class HourlyHeadLP(HydraulicLP):
    """Same sparse hydraulic network with hour-dependent PQ widths and slopes."""

    def solve(self, case: dict, time_limit: float = 20.) -> dict:
        network, t = self.network, self.horizon
        r, g = len(network.pools), len(network.units)
        energy = hourly_segments(case["energy"], g, self.segments, t)
        widths = hourly_segments(case["widths"], g, self.segments, t)
        if (np.diff(energy, axis=1) > 1e-6).any():
            raise ValueError("Production segment slopes must be nonincreasing each hour")
        negative = case["price"] < 0
        average_energy = (energy * widths).sum(axis=1) / widths.sum(axis=1)
        energy[:, :, negative] = average_energy[:, None, negative]
        scale = 1 + np.abs(case["price"]).mean()
        penalty = case.get("slack_penalty", 1e7 + 100 * np.max(np.abs(case["terminal_value"])))
        c = np.zeros(self.n_variables)
        c[self.q] = -energy * case["price"][None, None, :]
        c[self.v[:, -1]] = -case["terminal_value"]
        c[self.bypass] = case.get("bypass_cost", 10 / .0036)
        c[self.spill] = case.get("spill_cost", 1000 / .0036)
        for indices in [self.low_slack, self.high_slack, self.bypass_slack, self.vest_slack]:
            c[indices] = penalty
        bounds = np.column_stack([np.zeros(self.n_variables), np.full(self.n_variables, np.inf)])
        bounds[self.q.ravel(), 1] = widths.ravel()
        bounds[self.bypass.ravel(), 1] = case.get("bypass_capacity", 1000 * .0036)
        rhs = np.asarray(case["natural"], dtype=np.float64).copy()
        rhs[:, 0] += case["initial"]
        b_ub = np.concatenate([-case["minimum"].ravel(), np.repeat(case["capacity"], t),
                               -case["bypass_minimum"].ravel(), case["vest_natural"] - case["vest_minimum"]])
        options = {"threads": 1, "time_limit": time_limit, "dual_feasibility_tolerance": 1e-7, "primal_feasibility_tolerance": 1e-7}
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message="Unrecognized options detected.*")
            optimized = linprog(c / scale, A_eq=self.a_eq, b_eq=rhs.ravel(), A_ub=self.a_ub,
                                b_ub=b_ub, bounds=bounds, method="highs", options=options)
        if not optimized.success:
            raise RuntimeError(f"Hourly-head LP failed: {optimized.message}")
        solution = optimized.x
        flow = solution[self.q].sum(axis=1)
        power = (solution[self.q] * energy).sum(axis=1)
        sea = flow[network.unit_down < 0].sum() + solution[self.bypass][network.bypass_down < 0].sum() + solution[self.spill][network.spill_down < 0].sum()
        residual = float(np.max(np.abs(self.a_eq @ solution - rhs.ravel())))
        if residual > 1e-5:
            raise RuntimeError(f"Water balance residual {residual}")
        return {"flow": flow, "power": power, "storage": solution[self.v], "bypass": solution[self.bypass],
                "spill": solution[self.spill], "water_shadow": -optimized.eqlin.marginals.reshape(r, t) * scale,
                "minimum_slack": solution[self.low_slack], "maximum_slack": solution[self.high_slack],
                "bypass_slack": solution[self.bypass_slack], "vest_slack": solution[self.vest_slack],
                "objective": float(-optimized.fun * scale), "balance_residual": residual,
                "total_balance_error": float(solution[self.v][:, -1].sum() + sea - case["initial"].sum() - case["natural"].sum()),
                "iterations": int(optimized.nit), "negative_price_hours": int(negative.sum()), "status": int(optimized.status),
                "effective_segment_energy": energy}
'''
(PKG / "lp_head_update.py").write_text(module_source.lstrip("\n"), encoding="utf-8")


### Lp Shared Loss Update

In [ ]:
module_source = r'''
"""One frozen-flow update of Tokke/Vinje shared hydraulic losses.

Keep initial reservoir heads and terminal values fixed. Subtract only the
additional loss omitted by unit-alone PQ curves. This is an approximation at
the preceding LP dispatch, not a coupled nonlinear optimization or derivative.
"""

import numpy as np

from sintef_project.lp_features import UNIT_ORDER, unit_curve
from sintef_project.lp_head_update import hourly_segments, shared_loss_correction


def update_case_shared_losses(data, index: int, case: dict, previous: dict,
                              plants: tuple[str, ...] = ("Tokke", "Vinje")) -> tuple[dict, dict]:
    """Rebuild hourly PQ curves with frozen extra shared loss; no head trajectory."""
    if not set(plants).issubset({"Tokke", "Vinje"}):
        raise ValueError("This isolated ablation supports only Tokke and Vinje")
    g, k = case["energy"].shape
    horizon = len(case["price"])
    result = dict(case)
    for key in ["energy", "widths"]:
        result[key] = hourly_segments(case[key], g, k, horizon)
    for key in ["qmin", "qmax", "pmax"]:
        result[key] = np.repeat(case[key][:, None], horizon, axis=1)
    initial = data.volume.loc[data.cases.iloc[index].starttime]

    def level(name):
        curve = data.model["reservoir"][name]["vol_head"]
        return float(np.interp(initial[name], curve["x"], curve["y"]))

    upstream = {"Tokke": "Vinjevatn", "Vinje": "Vaamarvatn"}
    downstream = {"Tokke": "Bandak", "Vinje": "Vinjevatn"}
    details = {}
    for plant_name in plants:
        plant = data.model["plant"][plant_name]
        gross_head = level(upstream[plant_name]) - max(float(plant["outlet_line"]), level(downstream[plant_name]))
        corrections = shared_loss_correction(data, previous, plant_name)
        for unit, correction in corrections.items():
            unit_index = UNIT_ORDER.index(unit)
            if (correction < -1e-8).any():
                raise ValueError("Additional shared loss must be nonnegative")
            adjusted_head = gross_head - correction
            if (adjusted_head <= 0).any():
                raise ValueError("Frozen shared loss exceeds available gross head")
            # Exact repeated heads occur when prior dispatch loads the same unit
            # combinations; reuse identical curves without rounding the heads.
            unique, inverse = np.unique(adjusted_head, return_inverse=True)
            curves = [unit_curve(data.model["generator"][unit], plant, float(head), segments=k) for head in unique]
            for key in ["energy", "widths"]:
                result[key][unit_index] = np.stack([curve[key] for curve in curves])[inverse].T
            for key in ["qmin", "qmax", "pmax"]:
                result[key][unit_index] = np.array([curve[key] for curve in curves])[inverse]
            details[unit] = {"initial_gross_head": gross_head, "additional_loss": correction,
                             "adjusted_gross_head": adjusted_head, "unique_curves": len(unique)}
    return result, details


def precise_conservation(case: dict, result: dict, network) -> float:
    """Check the exact supplied float values using float64 accumulation throughout."""
    sea = result["flow"][network.unit_down < 0].sum(dtype=np.float64)
    sea += result["bypass"][network.bypass_down < 0].sum(dtype=np.float64)
    sea += result["spill"][network.spill_down < 0].sum(dtype=np.float64)
    return float(result["storage"][:, -1].sum(dtype=np.float64) + sea
                 - np.asarray(case["initial"], dtype=np.float64).sum()
                 - np.asarray(case["natural"], dtype=np.float64).sum())
'''
(PKG / "lp_shared_loss_update.py").write_text(module_source.lstrip("\n"), encoding="utf-8")


### Lp Joint Plant

In [ ]:
module_source = r'''
"""Joint Tokke/Vinje finite-grid operating-point convex mixtures.

Each vertex uses simultaneous main-tunnel and penstock losses. Convex mixtures
represent relaxed time sharing, not exact hourly integer commitment or SHOP.
"""

from dataclasses import dataclass
import json
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from scipy.optimize import linprog
from scipy.sparse import coo_matrix, hstack, vstack

from sintef_project.lp_features import HydraulicLP, UNIT_ORDER, math_specification, result_features
from sintef_project.lp_shared_loss_update import precise_conservation


# Predetermined once, before benchmark results. No grid-size selection.
COMMON_LOAD_GRID = np.arange(1, 129, dtype=np.float64) / 128


@dataclass
class JointCurve:
    """Feasible joint vertices and their exact unit allocations."""

    plant: str
    unit_indices: np.ndarray
    total_flow: np.ndarray  # Mm3 per one-hour interval
    total_power: np.ndarray  # MW = MWh per one-hour interval
    unit_flow: np.ndarray  # vertices × local units, Mm3
    unit_power: np.ndarray  # vertices × local units, MW
    unit_on: np.ndarray  # vertices × local units, binary before mixture
    upper_vertex: np.ndarray
    specification: dict


def simultaneous_points(generators: dict, plant: dict, gross_head: float,
                        units: list[str], flow_m3s: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Evaluate grid points with true simultaneous losses and feasibility checks."""
    flow_m3s = np.asarray(flow_m3s, dtype=np.float64)
    if flow_m3s.ndim != 2 or flow_m3s.shape[1] != len(units) or (flow_m3s < 0).any():
        raise ValueError("Expected nonnegative point-by-unit discharges")
    n_points = len(flow_m3s)
    total = flow_m3s.sum(axis=1)
    main_loss = sum(float(value) for value in plant["main_loss"]) * total ** 2
    power = np.zeros_like(flow_m3s)
    head = np.zeros_like(flow_m3s)
    valid = np.ones(n_points, dtype=bool)
    penstocks = np.array([generators[unit]["penstock"] for unit in units])
    for j, unit in enumerate(units):
        generator = generators[unit]
        group_q = flow_m3s[:, penstocks == penstocks[j]].sum(axis=1)
        head[:, j] = gross_head - main_loss - float(plant["penstock_loss"][penstocks[j] - 1]) * group_q ** 2
        curves = sorted(generator["turb_eff_curves"], key=lambda curve: curve["ref"])
        references = np.array([curve["ref"] for curve in curves], dtype=float)
        efficiencies = np.stack([np.interp(flow_m3s[:, j], curve["x"], curve["y"]) for curve in curves]) / 100
        if len(references) == 1:
            turbine = efficiencies[0]
        else:
            lower = np.clip(np.searchsorted(references, head[:, j], side="right") - 1, 0, len(references) - 2)
            weight = np.clip((head[:, j] - references[lower]) / (references[lower + 1] - references[lower]), 0, 1)
            turbine = efficiencies[lower, np.arange(n_points)] * (1 - weight) + efficiencies[lower + 1, np.arange(n_points)] * weight
        hydraulic = .00981 * flow_m3s[:, j] * head[:, j] * turbine
        production = hydraulic.copy()
        for _ in range(8):
            production = hydraulic * np.interp(production, generator["gen_eff_curve"]["x"], generator["gen_eff_curve"]["y"]) / 100
        power[:, j] = production
        qmin = np.interp(head[:, j], references, [min(curve["x"]) for curve in curves])
        qmax = np.interp(head[:, j], references, [max(curve["x"]) for curve in curves])
        active = flow_m3s[:, j] > 1e-12
        valid &= ~active | ((head[:, j] > 0) & (flow_m3s[:, j] >= qmin - 1e-8)
                            & (flow_m3s[:, j] <= qmax + 1e-8)
                            & (production >= float(generator["p_min"]) - 1e-8)
                            & (production <= float(generator["p_max"]) + 1e-8))
    valid &= np.isfinite(power).all(axis=1) & np.isfinite(head).all(axis=1)
    return power, head, valid


def envelope_indices(flow: np.ndarray, power: np.ndarray, upper: bool) -> list[int]:
    """Return a deterministic upper/lower convex boundary of finite points."""
    order = np.lexsort((np.arange(len(flow)), -power if upper else power, flow))
    distinct = []
    for index in order:
        if distinct and abs(flow[index] - flow[distinct[-1]]) <= 1e-12:
            continue
        distinct.append(int(index))
    hull = []
    for index in distinct:
        while len(hull) >= 2:
            a, b = hull[-2:]
            first = (power[b] - power[a]) / (flow[b] - flow[a])
            second = (power[index] - power[b]) / (flow[index] - flow[b])
            remove = first <= second + 1e-10 if upper else first >= second - 1e-10
            if not remove:
                break
            hull.pop()
        hull.append(index)
    return hull


def build_joint_curve(data, index: int, plant_name: str) -> JointCurve:
    """Enumerate every active subset at the fixed128 common-discharge fractions."""
    if plant_name not in ["Tokke", "Vinje"]:
        raise ValueError("Joint ablation is limited to Tokke and Vinje")
    unit_indices = np.array([j for j, unit in enumerate(UNIT_ORDER) if unit.rsplit("_G", 1)[0] == plant_name])
    units = [UNIT_ORDER[j] for j in unit_indices]
    generators = data.model["generator"]
    plant = data.model["plant"][plant_name]
    initial = data.volume.loc[data.cases.iloc[index].starttime]
    upstream = {"Tokke": "Vinjevatn", "Vinje": "Vaamarvatn"}[plant_name]
    downstream = {"Tokke": "Bandak", "Vinje": "Vinjevatn"}[plant_name]

    def level(reservoir):
        curve = data.model["reservoir"][reservoir]["vol_head"]
        return float(np.interp(initial[reservoir], curve["x"], curve["y"]))

    gross_head = level(upstream) - max(float(plant["outlet_line"]), level(downstream))
    reference_qmax = np.array([max(max(curve["x"]) for curve in generators[unit]["turb_eff_curves"]) for unit in units])
    all_q, all_p, masks, fractions = [np.zeros((1, len(units)))], [np.zeros((1, len(units)))], [0], [0.]
    candidate_count = 0
    feasible_by_subset = {}
    for subset in range(1, 2 ** len(units)):
        active = np.array([bool(subset & (1 << j)) for j in range(len(units))])
        q = COMMON_LOAD_GRID[:, None] * reference_qmax[None, :] * active[None, :]
        p, _, feasible = simultaneous_points(generators, plant, gross_head, units, q)
        all_q.append(q[feasible])
        all_p.append(p[feasible])
        masks.extend([subset] * int(feasible.sum()))
        fractions.extend(COMMON_LOAD_GRID[feasible].tolist())
        feasible_by_subset[str(subset)] = int(feasible.sum())
        candidate_count += len(q)
    q, p = np.vstack(all_q), np.vstack(all_p)
    total_q, total_p = q.sum(axis=1), p.sum(axis=1)
    upper = envelope_indices(total_q, total_p, upper=True)
    lower = envelope_indices(total_q, total_p, upper=False)
    # The all-off origin is represented by unused convex-combination mass.
    vertices = sorted(set(upper + lower) - {0}, key=lambda i: (total_q[i], total_p[i], i))
    if not vertices:
        raise ValueError(f"No feasible operating vertices for {plant_name}")
    upper_set = set(upper)
    if (np.interp(total_q, total_q[upper], total_p[upper]) < total_p - 1e-7).any():
        raise ValueError("Upper envelope fails to dominate a candidate point")
    if (np.interp(total_q, total_q[lower], total_p[lower]) > total_p + 1e-7).any():
        raise ValueError("Lower envelope exceeds a candidate point")
    return JointCurve(plant_name, unit_indices, total_q[vertices] * .0036, total_p[vertices],
                      q[vertices] * .0036, p[vertices], (q[vertices] > 1e-12).astype(float),
                      np.array([i in upper_set for i in vertices]),
                      {"units": units, "initial_gross_head_m": gross_head, "reference_qmax_m3s": reference_qmax.tolist(),
                       "common_load_grid": COMMON_LOAD_GRID.tolist(), "candidate_count": candidate_count,
                       "feasible_count_including_off": len(q), "feasible_by_subset": feasible_by_subset,
                       "upper_vertices_including_off": len(upper), "lower_vertices_including_off": len(lower),
                       "retained_nonzero_vertices": len(vertices), "vertex_subsets": [masks[i] for i in vertices],
                       "vertex_load_fractions": [fractions[i] for i in vertices]})


class JointPlantLP(HydraulicLP):
    """Augment the frozen network with convex-mixture columns and exact routing."""

    def solve(self, case: dict, joint_curves: list[JointCurve], time_limit: float = 20.) -> dict:
        network, t = self.network, self.horizon
        g, r = len(network.units), len(network.pools)
        joint_units = np.concatenate([curve.unit_indices for curve in joint_curves])
        if len(np.unique(joint_units)) != len(joint_units) or network.vest_units[joint_units].any():
            raise ValueError("Joint units overlap or require unsupported Vest routing")
        mode_indices, cursor = [], self.n_variables
        rows, cols, values = [], [], []
        mix_rows, mix_cols = [], []
        for plant_index, curve in enumerate(joint_curves):
            size = len(curve.total_flow) * t
            indices = np.arange(cursor, cursor + size).reshape(-1, t)
            mode_indices.append(indices)
            cursor += size
            up = np.unique(network.unit_up[curve.unit_indices])
            down = np.unique(network.unit_down[curve.unit_indices])
            if len(up) != 1 or len(down) != 1:
                raise ValueError("Joint plant units must share upstream/downstream routing")
            for mode in range(len(curve.total_flow)):
                for hour in range(t):
                    column = indices[mode, hour] - self.n_variables
                    rows.append(up[0] * t + hour); cols.append(column); values.append(curve.total_flow[mode])
                    if down[0] >= 0:
                        rows.append(down[0] * t + hour); cols.append(column); values.append(-curve.total_flow[mode])
                    mix_rows.append(plant_index * t + hour); mix_cols.append(indices[mode, hour])
        added = cursor - self.n_variables
        balance_modes = coo_matrix((values, (rows, cols)), shape=(r * t, added)).tocsc()
        a_eq = hstack([self.a_eq, balance_modes], format="csc")
        mix = coo_matrix((np.ones(len(mix_rows)), (mix_rows, mix_cols)), shape=(len(joint_curves) * t, cursor)).tocsc()
        a_ub = vstack([hstack([self.a_ub, coo_matrix((self.a_ub.shape[0], added))]), mix], format="csc")
        energy = np.repeat(case["energy"][:, :, None], t, axis=2)
        negative = case["price"] < 0
        average_energy = (case["energy"] * case["widths"]).sum(axis=1) / case["widths"].sum(axis=1)
        energy[:, :, negative] = average_energy[:, None, None]
        scale = 1 + np.abs(case["price"]).mean()
        penalty = case.get("slack_penalty", 1e7 + 100 * np.max(np.abs(case["terminal_value"])))
        c = np.zeros(cursor)
        c[self.q] = -energy * case["price"][None, None, :]
        c[self.v[:, -1]] = -case["terminal_value"]
        c[self.bypass] = case.get("bypass_cost", 10 / .0036)
        c[self.spill] = case.get("spill_cost", 1000 / .0036)
        for indices in [self.low_slack, self.high_slack, self.bypass_slack, self.vest_slack]:
            c[indices] = penalty
        bounds = np.column_stack([np.zeros(cursor), np.full(cursor, np.inf)])
        bounds[self.q.ravel(), 1] = np.repeat(case["widths"][:, :, None], t, axis=2).ravel()
        bounds[self.q[joint_units].ravel(), 1] = 0
        bounds[self.bypass.ravel(), 1] = case.get("bypass_capacity", 1000 * .0036)
        for curve, indices in zip(joint_curves, mode_indices):
            c[indices] = -curve.total_power[:, None] * case["price"][None, :]
            bounds[indices.ravel(), 1] = 1
        rhs = np.asarray(case["natural"], dtype=np.float64).copy()
        rhs[:, 0] += case["initial"]
        b_ub = np.concatenate([-case["minimum"].ravel(), np.repeat(case["capacity"], t),
                               -case["bypass_minimum"].ravel(), case["vest_natural"] - case["vest_minimum"],
                               np.ones(len(joint_curves) * t)])
        options = {"threads": 1, "time_limit": time_limit, "dual_feasibility_tolerance": 1e-7, "primal_feasibility_tolerance": 1e-7}
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message="Unrecognized options detected.*")
            optimized = linprog(c / scale, A_eq=a_eq, b_eq=rhs.ravel(), A_ub=a_ub,
                                b_ub=b_ub, bounds=bounds, method="highs", options=options)
        if not optimized.success:
            raise RuntimeError(f"Joint-plant LP failed: {optimized.message}")
        solution = optimized.x
        flow = solution[self.q].sum(axis=1)
        power = (solution[self.q] * energy).sum(axis=1)
        on_fraction = np.zeros((g, t))
        output = {}
        max_mass = 0.
        for curve, indices in zip(joint_curves, mode_indices):
            weights = solution[indices]
            local_flow, local_power = curve.unit_flow.T @ weights, curve.unit_power.T @ weights
            flow[curve.unit_indices] += local_flow
            power[curve.unit_indices] += local_power
            on_fraction[curve.unit_indices] = curve.unit_on.T @ weights
            max_mass = max(max_mass, float(weights.sum(axis=0).max()))
            if not np.allclose(local_flow.sum(axis=0), curve.total_flow @ weights, atol=1e-10):
                raise ValueError("Joint unit flow allocations do not sum to plant flow")
            if not np.allclose(local_power.sum(axis=0), curve.total_power @ weights, atol=1e-10):
                raise ValueError("Joint unit power allocations do not sum to plant power")
            output[f"mode_weights_{curve.plant}"] = weights
        old_max = float(solution[self.q[joint_units]].max())
        if max_mass > 1 + 1e-8 or old_max > 1e-10 or on_fraction.max() > 1 + 1e-8:
            raise ValueError("Invalid convex mass or nonzero disabled independent-unit dispatch")
        output.update(flow=flow, power=power, storage=solution[self.v], bypass=solution[self.bypass],
                      spill=solution[self.spill], water_shadow=-optimized.eqlin.marginals.reshape(r, t) * scale,
                      minimum_slack=solution[self.low_slack], maximum_slack=solution[self.high_slack],
                      bypass_slack=solution[self.bypass_slack], vest_slack=solution[self.vest_slack],
                      joint_on_fraction=on_fraction, joint_unit_indices=joint_units, maximum_convex_mass=max_mass,
                      old_joint_q_max=old_max, objective=float(-optimized.fun * scale),
                      balance_residual=float(np.max(np.abs(a_eq @ solution - rhs.ravel()))),
                      iterations=int(optimized.nit), negative_price_hours=int(negative.sum()),
                      n_variables=cursor, status=int(optimized.status))
        output["total_balance_error"] = precise_conservation(case, output, network)
        if output["balance_residual"] > 1e-5 or abs(output["total_balance_error"]) > 1e-8:
            raise ValueError("Joint model water balance failed")
        return output


def result_joint_features(case: dict, result: dict, network) -> pd.DataFrame:
    """Export87 dimensionless predictors with joint on-fraction/count semantics."""
    frame = result_features(case, result, network)
    frame.columns = [name.replace("lp_", "lpjoint_", 1) for name in frame]
    for j in result["joint_unit_indices"]:
        frame[f"lpjoint_on_fraction_{network.units[j]}"] = result["joint_on_fraction"][j]
    for plant in ["Tokke", "Vinje"]:
        indices = [j for j, unit in enumerate(network.units) if unit.rsplit("_G", 1)[0] == plant]
        frame[f"lpjoint_active_unit_count_{plant}"] = result["joint_on_fraction"][indices].sum(axis=0)
    frame = frame.astype(np.float32)
    if len(frame.columns) != 87 or not np.isfinite(frame.to_numpy()).all():
        raise ValueError("Invalid joint-plant feature schema or nonfinite values")
    return frame


def joint_math_specification() -> dict:
    """Return the frozen grid, mathematical formulation and feature assumptions."""
    return {"version": "joint_plant_v1", "base": math_specification(),
            "plants": ["Tokke", "Vinje"], "common_load_grid": COMMON_LOAD_GRID.tolist(),
            "operating_points": "All15/7 nonempty active-unit subsets at fixed fractions1/128..128/128 of each unit's maximum turbine-curve discharge; simultaneous main-tunnel/assigned-penstock flow squared losses; interpolate turbine/head efficiency and8 generator-efficiency iterations. Reject nonpositive head, interpolated turbine-flow bounds, Pmin/Pmax violations. Include off origin.",
            "hull": "Deterministic upper and lower convex hull vertices in plant total discharge/production, retaining each actual point's unit discharge, power and binary active mask. Keep both frontiers for negative-price behavior; no independent incremental segment decomposition of unit allocations.",
            "LP": "Nonnegative lambda(vertex,hour), sum(lambda)<=1 perplant/hour; unused mass=all-off. Route sum(lambda*Qtotal) exactly once along original plant route, maximize price*sum(lambda*Ptotal). Old independent Tokke/Vinje q variables have zero upper bound. All other7 plants, constraints and initial-head terminal conversions unchanged fromv1.",
            "allocation": "Unit flow/power/onfraction are convex-weighted vertex allocations. Verify allocations sum to plant Q/P, lambda mass<=1 and old independent dispatch=0. On fractions are continuous relaxed time-sharing features, not binary labels or calibrated probabilities.",
            "features": "78 v1-comparable features renamed lpjoint_ plus7 joint-unit on fractions and2 plant active-unit counts, all dimensionless. The q/Qmin and economic-margin denominators retain each unit's v1 initial-head solo curve to preserve a comparable coordinate; this is not a true unit marginal production factor for the joint hull.",
            "limitations": "Finite common-load grid restricts feasible operating points; convexification permits time-sharing across discrete subsets within an hour. This is neither exactMIP nor full nonlinearSHOP. No startup costs, iterative reservoir heads, head-dependent spill, individual within-pool storage or command-state reconstruction. Feasible polytope/allocations may be degenerate.",
            "solver": "scipy HiGHS one thread,20s percase, primal/dual tolerance1e-7; no training, seed or tuning"}


def add_lp_joint_features(x: pd.DataFrame, root: Path,
                          cache_dir: Path | None = None) -> tuple[pd.DataFrame, dict]:
    """Append a complete strict-verified joint cache, replacing v1 in an ablation."""
    from sintef_project.lp_features import add_lp_features

    root = Path(root).resolve()
    repository = root.parent if root.name == "sintef_project" else root
    cache_dir = repository / "artifacts/lp-joint-full-001" if cache_dir is None else Path(cache_dir)
    if not cache_dir.is_absolute():
        cache_dir = repository / cache_dir
    manifest = json.loads((cache_dir / "manifest.json").read_text())
    if manifest["math"]["version"] != "joint_plant_v1":
        raise ValueError("Expected a joint-plant v1 cache")
    result, spec = add_lp_features(x, root, cache_dir)
    if len(spec["columns"]) != 87 or not all(column.startswith("lpjoint_") for column in spec["columns"]):
        raise ValueError("Unexpected joint-plant feature schema")
    spec["monetary_schema"] = "All87 lpjoint_* columns are dimensionless; preserve during normalization. Joint economic margins retain baseline unit-solo best-energy denominators as comparative predictors, not exact unit derivatives of joint production. On fractions/counts are physical convex allocations, not probabilities forced into labels."
    return result, spec
'''
(PKG / "lp_joint_plant.py").write_text(module_source.lstrip("\n"), encoding="utf-8")


### Lp Shop Relaxation

In [ ]:
module_source = r'''
"""Fixed-head continuous generator-state ablation of the frozen hydraulic LP.

This tests a SHOP-inspired H1/H2 relaxation, not SHOP output or physical binary
commitment. The original LP, its input conversion and cached results are intact.
"""

from __future__ import annotations

import warnings

import numpy as np
from scipy.optimize import linprog
from scipy.sparse import coo_matrix, hstack, vstack

from sintef_project.lp_features import HydraulicLP, Network


def effective_energy(case: dict, horizon: int) -> np.ndarray:
    """Reproduce the frozen LP's negative-price average conversion exactly."""
    energy = np.repeat(np.asarray(case["energy"])[:, :, None], horizon, axis=2)
    average = (case["energy"] * case["widths"]).sum(axis=1) / case["widths"].sum(axis=1)
    energy[:, :, case["price"] < 0] = average[:, None, None]
    return energy


def state_interval(case: dict, flow: np.ndarray, power: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Return state bounds implied by both flow and power, without clipping."""
    lower = np.maximum(power / np.asarray(case["pmax"])[:, None],
                       flow / np.asarray(case["qmax"])[:, None])
    upper = np.minimum(1.0, np.minimum(power / np.asarray(case["pmin"])[:, None],
                                       flow / np.asarray(case["qmin"])[:, None]))
    return lower, upper


class ShopRelaxationLP:
    """Jointly optimize fractional generator states, water and transition costs.

    P is MW (equivalently MWh in one hour); Q is Mm3 per one-hour interval.
    Nominal YAML Pmin and base case-feasible Qmin/Qmax/Pmax are held fixed.
    The new state is bounded independently of the PQ segment decomposition.
    """

    def __init__(self, network: Network, horizon: int = 168, segments: int = 4):
        self.base = HydraulicLP(network, horizon, segments)
        self.network, self.horizon, self.segments = network, horizon, segments
        g, t = len(network.units), horizon
        if t < 1:
            raise ValueError("Horizon must be positive")
        cursor = self.base.n_variables

        def allocate(shape: tuple[int, int]) -> np.ndarray:
            nonlocal cursor
            result = np.arange(cursor, cursor + np.prod(shape)).reshape(shape)
            cursor += int(np.prod(shape))
            return result

        self.state = allocate((g, t))
        self.start = allocate((g, t - 1))
        self.stop = allocate((g, t - 1))
        self.n_variables = cursor
        self.a_eq = hstack([self.base.a_eq, coo_matrix((self.base.a_eq.shape[0],
                                                       cursor - self.base.n_variables))], format="csc")
        self.base_a_ub = hstack([self.base.a_ub, coo_matrix((self.base.a_ub.shape[0],
                                                           cursor - self.base.n_variables))], format="csc")
        rows, columns, values = [], [], []
        row = 0
        for unit in range(g):
            for hour in range(1, t):
                for col, val in [(self.state[unit, hour], 1), (self.state[unit, hour - 1], -1),
                                 (self.start[unit, hour - 1], -1)]:
                    rows.append(row); columns.append(col); values.append(val)
                row += 1
                for col, val in [(self.state[unit, hour - 1], 1), (self.state[unit, hour], -1),
                                 (self.stop[unit, hour - 1], -1)]:
                    rows.append(row); columns.append(col); values.append(val)
                row += 1
        self.transition_a_ub = coo_matrix((values, (rows, columns)), shape=(row, cursor)).tocsc()

    def solve(self, case: dict, transition_cost_scale: float = 1.0,
              time_limit: float = 20.0) -> dict:
        """Solve one fixed-head case, preserving raw states and feasibility proof."""
        base, network, t = self.base, self.network, self.horizon
        g, r = len(network.units), len(network.pools)
        for name in ["pmin", "pmax", "qmin", "qmax", "start_cost", "stop_cost"]:
            value = np.asarray(case[name], dtype=np.float64)
            if value.shape != (g,) or not np.isfinite(value).all():
                raise ValueError(f"Invalid unit parameter: {name}")
            if (value < 0).any() or (name not in ["start_cost", "stop_cost"] and (value == 0).any()):
                raise ValueError(f"Invalid parameter bounds: {name}")
        if transition_cost_scale not in (0.0, 1.0):
            raise ValueError("Only the predeclared zero-cost and supplied-cost arms are supported")
        if np.shape(case["energy"]) != (g, self.segments) or np.shape(case["widths"]) != (g, self.segments):
            raise ValueError("Only fixed initial-head segment arrays are supported")
        if not np.isfinite(case["energy"]).all() or not (np.asarray(case["widths"]) > 0).all():
            raise ValueError("Invalid segment inputs")
        np.testing.assert_allclose(np.sum(case["widths"], axis=1), case["qmax"], rtol=1e-12, atol=1e-12)
        energy = effective_energy(case, t)
        scale = 1 + np.abs(case["price"]).mean()
        penalty = case.get("slack_penalty", 1e7 + 100 * np.max(np.abs(case["terminal_value"])))
        c = np.zeros(self.n_variables)
        c[base.q] = -energy * case["price"][None, None, :]
        c[base.v[:, -1]] = -case["terminal_value"]
        c[base.bypass] = case.get("bypass_cost", 10 / 0.0036)
        c[base.spill] = case.get("spill_cost", 1000 / 0.0036)
        for index in (base.low_slack, base.high_slack, base.bypass_slack, base.vest_slack):
            c[index] = penalty
        c[self.start] = transition_cost_scale * np.asarray(case["start_cost"])[:, None]
        c[self.stop] = transition_cost_scale * np.asarray(case["stop_cost"])[:, None]
        bounds = np.column_stack([np.zeros(self.n_variables), np.full(self.n_variables, np.inf)])
        bounds[base.q.ravel(), 1] = np.repeat(case["widths"][:, :, None], t, axis=2).ravel()
        bounds[base.bypass.ravel(), 1] = case.get("bypass_capacity", 1000 * 0.0036)
        bounds[base.n_variables:, 1] = 1.0
        rhs = np.asarray(case["natural"], dtype=np.float64).copy()
        rhs[:, 0] += case["initial"]
        original_b_ub = np.concatenate([-case["minimum"].ravel(), np.repeat(case["capacity"], t),
                                        -case["bypass_minimum"].ravel(), case["vest_natural"] - case["vest_minimum"]])
        rows, columns, values = [], [], []
        row = 0
        for unit in range(g):
            for hour in range(t):
                # Lower P, upper P, lower Q, upper Q, respectively.
                for coefficients, state_coefficient in [
                    (-energy[unit, :, hour], case["pmin"][unit]),
                    (energy[unit, :, hour], -case["pmax"][unit]),
                    (-np.ones(self.segments), case["qmin"][unit]),
                    (np.ones(self.segments), -case["qmax"][unit]),
                ]:
                    for segment, coefficient in enumerate(coefficients):
                        rows.append(row); columns.append(base.q[unit, segment, hour]); values.append(coefficient)
                    rows.append(row); columns.append(self.state[unit, hour]); values.append(state_coefficient)
                    row += 1
        link_a_ub = coo_matrix((values, (rows, columns)), shape=(row, self.n_variables)).tocsc()
        a_ub = vstack([self.base_a_ub, link_a_ub, self.transition_a_ub], format="csc")
        b_ub = np.concatenate([original_b_ub, np.zeros(row + self.transition_a_ub.shape[0])])
        options = {"threads": 1, "time_limit": time_limit,
                   "dual_feasibility_tolerance": 1e-7, "primal_feasibility_tolerance": 1e-7}
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message="Unrecognized options detected.*")
            solved = linprog(c / scale, A_ub=a_ub, b_ub=b_ub, A_eq=self.a_eq,
                             b_eq=rhs.ravel(), bounds=bounds, method="highs", options=options)
        if not solved.success:
            raise RuntimeError(f"Linked-state LP failed: {solved.status}: {solved.message}")
        x = solved.x
        flow = x[base.q].sum(axis=1)
        power = (x[base.q] * energy).sum(axis=1)
        state = x[self.state]
        diff = np.diff(state, axis=1)
        reconstructed_cost = transition_cost_scale * float(
            np.sum(np.maximum(diff, 0) * np.asarray(case["start_cost"])[:, None])
            + np.sum(np.maximum(-diff, 0) * np.asarray(case["stop_cost"])[:, None]))
        raw_cost = float(c[self.start].ravel() @ x[self.start].ravel()
                         + c[self.stop].ravel() @ x[self.stop].ravel())
        np.testing.assert_allclose(raw_cost, reconstructed_cost, rtol=1e-9, atol=1e-5)
        equality_error = float(np.max(np.abs(self.a_eq @ x - rhs.ravel())))
        inequality_error = float(max(0.0, np.max(a_ub @ x - b_ub)))
        bound_error = float(max(0.0, np.max(bounds[:, 0] - x), np.max(x - bounds[:, 1])))
        if max(equality_error, inequality_error, bound_error) > 1e-5:
            raise RuntimeError("Linked-state LP feasibility check failed")
        lower, upper = state_interval(case, flow, power)
        if np.max(lower - state) > 1e-5 or np.max(state - upper) > 1e-5:
            raise RuntimeError("Linked-state interval check failed")
        sea = (flow[network.unit_down < 0].sum()
               + x[base.bypass][network.bypass_down < 0].sum()
               + x[base.spill][network.spill_down < 0].sum())
        balance = float(x[base.v][:, -1].sum() + sea
                        - np.sum(case["initial"], dtype=np.float64)
                        - np.sum(case["natural"], dtype=np.float64))
        if abs(balance) > 1e-5:
            raise RuntimeError("Linked-state LP aggregate water balance failed")
        objective = float(-solved.fun * scale)
        reconstructed_objective = float(
            np.sum(power * case["price"][None, :])
            + case["terminal_value"] @ x[base.v][:, -1]
            - case.get("bypass_cost", 10 / 0.0036) * x[base.bypass].sum()
            - case.get("spill_cost", 1000 / 0.0036) * x[base.spill].sum()
            - penalty * sum(x[index].sum() for index in
                            [base.low_slack, base.high_slack, base.bypass_slack, base.vest_slack])
            - reconstructed_cost)
        np.testing.assert_allclose(objective, reconstructed_objective, rtol=2e-11, atol=1e-5)
        return {"flow": flow, "power": power, "state": state,
                "raw_start": x[self.start], "raw_stop": x[self.stop],
                "state_lower": lower, "state_upper": upper,
                "segment_flow": x[base.q], "storage": x[base.v],
                "bypass": x[base.bypass], "spill": x[base.spill],
                "water_shadow": -solved.eqlin.marginals.reshape(r, t) * scale,
                "minimum_slack": x[base.low_slack], "maximum_slack": x[base.high_slack],
                "bypass_slack": x[base.bypass_slack], "vest_slack": x[base.vest_slack],
                "objective": objective, "reconstructed_objective": reconstructed_objective,
                "transition_cost": reconstructed_cost, "gross_objective": objective + reconstructed_cost,
                "balance_residual": equality_error, "inequality_residual": inequality_error,
                "bound_residual": bound_error, "total_balance_error": balance,
                "iterations": int(solved.nit), "status": int(solved.status),
                "negative_price_hours": int(np.sum(case["price"] < 0))}


def specification() -> dict:
    """Return the predeclared mechanism and explicit differences from SHOP."""
    return {
        "version": "shop_relaxed_state_fixed_head_v1",
        "states": "0<=u<=1; Pmin*u<=P<=Pmax*u; Qmin*u<=Q<=Qmax*u",
        "power": "P=sum(segment_Mm3_per_hour * MWh_per_Mm3), numerically MW for1h",
        "transition": "s>=u[t]-u[t-1], d>=u[t-1]-u[t], only t1..167; bounds0..1; supplied asymmetric constant costs",
        "boundaries": "No initial or final transition charge; no unknown initial state inferred",
        "minimums": "Pmin is supplied nominal YAML value; Qmin/Qmax/Pmax are frozen base initial-head values, not a reconstruction of SHOP's effective minimum algorithm",
        "arms": ["cached_base", "base_float64_rhs", "linked_zero_cost", "linked_supplied_cost"],
        "numerical_control": "New arms use float64 natural inflow before initial-volume RHS addition; fresh base on identical input isolates this from cached float32 RHS rounding",
        "unchanged": "Frozen v1 storage pools/routing/PQ/terminal conversion/slacks/negative-price average conversion; all14 units, fixed initial heads",
        "omissions": "No H3 feedback, head/reference-flow iteration, integer states, river-cost/tunnel reconstruction, unit priority, initial-state inference, labels or fitting",
        "source_scope": "Competition17.17 commands unknown; newer18.4i source and17.9.1 report motivate a falsifiable mechanism",
        "references": ["https://docs.shop.sintef.energy/commands/set_linear_startup.html",
                       "https://docs.shop.sintef.energy/commands/print_pqcurves.html",
                       "https://nva.sikt.no/registration/01a0f3805ef8-2645f514-8c93-4994-af5b-3f76bc65c52f"],
    }
'''
(PKG / "lp_shop_relaxation.py").write_text(module_source.lstrip("\n"), encoding="utf-8")


### Lp Shop Features

In [ ]:
module_source = r'''
"""Compact dimensionless additions from the fixed relaxed-state LP cache."""

from __future__ import annotations

import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd

from sintef_project.lp_features import Network, UNIT_ORDER

KINDS = ["state", "delta_qmin_ratio", "delta_power_fraction", "delta_economic_margin"]
COLUMNS = [f"shop_{kind}_{unit}" for unit in UNIT_ORDER for kind in KINDS]
VERSION = "shop_relaxed_state_56_v1"


def digest(path: Path) -> str:
    """Return a file hash without decoding raw target values."""
    value = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            value.update(block)
    return value.hexdigest()


def compute_features(case: dict, result: dict, baseline: dict, network: Network) -> np.ndarray:
    """Return168×56 input-only state/change predictors in frozen unit-major order."""
    if network.units != UNIT_ORDER:
        raise ValueError("Unexpected generator order")
    for name in ["state", "flow", "power"]:
        if np.shape(result[name]) != (14, 168) or not np.isfinite(result[name]).all():
            raise ValueError(f"Invalid relaxed-state result: {name}")
    if np.min(result["state"]) < -1e-5 or np.max(result["state"]) > 1+1e-5:
        raise ValueError("Relaxed state exceeds feasibility tolerance")
    scale = 1 + np.abs(case["price"]).mean()
    columns = []
    for j, _ in enumerate(UNIT_ORDER):
        up, down = network.unit_up[j], network.unit_down[j]
        new_nu = np.asarray(result["water_shadow"])[up].copy()
        old_nu = np.asarray(baseline["water_shadow"])[up].copy()
        if down >= 0:
            new_nu -= result["water_shadow"][down]
            old_nu -= baseline["water_shadow"][down]
        if min(case["qmin"][j], case["pmax"][j], case["energy"][j, 0], scale) <= 0:
            raise ValueError("Nonpositive feature conversion factor")
        columns.extend([np.clip(result["state"][j], 0, 1),
                        (result["flow"][j] - baseline["flow"][j]) / case["qmin"][j],
                        (result["power"][j] - baseline["power"][j]) / case["pmax"][j],
                        (old_nu - new_nu) / (case["energy"][j, 0] * scale)])
    values = np.column_stack(columns).astype(np.float32)
    if values.shape != (168, 56) or not np.isfinite(values).all():
        raise ValueError("Invalid compact relaxed-state features")
    return values


def feature_specification() -> dict:
    """Describe all appended columns; no post-result feature search is allowed."""
    return {"version": VERSION, "columns": COLUMNS, "dtype": "float32", "units": UNIT_ORDER,
            "state": "clip raw linked u to[0,1] only for numerical tolerance; raw u retained",
            "delta_qmin_ratio": "(newQ-cached_baseQ)/fixed_initial_Qmin; all Q in Mm3 per hour",
            "delta_power_fraction": "(newP-cached_baseP)/fixed_initial_case_feasible_Pmax",
            "delta_economic_margin": "(cached_base_local_dual-new_local_dual)/(initial_best_energy*(1+mean(abs(caseprice)))); local_dual=lambda_up-lambda_down, EUR/Mm3",
            "baseline": "Original frozen lp-full-001, preserving its numerical behavior and the existing608 feature prefix",
            "scope": "All14 units; only case inputs and input-derived LP solutions; no targets, fitted statistics, calendar/case-ID features, weights or threshold selection"}


def add_shop_features(x: pd.DataFrame, root: Path, cache_dir: Path) -> tuple[pd.DataFrame, dict]:
    """Verify a completed cache, then append56 columns to the unchanged608 prefix."""
    root = Path(root).resolve()
    repository = root.parent if root.name == "sintef_project" else root
    cache = Path(cache_dir)
    if not cache.is_absolute():
        cache = repository / cache
    manifest = json.loads((cache / "manifest.json").read_text())
    if manifest["status"] != "complete" or manifest["feature_specification"] != feature_specification():
        raise ValueError("Incomplete or different relaxed-state cache")
    if x.columns.tolist() != manifest["prefix_feature_columns"] or len(x.columns) != 608:
        raise ValueError("Existing608 feature prefix differs")
    for group in ["source", "data"]:
        for name, expected in manifest[group].items():
            if digest(repository / name) != expected:
                raise ValueError(f"Relaxed-state cache {group} changed: {name}")
    if digest(repository / manifest["baseline_manifest_path"]) != manifest["baseline_manifest_sha256"]:
        raise ValueError("Frozen baseline cache manifest changed")
    path = cache / "features.npz"
    if digest(path) != manifest["artifacts"]["features.npz"]:
        raise ValueError("Relaxed-state cache feature bytes changed")
    project = repository / "sintef_project"
    train = pd.read_csv(project / "data/kernel/Unit_commitment_decisions.csv", usecols=["Run No", "starttime"])
    cases = pd.concat([train, pd.read_csv(project / "prediction_mapping.csv")], ignore_index=True)
    if len(x) != len(cases)*168 or not np.array_equal(x["horizon_hour"].to_numpy(), np.tile(np.arange(168), len(cases))):
        raise ValueError("Expected complete case-major rows in the608 prefix")
    with np.load(path, allow_pickle=False) as z:
        if (z["case_ids"].tolist() != cases["Run No"].tolist() or z["columns"].tolist() != COLUMNS
                or z["units"].tolist() != UNIT_ORDER or z["features"].shape != (len(cases), 168, 56)
                or manifest["case_dates"] != pd.to_datetime(cases.starttime, utc=True).astype(str).tolist()):
            raise ValueError("Relaxed-state cache ID/date/unit/column alignment failed")
        values = z["features"].reshape(len(x), 56).copy()
    if not np.isfinite(values).all() or set(COLUMNS).intersection(x.columns):
        raise ValueError("Invalid or duplicate appended features")
    additions = pd.DataFrame(values, columns=COLUMNS, index=x.index)
    output = pd.concat([x, additions], axis=1)
    for name in x.columns:
        if output[name].dtype != x[name].dtype or not np.array_equal(output[name].to_numpy(), x[name].to_numpy()):
            raise ValueError("Appending relaxed features changed original608 values")
    return output, {"version": VERSION, "cache_directory": str(cache),
                    "manifest_sha256": digest(cache / "manifest.json"), "features_sha256": digest(path),
                    "shape": list(additions.shape), "columns": COLUMNS, "mathematics": feature_specification(),
                    "source_verified": manifest["source"], "data_verified": manifest["data"]}
'''
(PKG / "lp_shop_features.py").write_text(module_source.lstrip("\n"), encoding="utf-8")


### Lp Context

In [ ]:
module_source = r'''
"""Input-only temporal summaries of an already verified hydraulic LP cache.

The frozen LP itself is unchanged. These summaries expose its neighboring
discharge decisions to the supervised model so it can learn omitted switching
costs. They are predictors, not probabilities or asserted commitment rules.
"""
from pathlib import Path

import numpy as np
import pandas as pd

HORIZON = 168
UNITS = (
    "Hogga_G1", "Tokke_G1", "Tokke_G2", "Tokke_G3", "Tokke_G4",
    "Lio_G1", "Byrte_G1", "Vinje_G1", "Vinje_G2", "Vinje_G3",
    "Haukeli_G1", "Songa_G1", "Kjela_G1", "Vesle Kjela_G1",
)


def add_lp_context_features(x: pd.DataFrame, root: Path) -> tuple[pd.DataFrame, dict]:
    """Append nine case-local q/Qmin summaries for each of fourteen units.

    Requires raw or normalized features containing a verified LP cache and all
    168 hours of each case. Lags/leads clamp to that case's boundary. Six-hour
    means exclude the current hour and use the available-hour count; an empty
    neighborhood falls back to the current ratio. All added columns are float32
    and dimensionless. ``root`` is an unused common-interface parameter.
    """
    del root
    required = [f"lp_qmin_ratio_{unit}" for unit in UNITS]
    missing = [name for name in ["horizon_hour", *required] if name not in x]
    if missing:
        raise ValueError(f"Missing verified LP input columns: {missing}")
    if not len(x) or len(x) % HORIZON:
        raise ValueError("LP context requires complete 168-hour cases before training sampling")
    hours = np.arange(HORIZON)
    if not np.array_equal(x["horizon_hour"].to_numpy(), np.tile(hours, len(x) // HORIZON)):
        raise ValueError("LP context requires case-major hours 0 through 167")
    additions = {}
    previous_start = np.maximum(hours - 6, 0)
    next_end = np.minimum(hours + 7, HORIZON)
    for unit, column in zip(UNITS, required):
        q = x[column].to_numpy(dtype=np.float64).reshape(-1, HORIZON)
        if not np.isfinite(q).all():
            raise ValueError(f"Nonfinite LP discharge ratio: {unit}")
        for offset in (1, 3):
            additions[f"lp_context_qmin_previous_{offset}h_{unit}"] = q[:, np.maximum(hours - offset, 0)]
            additions[f"lp_context_qmin_next_{offset}h_{unit}"] = q[:, np.minimum(hours + offset, HORIZON - 1)]
        prefix = np.concatenate([np.zeros_like(q[:, :1]), q.cumsum(axis=1)], axis=1)
        for direction, sums, counts in (
            ("previous", prefix[:, hours] - prefix[:, previous_start], hours - previous_start),
            ("next", prefix[:, next_end] - prefix[:, hours + 1], next_end - hours - 1),
        ):
            additions[f"lp_context_qmin_{direction}_6h_mean_{unit}"] = np.divide(
                sums, counts[None, :], out=q.copy(), where=counts[None, :] > 0
            )
        additions[f"lp_context_qmin_week_mean_{unit}"] = np.repeat(q.mean(axis=1), HORIZON)
        additions[f"lp_context_qmin_week_at_least_one_fraction_{unit}"] = np.repeat((q >= 1).mean(axis=1), HORIZON)
        additions[f"lp_context_qmin_week_positive_fraction_{unit}"] = np.repeat((q > 0).mean(axis=1), HORIZON)
    overlap = set(additions).intersection(x.columns)
    if overlap:
        raise ValueError("LP context features have already been added")
    with np.errstate(over="ignore", invalid="ignore"):
        frame = pd.DataFrame({name: np.asarray(values, dtype=np.float32).reshape(-1)
                              for name, values in additions.items()}, index=x.index)
    if frame.shape != (len(x), 126) or not np.isfinite(frame.to_numpy()).all():
        raise ValueError("LP context must produce 126 finite float32 columns")
    spec = {
        "version": "lp_temporal_context_v1",
        "source_module": "sintef_project/lp_context.py",
        "input_columns": required,
        "units": list(UNITS),
        "added_columns": list(frame.columns),
        "source_provenance": "Derived only from lp_qmin_ratio_<unit> in the already verified LP cache. add_lp_features must verify cache source/data/artifact hashes before this transform. Trainer must archive this module and retain the upstream LP feature specification.",
        "lag_lead": "q(case,clip(hour+offset,0,167)), offset=-3,-1,+1,+3; never cross a case boundary",
        "neighbor_means": "Previous max(0,h-6)..h-1 and next h+1..min(167,h+6), excluding current; divide by available count; empty neighborhood=current q/Qmin",
        "week_summaries": "Mean q/Qmin, fraction q/Qmin>=1, fraction q/Qmin>0 over all168 hours, repeated per case; exact comparisons, no tolerance or clipping",
        "monetary_schema": "All126 added lp_context_* columns are dimensionless and must survive later economic normalization unchanged",
        "fit_scope": "No fitting, labels, filesystem reads, model prediction or LP solving; derive full-case context before training-row sampling",
        "interpretation": "LP relaxation omits unit switching/integer decisions. Neighbor and week activity summaries help the supervised learner model those corrections; q>=1/q>0 summaries are not commitment labels or hard feasibility rules.",
        "deferred": "No storage end-minus-current features in this bounded126-column ablation; their separate contribution is unmeasured",
    }
    return pd.concat([x, frame], axis=1), spec
'''
(PKG / "lp_context.py").write_text(module_source.lstrip("\n"), encoding="utf-8")


### Canonical Unit Features

In [ ]:
module_source = r'''
"""Align existing input-only features into one common generator representation.

No labels, predictions, fitted statistics or new physical solutions enter this
adapter. Row order is original case-hour, then the supplied generator order.
"""
from __future__ import annotations

from collections import Counter
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd

UNIT_FAMILIES = [
    'lp_qmin_ratio', 'lp_power_fraction', 'lp_economic_margin',
    'lpjoint_qmin_ratio', 'lpjoint_power_fraction', 'lpjoint_economic_margin',
    'shop_state', 'shop_delta_qmin_ratio', 'shop_delta_power_fraction',
    'shop_delta_economic_margin',
    'lp_context_qmin_previous_1h', 'lp_context_qmin_next_1h',
    'lp_context_qmin_previous_3h', 'lp_context_qmin_next_3h',
    'lp_context_qmin_previous_6h_mean', 'lp_context_qmin_next_6h_mean',
    'lp_context_qmin_week_mean', 'lp_context_qmin_week_at_least_one_fraction',
    'lp_context_qmin_week_positive_fraction',
]
PLANT_FAMILIES = [
    'physics_head', 'physics_pool_fill', 'physics_supply_fraction',
    'physics_catchment_inflow_fraction', 'physics_dispatch_to_avoid_spill',
    'physics_profitable_remaining_fraction',
    'lp_plant_power_fraction', 'lp_storage_fill', 'lp_storage_min_slack',
    'lp_storage_max_slack', 'lpjoint_plant_power_fraction',
    'lpjoint_storage_fill', 'lpjoint_storage_min_slack', 'lpjoint_storage_max_slack',
    'norm_physics_switch_margin_minimum', 'norm_physics_switch_margin_maximum',
]
CALENDAR = [
    'horizon_hour', 'horizon_day', 'hour_of_day', 'hour_sin', 'hour_cos',
    'weekday', 'month', 'day_of_year', 'season_sin', 'season_cos',
    'price_week_zscore', 'price_week_rank',
]


def feature_specification(prefix_columns: list[str], units: list[str]) -> dict:
    """Declare a deterministic 115-column mapping from the original664 schema."""
    if len(prefix_columns) != 664 or len(set(prefix_columns)) != 664:
        raise ValueError('The exact664 prefix schema is required')
    if len(units) != 14 or len(set(units)) != 14:
        raise ValueError('Fourteen distinct ordered generators are required')
    unit_plants, positions = [], []
    for unit in units:
        plant, number = unit.rsplit('_G', 1)
        position = int(number)
        if position < 1:
            raise ValueError('Generator position must be positive')
        unit_plants.append(plant)
        positions.append(position)
    plants = list(dict.fromkeys(unit_plants))
    if len(plants) != 9:
        raise ValueError('Expected the original nine generating plants')
    common = CALENDAR + ['context_rank_remaining', 'monetary_scale_inverse'] + [
        name for name in prefix_columns
        if name.startswith('norm_price') and not name.startswith('norm_price_minus_water_')]
    global_context = ['shop_state_' + unit for unit in units] + [
        'lpjoint_storage_fill_' + plant for plant in plants]
    own = [[family + '_' + unit for family in UNIT_FAMILIES] +
           [family + '_' + plant for family in PLANT_FAMILIES]
           for unit, plant in zip(units, unit_plants)]
    required = common + global_context + [name for unit_names in own for name in unit_names]
    missing = sorted(set(required) - set(prefix_columns))
    if missing or len(common) != 53:
        raise ValueError(f'Unexpected common schema or missing source columns: {missing}')
    columns = common + ['own_' + name for name in UNIT_FAMILIES + PLANT_FAMILIES] + [
        'global_' + name for name in global_context] + [
        'unit_id', 'plant_id', 'unit_position', 'plant_unit_count']
    if len(columns) != len(set(columns)) or len(columns) != 115:
        raise ValueError('Canonical schema must contain115 distinct columns')
    counts = Counter(unit_plants)
    identities = [[j, plants.index(plant), position, counts[plant]]
                  for j, (plant, position) in enumerate(zip(unit_plants, positions))]
    return {'version': 'canonical_existing664_unit115_v1', 'prefix_columns': prefix_columns,
            'units': units, 'plants': plants, 'columns': columns,
            'common_sources': common, 'own_sources': own,
            'global_sources': global_context, 'identity_values': identities,
            'categorical_columns': ['unit_id', 'plant_id'],
            'row_order': 'case_hour_then_unit', 'dtype': '<f4',
            'source_cast': 'Existing numeric features cast directly to float32; no fitted normalization.'}


def canonical_block(frame: pd.DataFrame, rows: np.ndarray, specification: dict) -> np.ndarray:
    """Remap a bounded row block without accessing targets or any other case rows."""
    if frame.columns.tolist() != specification['prefix_columns']:
        raise ValueError('Source column identity or ordering changed')
    if rows.ndim != 1 or rows.dtype.kind not in 'iu' or not len(rows):
        raise ValueError('Expected nonempty integer case-hour row indices')
    if np.any(rows < 0) or np.any(rows >= len(frame)) or len(np.unique(rows)) != len(rows):
        raise ValueError('Invalid or duplicate case-hour rows')
    expected = feature_specification(specification['prefix_columns'], specification['units'])
    if specification != expected:
        raise ValueError('Canonical mapping changed')
    source = frame.iloc[rows].to_numpy(dtype=np.float32)
    if not np.isfinite(source).all():
        raise ValueError('Original feature inputs must be finite')
    indices = {name: j for j, name in enumerate(specification['prefix_columns'])}
    common = [indices[name] for name in specification['common_sources']]
    own = np.asarray([[indices[name] for name in names] for names in specification['own_sources']])
    shared = [indices[name] for name in specification['global_sources']]
    result = np.empty((len(rows), 14, 115), dtype=np.float32)
    result[:, :, :53] = source[:, None, common]
    result[:, :, 53:88] = source[:, own]
    result[:, :, 88:111] = source[:, None, shared]
    result[:, :, 111:] = np.asarray(specification['identity_values'], dtype=np.float32)
    return result.reshape(len(rows) * 14, 115)


def write_canonical_rows(frame: pd.DataFrame, rows: np.ndarray, path: Path,
                         specification: dict, block_rows: int = 2048) -> dict:
    """Stream the remapping to an immutable float32 NPY with per-block byte hashes."""
    if path.exists() or block_rows < 1:
        raise ValueError('Output exists or block size is invalid')
    if rows.ndim != 1 or rows.dtype.kind not in 'iu' or not len(rows):
        raise ValueError('Expected nonempty integer case-hour row indices')
    if len(np.unique(rows)) != len(rows):
        raise ValueError('Duplicate case-hour rows')
    output = np.lib.format.open_memmap(path, mode='w+', dtype='<f4', shape=(len(rows) * 14, 115))
    records = []
    for begin in range(0, len(rows), block_rows):
        block = canonical_block(frame, rows[begin:begin + block_rows], specification)
        output[begin * 14:begin * 14 + len(block)] = block
        stored = output[begin * 14:begin * 14 + len(block)]
        if stored.tobytes() != block.tobytes():
            raise ValueError('Written canonical feature bytes changed')
        records.append({'source_row_begin': begin, 'source_row_end': begin + len(block) // 14,
                        'float32_sha256': hashlib.sha256(block.tobytes()).hexdigest()})
    output.flush()
    del output
    with path.open('rb') as stream:
        checksum = hashlib.file_digest(stream, 'sha256').hexdigest()
    return {'shape': [len(rows) * 14, 115], 'dtype': '<f4', 'sha256': checksum,
            'row_index_bytes_sha256': hashlib.sha256(rows.tobytes()).hexdigest(),
            'blocks': records, 'feature_specification': specification}
'''
(PKG / "canonical_unit_features.py").write_text(module_source.lstrip("\n"), encoding="utf-8")


### Postprocess

In [ ]:
module_source = r'''
"""Deterministic probability transforms confined to each scheduling case."""
from __future__ import annotations

from collections.abc import Sequence
import numpy as np

HORIZON = 168
DEFAULT_UNITS = (
    "Hogga_G1", "Tokke_G1", "Tokke_G2", "Tokke_G3", "Tokke_G4",
    "Lio_G1", "Byrte_G1", "Vinje_G1", "Vinje_G2", "Vinje_G3",
    "Haukeli_G1", "Songa_G1", "Kjela_G1", "Vesle Kjela_G1",
)


def _validate(p: np.ndarray) -> np.ndarray:
    values = np.asarray(p, dtype=np.float64)
    if values.ndim != 2 or not values.size or len(values) % HORIZON:
        raise ValueError("Expected nonempty (n_cases * 168, n_units) probabilities")
    if not np.isfinite(values).all() or ((values < 0) | (values > 1)).any():
        raise ValueError("Probabilities must be finite and within [0, 1]")
    return values


def smooth_predictions(p: np.ndarray, window: int, weight: float) -> np.ndarray:
    """Blend probabilities with a centered, edge-normalized within-case mean.

    The odd-width window includes the current hour. Boundary windows use only
    available hours of that case, with their actual count as denominator.
    No adjacent-case data or labels are read. Input arrays are never mutated.
    """
    values = _validate(p)
    if isinstance(window, bool) or not isinstance(window, (int, np.integer)):
        raise ValueError("window must be an odd integer between 1 and 167")
    if window < 1 or window >= HORIZON or window % 2 != 1:
        raise ValueError("window must be an odd integer between 1 and 167")
    if not np.isfinite(weight) or not 0 <= weight <= 1:
        raise ValueError("weight must be finite and within [0, 1]")
    if weight == 0 or window == 1:
        return values.copy()
    cube = values.reshape(-1, HORIZON, values.shape[1])
    prefix = np.concatenate([np.zeros_like(cube[:, :1]), cube.cumsum(axis=1)], axis=1)
    hours = np.arange(HORIZON)
    left = np.maximum(hours - window // 2, 0)
    right = np.minimum(hours + window // 2 + 1, HORIZON)
    local_mean = (prefix[:, right] - prefix[:, left]) / (right - left)[None, :, None]
    result = (1 - weight) * cube + weight * local_mean
    return np.clip(result.reshape(values.shape), 0, 1)


def project_vinje(p: np.ndarray, units: Sequence[str] = DEFAULT_UNITS) -> np.ndarray:
    """L2-project Vinje G1/G2/G3 probabilities onto their decreasing order.

    This empirical constraint is an optional ablation, not a physical certainty.
    Other generators are unchanged. The three-unit sum is preserved to floating
    precision; no probability is reassigned to a different generator.
    """
    values = _validate(p).copy()
    if len(units) != values.shape[1] or len(set(units)) != len(units):
        raise ValueError("Unique unit names must match probability columns")
    try:
        indices = [list(units).index(f"Vinje_G{i}") for i in (1, 2, 3)]
    except ValueError as error:
        raise ValueError("All three named Vinje generators are required") from error
    a, b, c = values[:, indices].T
    projected = np.column_stack([a, b, c])
    left_violation = a < b
    left_mean = (a + b) / 2
    right_violation = (~left_violation) & (b < c)
    right_mean = (b + c) / 2
    pool_all = (left_violation & (left_mean < c)) | (right_violation & (a < right_mean))
    pool_left = left_violation & ~pool_all
    pool_right = right_violation & ~pool_all
    projected[pool_left, :2] = left_mean[pool_left, None]
    projected[pool_right, 1:] = right_mean[pool_right, None]
    projected[pool_all] = ((a + b + c) / 3)[pool_all, None]
    values[:, indices] = projected
    return np.clip(values, 0, 1)
'''
(PKG / "postprocess.py").write_text(module_source.lstrip("\n"), encoding="utf-8")


### Startup

In [ ]:
module_source = r'''
import numpy as np
import pandas as pd
from sintef_project.canonical_unit_features import canonical_block
from sintef_project.lp_features import UNIT_ORDER as UNITS

PARAMS = {"boosting_type": "gbdt", "colsample_bytree": 0.9, "learning_rate": 0.06, "max_depth": -1,
          "min_child_samples": 100, "min_child_weight": 0.001, "min_split_gain": 0.0, "n_estimators": 1400,
          "num_leaves": 31, "objective": "binary", "reg_alpha": 0.0, "reg_lambda": 1.0, "subsample": 1.0,
          "subsample_freq": 0, "subsample_for_bin": 200000, "max_bin": 255, "deterministic": True,
          "force_col_wise": True, "verbosity": -1, "random_state": 4173}

STARTUP_COLUMNS = ["su_pmin_frac", "su_pmin_scaled", "su_pmax_scaled", "su_start_cost_per_pmax", "su_cycle_cost_per_pmin",
                   "su_margin_min_minus_cycle", "su_margin_max_minus_cycle", "su_lp_margin_times_pmin", "su_lp_margin_times_pmax",
                   "su_recover_hours_pmin", "su_recover_hours_pmax", "su_margin_next6", "su_margin_prev6",
                   "su_margin_next24", "su_margin_prev24", "su_margin_remaining"]

def startup_fields(block: np.ndarray, columns: list[str], consts: dict) -> np.ndarray:
    """block: (n_hours*14, 115) canonical rows, unit-minor. Returns (n, 16) float32."""
    idx = {c: j for j, c in enumerate(columns)}
    n = len(block); units = np.tile(np.arange(14), n // 14)
    pmin = np.asarray([consts[u]["pmin"] for u in UNITS])[units]; pmax = np.asarray([consts[u]["pmax"] for u in UNITS])[units]
    start = np.asarray([consts[u]["start"] for u in UNITS])[units]; stop = np.asarray([consts[u]["stop"] for u in UNITS])[units]
    ref = consts[UNITS[0]]["ref"]
    inv_scale = block[:, idx["monetary_scale_inverse"]].astype(np.float64)
    lp_margin = block[:, idx["own_lp_economic_margin"]].astype(np.float64)
    m_min = block[:, idx["own_norm_physics_switch_margin_minimum"]].astype(np.float64)
    m_max = block[:, idx["own_norm_physics_switch_margin_maximum"]].astype(np.float64)
    price = block[:, idx["norm_price"]].astype(np.float64)
    def diff(name): return block[:, idx[name]].astype(np.float64) - price
    start_norm = start * inv_scale / pmax; cycle_pmin = (start + stop) * inv_scale / pmin
    eps = 1e-3
    out = np.column_stack([
        pmin / pmax, pmin / ref, pmax / ref, start_norm, cycle_pmin,
        m_min - cycle_pmin, m_max - start_norm,
        lp_margin * pmin / ref, lp_margin * pmax / ref,
        np.clip(start * inv_scale / (pmin * np.maximum(lp_margin, eps)), 0, 168),
        np.clip(start * inv_scale / (pmax * np.maximum(lp_margin, eps)), 0, 168),
        lp_margin + diff("norm_price_next_mean_6h"), lp_margin + diff("norm_price_previous_mean_6h"),
        lp_margin + diff("norm_price_next_mean_24h"), lp_margin + diff("norm_price_previous_mean_24h"),
        lp_margin + diff("norm_price_remaining_mean"),
    ]).astype(np.float32)
    assert out.shape == (n, len(STARTUP_COLUMNS)) and np.isfinite(out).all()
    return out

def build_rows(frame: pd.DataFrame, rows: np.ndarray, spec: dict, consts: dict, block_rows: int = 4096, cc: np.ndarray | None = None) -> np.ndarray:
    parts = []
    for b in range(0, len(rows), block_rows):
        r = rows[b:b + block_rows]
        block = canonical_block(frame, r, spec)
        pieces = [block, startup_fields(block, spec["columns"], consts)]
        if cc is not None:
            pieces.append(cc[r // 168, r % 168].reshape(len(r) * 14, cc.shape[-1]))
        parts.append(np.hstack(pieces))
    return np.vstack(parts)
'''
(PKG / "startup.py").write_text(module_source.lstrip("\n"), encoding="utf-8")


### Tokke

In [ ]:
module_source = r'''
import numpy as np
from sintef_project.lp_features import UNIT_ORDER as UNITS

TOKKE = [UNITS.index(f"Tokke_G{i}") for i in (1, 2, 3, 4)]

def project_tokke(p: np.ndarray, iterations: int = 50) -> np.ndarray:
    """Approximate L2 projection onto G1 >= G3 >= G4 and G3 >= G2 (no G2/G4 order).

    Alternating pool-adjacent-violators on the two chains until no violation.
    Pair averaging preserves the four-probability sum up to roundoff; values stay in [0, 1].
    """
    values = np.asarray(p, dtype=np.float64).copy()
    g1, g2, g3, g4 = (values[:, TOKKE[i]].copy() for i in range(4))
    for _ in range(iterations):
        changed = False
        # chain G1 >= G3 >= G4
        for a, b in ((g1, g3), (g3, g4)):
            bad = a < b
            if bad.any():
                m = (a[bad] + b[bad]) / 2; a[bad] = m; b[bad] = m; changed = True
        bad = g3 < g2
        if bad.any():
            m = (g3[bad] + g2[bad]) / 2; g3[bad] = m; g2[bad] = m; changed = True
        if not changed:
            break
    for i, g in enumerate((g1, g2, g3, g4)):
        values[:, TOKKE[i]] = g
    return np.clip(values, 0, 1)
'''
(PKG / "tokke.py").write_text(module_source.lstrip("\n"), encoding="utf-8")


In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import yaml
from importlib.metadata import version
from threadpoolctl import threadpool_limits
threadpool_limits(THREADS)
EXPECTED_PACKAGES = {"numpy":"2.4.6", "pandas":"3.0.6", "scipy":"1.17.1",
    "PyYAML":"6.0.3", "lightgbm":"4.7.0", "xgboost":"3.4.1",
    "scikit-learn":"1.9.1", "threadpoolctl":"3.7.0"}
package_versions = {name:version(name) for name in EXPECTED_PACKAGES}
assert package_versions == EXPECTED_PACKAGES, f"Install the pinned requirements before running: {package_versions}"
assert sys.version_info[:2] == (3,11), "Use the documented Python 3.11 environment."
from sintef_project.features import build_features
from sintef_project.physics import add_physics_features
from sintef_project.context_features import add_context_features
from sintef_project.normalized import add_normalized_features
from sintef_project.lp_context import add_lp_context_features
from sintef_project.lp_features import HydroCaseData, HydraulicLP, result_features, UNIT_ORDER
from sintef_project.lp_joint_plant import JointPlantLP, build_joint_curve, result_joint_features
from sintef_project.lp_shop_relaxation import ShopRelaxationLP
from sintef_project.lp_shop_features import compute_features, COLUMNS as SHOP_COLUMNS
from sintef_project.canonical_unit_features import feature_specification
from sintef_project.startup import PARAMS, build_rows
from sintef_project.postprocess import project_vinje
from sintef_project.tokke import project_tokke
print("Versions:", np.__version__, pd.__version__, lgb.__version__, xgb.__version__)
assert xgb.__version__ == "3.4.1", "Use the archived CUDA XGBoost version, 3.4.1."
assert xgb.build_info().get("USE_CUDA"), "A CUDA-enabled XGBoost build and one GPU are required."


## 3. Compute hydraulic features from supplied inputs

In [ ]:
# Solve all three approximations afresh, using case inputs and no commitment targets.
inputs = HydroCaseData(WORK)
base_solver = HydraulicLP(inputs.network)
joint_solver = JointPlantLP(inputs.network)
shop_solver = ShopRelaxationLP(inputs.network)
lp_rows, joint_rows, shop_rows = [], [], []
for i in range(len(inputs.cases)):
    case = inputs.case(i)
    base_result = base_solver.solve(case, time_limit=20.0)
    base_frame = result_features(case, base_result, inputs.network)
    curves = [build_joint_curve(inputs, i, plant) for plant in ("Tokke", "Vinje")]
    joint_result = joint_solver.solve(case, curves, time_limit=20.0)
    joint_frame = result_joint_features(case, joint_result, inputs.network)
    # Preserve the original baseline arithmetic, then use the corrected RHS in the linked-state LP.
    case["natural"] = np.asarray(case["natural"], dtype=np.float64)
    for name, field in (("pmin", "p_min"), ("start_cost", "startcost_const"), ("stop_cost", "stopcost_const")):
        case[name] = np.asarray([inputs.model["generator"][u][field] for u in UNIT_ORDER], dtype=np.float64)
    shop_result = shop_solver.solve(case, 1.0)
    lp_rows.append(base_frame.to_numpy(dtype=np.float32))
    joint_rows.append(joint_frame.to_numpy(dtype=np.float32))
    shop_rows.append(compute_features(case, shop_result, base_result, inputs.network))
    if (i+1) % 100 == 0:
        print("Hydraulic cases:", i+1, "/", len(inputs.cases), flush=True)
lp_frame = pd.DataFrame(np.concatenate(lp_rows), columns=base_frame.columns)
joint_frame = pd.DataFrame(np.concatenate(joint_rows), columns=joint_frame.columns)
shop_frame = pd.DataFrame(np.concatenate(shop_rows), columns=SHOP_COLUMNS)
assert len(lp_frame) == len(joint_frame) == len(shop_frame) == 3647*168
assert all(np.isfinite(f.to_numpy()).all() for f in (lp_frame, joint_frame, shop_frame))
del lp_rows, joint_rows, shop_rows, inputs, base_solver, joint_solver, shop_solver
del case, curves, base_result, joint_result, shop_result, base_frame
gc.collect()


## 4. Build complete-week features and the fixed training sample

In [ ]:
base, labels, train, mapping, meta = build_features(WORK)
base, _ = add_physics_features(base, WORK)
base, _ = add_context_features(base, WORK)
UNITS = meta["units"]
assert UNITS == UNIT_ORDER
Y = np.column_stack([labels[u] for u in UNITS]).astype(np.uint8)
NTRAIN = len(train)*168
assert (len(train), len(mapping), len(UNITS)) == (2922, 725, 14)
del labels, train, meta
sample = np.sort(np.random.default_rng(4173).choice(NTRAIN, size=int(NTRAIN*.4), replace=False))
assert len(sample) == 196358
np.save(WORK / "training_sample.npy", sample)
# The archived 395/521/404/608 feature sets use different normalization order.
def feature_table(name):
    if name == "joint":
        f = pd.concat([base, joint_frame], axis=1)
        return add_normalized_features(f, WORK)[0]
    f = pd.concat([base, lp_frame], axis=1)
    if name != "lp":
        f = add_lp_context_features(f, WORK)[0]
    f = add_normalized_features(f, WORK)[0]
    if name in ("context_joint", "gpu608", "gpu664", "lgb664"):
        f = pd.concat([f, joint_frame], axis=1)
    if name in ("gpu664", "lgb664"):
        f = pd.concat([f, shop_frame], axis=1)
    return f


In [ ]:
COMPONENT_PARAMS = {'lp': {'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 0.9, 'importance_type': 'split', 'learning_rate': 0.06, 'max_depth': -1, 'min_child_samples': 100, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 350, 'n_jobs': 3, 'num_leaves': 31, 'objective': 'binary', 'random_state': 4173, 'reg_alpha': 0.0, 'reg_lambda': 1.0, 'subsample': 1.0, 'subsample_for_bin': 200000, 'subsample_freq': 0, 'max_bin': 255, 'deterministic': True, 'force_col_wise': True, 'verbosity': -1}, 'context': {'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 0.9, 'importance_type': 'split', 'learning_rate': 0.06, 'max_depth': -1, 'min_child_samples': 100, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 350, 'n_jobs': 3, 'num_leaves': 31, 'objective': 'binary', 'random_state': 4173, 'reg_alpha': 0.0, 'reg_lambda': 1.0, 'subsample': 1.0, 'subsample_for_bin': 200000, 'subsample_freq': 0, 'max_bin': 255, 'deterministic': True, 'force_col_wise': True, 'verbosity': -1}, 'joint': {'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 0.9, 'importance_type': 'split', 'learning_rate': 0.06, 'max_depth': -1, 'min_child_samples': 100, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 350, 'n_jobs': 3, 'num_leaves': 31, 'objective': 'binary', 'random_state': 4173, 'reg_alpha': 0.0, 'reg_lambda': 1.0, 'subsample': 1.0, 'subsample_for_bin': 200000, 'subsample_freq': 0, 'max_bin': 255, 'deterministic': True, 'force_col_wise': True, 'verbosity': -1}, 'context_joint': {'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 0.9, 'importance_type': 'split', 'learning_rate': 0.06, 'max_depth': -1, 'min_child_samples': 100, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 350, 'n_jobs': 3, 'num_leaves': 31, 'objective': 'binary', 'random_state': 4173, 'reg_alpha': 0.0, 'reg_lambda': 1.0, 'subsample': 1.0, 'subsample_for_bin': 200000, 'subsample_freq': 0, 'max_bin': 255, 'deterministic': True, 'force_col_wise': True, 'verbosity': -1}, 'gpu608': {'n_estimators': 600, 'max_depth': 6, 'learning_rate': 0.05, 'min_child_weight': 20, 'reg_lambda': 4.0, 'reg_alpha': 0.0, 'max_bin': 128, 'subsample': 0.9, 'colsample_bytree': 0.9, 'colsample_bylevel': 1.0, 'colsample_bynode': 1.0, 'random_state': 4173, 'n_jobs': 2, 'objective': 'binary:logistic', 'tree_method': 'hist', 'device': 'cuda:1', 'eval_metric': 'logloss', 'grow_policy': 'depthwise', 'max_delta_step': 0.0, 'validate_parameters': True, 'verbosity': 0}, 'gpu664': {'n_estimators': 600, 'max_depth': 6, 'learning_rate': 0.05, 'min_child_weight': 20, 'reg_lambda': 4.0, 'reg_alpha': 0.0, 'max_bin': 128, 'subsample': 0.9, 'colsample_bytree': 0.9, 'colsample_bylevel': 1.0, 'colsample_bynode': 1.0, 'random_state': 4173, 'n_jobs': 2, 'objective': 'binary:logistic', 'tree_method': 'hist', 'device': 'cuda:1', 'eval_metric': 'logloss', 'grow_policy': 'depthwise', 'max_delta_step': 0.0, 'validate_parameters': True, 'verbosity': 0}, 'lgb664': {'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 0.9, 'importance_type': 'split', 'learning_rate': 0.06, 'max_depth': -1, 'min_child_samples': 100, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 350, 'n_jobs': 2, 'num_leaves': 31, 'objective': 'binary', 'random_state': 4173, 'reg_alpha': 0.0, 'reg_lambda': 1.0, 'subsample': 1.0, 'subsample_for_bin': 200000, 'subsample_freq': 0, 'max_bin': 255, 'deterministic': True, 'force_col_wise': True, 'verbosity': -1}}
EXPECTED_COLUMNS = {'lp': ['horizon_hour', 'horizon_day', 'hour_of_day', 'hour_sin', 'hour_cos', 'weekday', 'month', 'day_of_year', 'season_sin', 'season_cos', 'price_week_zscore', 'price_week_rank', 'volume_Kjelavatn', 'fill_Kjelavatn', 'price_over_water_Kjelavatn', 'headroom_after_inflow_Kjelavatn', 'volume_Staavatn', 'fill_Staavatn', 'price_over_water_Staavatn', 'headroom_after_inflow_Staavatn', 'volume_Langesae', 'fill_Langesae', 'price_over_water_Langesae', 'headroom_after_inflow_Langesae', 'volume_Foersvatn', 'fill_Foersvatn', 'price_over_water_Foersvatn', 'headroom_after_inflow_Foersvatn', 'volume_Venemo', 'fill_Venemo', 'price_over_water_Venemo', 'headroom_after_inflow_Venemo', 'volume_Botnedalsvatn', 'fill_Botnedalsvatn', 'price_over_water_Botnedalsvatn', 'headroom_after_inflow_Botnedalsvatn', 'volume_Bitdalsvatn', 'fill_Bitdalsvatn', 'price_over_water_Bitdalsvatn', 'headroom_after_inflow_Bitdalsvatn', 'volume_Vaamarvatn', 'fill_Vaamarvatn', 'price_over_water_Vaamarvatn', 'headroom_after_inflow_Vaamarvatn', 'volume_Langeidvatn', 'fill_Langeidvatn', 'price_over_water_Langeidvatn', 'headroom_after_inflow_Langeidvatn', 'volume_Totak', 'fill_Totak', 'price_over_water_Totak', 'headroom_after_inflow_Totak', 'volume_Songavatn', 'fill_Songavatn', 'price_over_water_Songavatn', 'headroom_after_inflow_Songavatn', 'volume_Vinjevatn', 'fill_Vinjevatn', 'price_over_water_Vinjevatn', 'headroom_after_inflow_Vinjevatn', 'volume_Bandak', 'fill_Bandak', 'price_over_water_Bandak', 'headroom_after_inflow_Bandak', 'volume_Bordalsvatn', 'fill_Bordalsvatn', 'price_over_water_Bordalsvatn', 'headroom_after_inflow_Bordalsvatn', 'volume_Byrtevatn', 'fill_Byrtevatn', 'price_over_water_Byrtevatn', 'headroom_after_inflow_Byrtevatn', 'volume_Hyljelihyl', 'fill_Hyljelihyl', 'price_over_water_Hyljelihyl', 'headroom_after_inflow_Hyljelihyl', 'volume_Vatjern', 'fill_Vatjern', 'price_over_water_Vatjern', 'headroom_after_inflow_Vatjern', 'inflow_Langesae', 'inflow_week_mean_Langesae', 'inflow_elapsed_mm3_Langesae', 'inflow_remaining_mm3_Langesae', 'inflow_Staavatn', 'inflow_week_mean_Staavatn', 'inflow_elapsed_mm3_Staavatn', 'inflow_remaining_mm3_Staavatn', 'inflow_Kjelavatn', 'inflow_week_mean_Kjelavatn', 'inflow_elapsed_mm3_Kjelavatn', 'inflow_remaining_mm3_Kjelavatn', 'inflow_Bordalsvatn', 'inflow_week_mean_Bordalsvatn', 'inflow_elapsed_mm3_Bordalsvatn', 'inflow_remaining_mm3_Bordalsvatn', 'inflow_Foersvatn', 'inflow_week_mean_Foersvatn', 'inflow_elapsed_mm3_Foersvatn', 'inflow_remaining_mm3_Foersvatn', 'inflow_Hyljelihyl', 'inflow_week_mean_Hyljelihyl', 'inflow_elapsed_mm3_Hyljelihyl', 'inflow_remaining_mm3_Hyljelihyl', 'inflow_Venemo', 'inflow_week_mean_Venemo', 'inflow_elapsed_mm3_Venemo', 'inflow_remaining_mm3_Venemo', 'inflow_Bitdalsvatn', 'inflow_week_mean_Bitdalsvatn', 'inflow_elapsed_mm3_Bitdalsvatn', 'inflow_remaining_mm3_Bitdalsvatn', 'inflow_Songavatn', 'inflow_week_mean_Songavatn', 'inflow_elapsed_mm3_Songavatn', 'inflow_remaining_mm3_Songavatn', 'inflow_Totak', 'inflow_week_mean_Totak', 'inflow_elapsed_mm3_Totak', 'inflow_remaining_mm3_Totak', 'inflow_Vaamarvatn', 'inflow_week_mean_Vaamarvatn', 'inflow_elapsed_mm3_Vaamarvatn', 'inflow_remaining_mm3_Vaamarvatn', 'inflow_Langeidvatn', 'inflow_week_mean_Langeidvatn', 'inflow_elapsed_mm3_Langeidvatn', 'inflow_remaining_mm3_Langeidvatn', 'inflow_Vatjern', 'inflow_week_mean_Vatjern', 'inflow_elapsed_mm3_Vatjern', 'inflow_remaining_mm3_Vatjern', 'inflow_Vinjevatn', 'inflow_week_mean_Vinjevatn', 'inflow_elapsed_mm3_Vinjevatn', 'inflow_remaining_mm3_Vinjevatn', 'inflow_Botnedalsvatn', 'inflow_week_mean_Botnedalsvatn', 'inflow_elapsed_mm3_Botnedalsvatn', 'inflow_remaining_mm3_Botnedalsvatn', 'inflow_Byrtevatn', 'inflow_week_mean_Byrtevatn', 'inflow_elapsed_mm3_Byrtevatn', 'inflow_remaining_mm3_Byrtevatn', 'inflow_Bandak', 'inflow_week_mean_Bandak', 'inflow_elapsed_mm3_Bandak', 'inflow_remaining_mm3_Bandak', 'inflow_r_Vest_Vassdraget', 'inflow_week_mean_r_Vest_Vassdraget', 'inflow_elapsed_mm3_r_Vest_Vassdraget', 'inflow_remaining_mm3_r_Vest_Vassdraget', 'min_volume_Totak', 'min_volume_week_max_Totak', 'min_volume_Byrtevatn', 'min_volume_week_max_Byrtevatn', 'min_volume_Staavatn', 'min_volume_week_max_Staavatn', 'min_flow_b_Kjelavatn', 'min_flow_week_max_b_Kjelavatn', 'min_flow_b_Hyljelihyl', 'min_flow_week_max_b_Hyljelihyl', 'min_flow_b_Vinjevatn', 'min_flow_week_max_b_Vinjevatn', 'min_flow_b_Byrtevatn', 'min_flow_week_max_b_Byrtevatn', 'min_flow_b_Bandak', 'min_flow_week_max_b_Bandak', 'min_flow_r_Vest_Vassdraget', 'min_flow_week_max_r_Vest_Vassdraget', 'physics_head_Hogga', 'physics_pool_fill_Hogga', 'physics_supply_fraction_Hogga', 'physics_catchment_inflow_fraction_Hogga', 'physics_dispatch_to_avoid_spill_Hogga', 'physics_profitable_remaining_fraction_Hogga', 'physics_head_Tokke', 'physics_pool_fill_Tokke', 'physics_supply_fraction_Tokke', 'physics_catchment_inflow_fraction_Tokke', 'physics_dispatch_to_avoid_spill_Tokke', 'physics_profitable_remaining_fraction_Tokke', 'physics_head_Lio', 'physics_pool_fill_Lio', 'physics_supply_fraction_Lio', 'physics_catchment_inflow_fraction_Lio', 'physics_dispatch_to_avoid_spill_Lio', 'physics_profitable_remaining_fraction_Lio', 'physics_head_Byrte', 'physics_pool_fill_Byrte', 'physics_supply_fraction_Byrte', 'physics_catchment_inflow_fraction_Byrte', 'physics_dispatch_to_avoid_spill_Byrte', 'physics_profitable_remaining_fraction_Byrte', 'physics_head_Vinje', 'physics_pool_fill_Vinje', 'physics_supply_fraction_Vinje', 'physics_catchment_inflow_fraction_Vinje', 'physics_dispatch_to_avoid_spill_Vinje', 'physics_profitable_remaining_fraction_Vinje', 'physics_head_Haukeli', 'physics_pool_fill_Haukeli', 'physics_supply_fraction_Haukeli', 'physics_catchment_inflow_fraction_Haukeli', 'physics_dispatch_to_avoid_spill_Haukeli', 'physics_profitable_remaining_fraction_Haukeli', 'physics_head_Songa', 'physics_pool_fill_Songa', 'physics_supply_fraction_Songa', 'physics_catchment_inflow_fraction_Songa', 'physics_dispatch_to_avoid_spill_Songa', 'physics_profitable_remaining_fraction_Songa', 'physics_head_Kjela', 'physics_pool_fill_Kjela', 'physics_supply_fraction_Kjela', 'physics_catchment_inflow_fraction_Kjela', 'physics_dispatch_to_avoid_spill_Kjela', 'physics_profitable_remaining_fraction_Kjela', 'physics_head_Vesle Kjela', 'physics_pool_fill_Vesle Kjela', 'physics_supply_fraction_Vesle Kjela', 'physics_catchment_inflow_fraction_Vesle Kjela', 'physics_dispatch_to_avoid_spill_Vesle Kjela', 'physics_profitable_remaining_fraction_Vesle Kjela', 'context_rank_remaining', 'lp_qmin_ratio_Hogga_G1', 'lp_power_fraction_Hogga_G1', 'lp_economic_margin_Hogga_G1', 'lp_qmin_ratio_Tokke_G1', 'lp_power_fraction_Tokke_G1', 'lp_economic_margin_Tokke_G1', 'lp_qmin_ratio_Tokke_G2', 'lp_power_fraction_Tokke_G2', 'lp_economic_margin_Tokke_G2', 'lp_qmin_ratio_Tokke_G3', 'lp_power_fraction_Tokke_G3', 'lp_economic_margin_Tokke_G3', 'lp_qmin_ratio_Tokke_G4', 'lp_power_fraction_Tokke_G4', 'lp_economic_margin_Tokke_G4', 'lp_qmin_ratio_Lio_G1', 'lp_power_fraction_Lio_G1', 'lp_economic_margin_Lio_G1', 'lp_qmin_ratio_Byrte_G1', 'lp_power_fraction_Byrte_G1', 'lp_economic_margin_Byrte_G1', 'lp_qmin_ratio_Vinje_G1', 'lp_power_fraction_Vinje_G1', 'lp_economic_margin_Vinje_G1', 'lp_qmin_ratio_Vinje_G2', 'lp_power_fraction_Vinje_G2', 'lp_economic_margin_Vinje_G2', 'lp_qmin_ratio_Vinje_G3', 'lp_power_fraction_Vinje_G3', 'lp_economic_margin_Vinje_G3', 'lp_qmin_ratio_Haukeli_G1', 'lp_power_fraction_Haukeli_G1', 'lp_economic_margin_Haukeli_G1', 'lp_qmin_ratio_Songa_G1', 'lp_power_fraction_Songa_G1', 'lp_economic_margin_Songa_G1', 'lp_qmin_ratio_Kjela_G1', 'lp_power_fraction_Kjela_G1', 'lp_economic_margin_Kjela_G1', 'lp_qmin_ratio_Vesle Kjela_G1', 'lp_power_fraction_Vesle Kjela_G1', 'lp_economic_margin_Vesle Kjela_G1', 'lp_plant_power_fraction_Hogga', 'lp_storage_fill_Hogga', 'lp_storage_min_slack_Hogga', 'lp_storage_max_slack_Hogga', 'lp_plant_power_fraction_Tokke', 'lp_storage_fill_Tokke', 'lp_storage_min_slack_Tokke', 'lp_storage_max_slack_Tokke', 'lp_plant_power_fraction_Lio', 'lp_storage_fill_Lio', 'lp_storage_min_slack_Lio', 'lp_storage_max_slack_Lio', 'lp_plant_power_fraction_Byrte', 'lp_storage_fill_Byrte', 'lp_storage_min_slack_Byrte', 'lp_storage_max_slack_Byrte', 'lp_plant_power_fraction_Vinje', 'lp_storage_fill_Vinje', 'lp_storage_min_slack_Vinje', 'lp_storage_max_slack_Vinje', 'lp_plant_power_fraction_Haukeli', 'lp_storage_fill_Haukeli', 'lp_storage_min_slack_Haukeli', 'lp_storage_max_slack_Haukeli', 'lp_plant_power_fraction_Songa', 'lp_storage_fill_Songa', 'lp_storage_min_slack_Songa', 'lp_storage_max_slack_Songa', 'lp_plant_power_fraction_Kjela', 'lp_storage_fill_Kjela', 'lp_storage_min_slack_Kjela', 'lp_storage_max_slack_Kjela', 'lp_plant_power_fraction_Vesle Kjela', 'lp_storage_fill_Vesle Kjela', 'lp_storage_min_slack_Vesle Kjela', 'lp_storage_max_slack_Vesle Kjela', 'norm_price', 'norm_price_offset_-24', 'norm_price_offset_-12', 'norm_price_offset_-6', 'norm_price_offset_-3', 'norm_price_offset_-1', 'norm_price_offset_+1', 'norm_price_offset_+3', 'norm_price_offset_+6', 'norm_price_offset_+12', 'norm_price_offset_+24', 'norm_price_week_mean', 'norm_price_week_std', 'norm_price_week_min', 'norm_price_week_max', 'norm_price_week_q10', 'norm_price_week_q25', 'norm_price_week_q50', 'norm_price_week_q75', 'norm_price_week_q90', 'norm_price_minus_week_mean', 'norm_price_day_mean', 'norm_price_day_min', 'norm_price_day_max', 'norm_price_day_std', 'norm_price_remaining_mean', 'norm_price_remaining_max', 'norm_water_value_Kjelavatn', 'norm_price_minus_water_Kjelavatn', 'norm_water_value_Staavatn', 'norm_price_minus_water_Staavatn', 'norm_water_value_Langesae', 'norm_price_minus_water_Langesae', 'norm_water_value_Foersvatn', 'norm_price_minus_water_Foersvatn', 'norm_water_value_Venemo', 'norm_price_minus_water_Venemo', 'norm_water_value_Botnedalsvatn', 'norm_price_minus_water_Botnedalsvatn', 'norm_water_value_Bitdalsvatn', 'norm_price_minus_water_Bitdalsvatn', 'norm_water_value_Vaamarvatn', 'norm_price_minus_water_Vaamarvatn', 'norm_water_value_Langeidvatn', 'norm_price_minus_water_Langeidvatn', 'norm_water_value_Totak', 'norm_price_minus_water_Totak', 'norm_water_value_Songavatn', 'norm_price_minus_water_Songavatn', 'norm_water_value_Vinjevatn', 'norm_price_minus_water_Vinjevatn', 'norm_water_value_Bandak', 'norm_price_minus_water_Bandak', 'norm_water_value_Bordalsvatn', 'norm_price_minus_water_Bordalsvatn', 'norm_water_value_Byrtevatn', 'norm_price_minus_water_Byrtevatn', 'norm_water_value_Hyljelihyl', 'norm_price_minus_water_Hyljelihyl', 'norm_water_value_Vatjern', 'norm_price_minus_water_Vatjern', 'norm_physics_switch_margin_minimum_Hogga', 'norm_physics_switch_margin_maximum_Hogga', 'norm_physics_switch_margin_minimum_Tokke', 'norm_physics_switch_margin_maximum_Tokke', 'norm_physics_switch_margin_minimum_Lio', 'norm_physics_switch_margin_maximum_Lio', 'norm_physics_switch_margin_minimum_Byrte', 'norm_physics_switch_margin_maximum_Byrte', 'norm_physics_switch_margin_minimum_Vinje', 'norm_physics_switch_margin_maximum_Vinje', 'norm_physics_switch_margin_minimum_Haukeli', 'norm_physics_switch_margin_maximum_Haukeli', 'norm_physics_switch_margin_minimum_Songa', 'norm_physics_switch_margin_maximum_Songa', 'norm_physics_switch_margin_minimum_Kjela', 'norm_physics_switch_margin_maximum_Kjela', 'norm_physics_switch_margin_minimum_Vesle Kjela', 'norm_physics_switch_margin_maximum_Vesle Kjela', 'norm_price_previous_mean_6h', 'norm_price_minus_previous_mean_6h', 'norm_price_next_mean_6h', 'norm_price_minus_next_mean_6h', 'norm_price_previous_mean_12h', 'norm_price_minus_previous_mean_12h', 'norm_price_next_mean_12h', 'norm_price_minus_next_mean_12h', 'norm_price_previous_mean_24h', 'norm_price_minus_previous_mean_24h', 'norm_price_next_mean_24h', 'norm_price_minus_next_mean_24h', 'monetary_scale_inverse'], 'context': ['horizon_hour', 'horizon_day', 'hour_of_day', 'hour_sin', 'hour_cos', 'weekday', 'month', 'day_of_year', 'season_sin', 'season_cos', 'price_week_zscore', 'price_week_rank', 'volume_Kjelavatn', 'fill_Kjelavatn', 'price_over_water_Kjelavatn', 'headroom_after_inflow_Kjelavatn', 'volume_Staavatn', 'fill_Staavatn', 'price_over_water_Staavatn', 'headroom_after_inflow_Staavatn', 'volume_Langesae', 'fill_Langesae', 'price_over_water_Langesae', 'headroom_after_inflow_Langesae', 'volume_Foersvatn', 'fill_Foersvatn', 'price_over_water_Foersvatn', 'headroom_after_inflow_Foersvatn', 'volume_Venemo', 'fill_Venemo', 'price_over_water_Venemo', 'headroom_after_inflow_Venemo', 'volume_Botnedalsvatn', 'fill_Botnedalsvatn', 'price_over_water_Botnedalsvatn', 'headroom_after_inflow_Botnedalsvatn', 'volume_Bitdalsvatn', 'fill_Bitdalsvatn', 'price_over_water_Bitdalsvatn', 'headroom_after_inflow_Bitdalsvatn', 'volume_Vaamarvatn', 'fill_Vaamarvatn', 'price_over_water_Vaamarvatn', 'headroom_after_inflow_Vaamarvatn', 'volume_Langeidvatn', 'fill_Langeidvatn', 'price_over_water_Langeidvatn', 'headroom_after_inflow_Langeidvatn', 'volume_Totak', 'fill_Totak', 'price_over_water_Totak', 'headroom_after_inflow_Totak', 'volume_Songavatn', 'fill_Songavatn', 'price_over_water_Songavatn', 'headroom_after_inflow_Songavatn', 'volume_Vinjevatn', 'fill_Vinjevatn', 'price_over_water_Vinjevatn', 'headroom_after_inflow_Vinjevatn', 'volume_Bandak', 'fill_Bandak', 'price_over_water_Bandak', 'headroom_after_inflow_Bandak', 'volume_Bordalsvatn', 'fill_Bordalsvatn', 'price_over_water_Bordalsvatn', 'headroom_after_inflow_Bordalsvatn', 'volume_Byrtevatn', 'fill_Byrtevatn', 'price_over_water_Byrtevatn', 'headroom_after_inflow_Byrtevatn', 'volume_Hyljelihyl', 'fill_Hyljelihyl', 'price_over_water_Hyljelihyl', 'headroom_after_inflow_Hyljelihyl', 'volume_Vatjern', 'fill_Vatjern', 'price_over_water_Vatjern', 'headroom_after_inflow_Vatjern', 'inflow_Langesae', 'inflow_week_mean_Langesae', 'inflow_elapsed_mm3_Langesae', 'inflow_remaining_mm3_Langesae', 'inflow_Staavatn', 'inflow_week_mean_Staavatn', 'inflow_elapsed_mm3_Staavatn', 'inflow_remaining_mm3_Staavatn', 'inflow_Kjelavatn', 'inflow_week_mean_Kjelavatn', 'inflow_elapsed_mm3_Kjelavatn', 'inflow_remaining_mm3_Kjelavatn', 'inflow_Bordalsvatn', 'inflow_week_mean_Bordalsvatn', 'inflow_elapsed_mm3_Bordalsvatn', 'inflow_remaining_mm3_Bordalsvatn', 'inflow_Foersvatn', 'inflow_week_mean_Foersvatn', 'inflow_elapsed_mm3_Foersvatn', 'inflow_remaining_mm3_Foersvatn', 'inflow_Hyljelihyl', 'inflow_week_mean_Hyljelihyl', 'inflow_elapsed_mm3_Hyljelihyl', 'inflow_remaining_mm3_Hyljelihyl', 'inflow_Venemo', 'inflow_week_mean_Venemo', 'inflow_elapsed_mm3_Venemo', 'inflow_remaining_mm3_Venemo', 'inflow_Bitdalsvatn', 'inflow_week_mean_Bitdalsvatn', 'inflow_elapsed_mm3_Bitdalsvatn', 'inflow_remaining_mm3_Bitdalsvatn', 'inflow_Songavatn', 'inflow_week_mean_Songavatn', 'inflow_elapsed_mm3_Songavatn', 'inflow_remaining_mm3_Songavatn', 'inflow_Totak', 'inflow_week_mean_Totak', 'inflow_elapsed_mm3_Totak', 'inflow_remaining_mm3_Totak', 'inflow_Vaamarvatn', 'inflow_week_mean_Vaamarvatn', 'inflow_elapsed_mm3_Vaamarvatn', 'inflow_remaining_mm3_Vaamarvatn', 'inflow_Langeidvatn', 'inflow_week_mean_Langeidvatn', 'inflow_elapsed_mm3_Langeidvatn', 'inflow_remaining_mm3_Langeidvatn', 'inflow_Vatjern', 'inflow_week_mean_Vatjern', 'inflow_elapsed_mm3_Vatjern', 'inflow_remaining_mm3_Vatjern', 'inflow_Vinjevatn', 'inflow_week_mean_Vinjevatn', 'inflow_elapsed_mm3_Vinjevatn', 'inflow_remaining_mm3_Vinjevatn', 'inflow_Botnedalsvatn', 'inflow_week_mean_Botnedalsvatn', 'inflow_elapsed_mm3_Botnedalsvatn', 'inflow_remaining_mm3_Botnedalsvatn', 'inflow_Byrtevatn', 'inflow_week_mean_Byrtevatn', 'inflow_elapsed_mm3_Byrtevatn', 'inflow_remaining_mm3_Byrtevatn', 'inflow_Bandak', 'inflow_week_mean_Bandak', 'inflow_elapsed_mm3_Bandak', 'inflow_remaining_mm3_Bandak', 'inflow_r_Vest_Vassdraget', 'inflow_week_mean_r_Vest_Vassdraget', 'inflow_elapsed_mm3_r_Vest_Vassdraget', 'inflow_remaining_mm3_r_Vest_Vassdraget', 'min_volume_Totak', 'min_volume_week_max_Totak', 'min_volume_Byrtevatn', 'min_volume_week_max_Byrtevatn', 'min_volume_Staavatn', 'min_volume_week_max_Staavatn', 'min_flow_b_Kjelavatn', 'min_flow_week_max_b_Kjelavatn', 'min_flow_b_Hyljelihyl', 'min_flow_week_max_b_Hyljelihyl', 'min_flow_b_Vinjevatn', 'min_flow_week_max_b_Vinjevatn', 'min_flow_b_Byrtevatn', 'min_flow_week_max_b_Byrtevatn', 'min_flow_b_Bandak', 'min_flow_week_max_b_Bandak', 'min_flow_r_Vest_Vassdraget', 'min_flow_week_max_r_Vest_Vassdraget', 'physics_head_Hogga', 'physics_pool_fill_Hogga', 'physics_supply_fraction_Hogga', 'physics_catchment_inflow_fraction_Hogga', 'physics_dispatch_to_avoid_spill_Hogga', 'physics_profitable_remaining_fraction_Hogga', 'physics_head_Tokke', 'physics_pool_fill_Tokke', 'physics_supply_fraction_Tokke', 'physics_catchment_inflow_fraction_Tokke', 'physics_dispatch_to_avoid_spill_Tokke', 'physics_profitable_remaining_fraction_Tokke', 'physics_head_Lio', 'physics_pool_fill_Lio', 'physics_supply_fraction_Lio', 'physics_catchment_inflow_fraction_Lio', 'physics_dispatch_to_avoid_spill_Lio', 'physics_profitable_remaining_fraction_Lio', 'physics_head_Byrte', 'physics_pool_fill_Byrte', 'physics_supply_fraction_Byrte', 'physics_catchment_inflow_fraction_Byrte', 'physics_dispatch_to_avoid_spill_Byrte', 'physics_profitable_remaining_fraction_Byrte', 'physics_head_Vinje', 'physics_pool_fill_Vinje', 'physics_supply_fraction_Vinje', 'physics_catchment_inflow_fraction_Vinje', 'physics_dispatch_to_avoid_spill_Vinje', 'physics_profitable_remaining_fraction_Vinje', 'physics_head_Haukeli', 'physics_pool_fill_Haukeli', 'physics_supply_fraction_Haukeli', 'physics_catchment_inflow_fraction_Haukeli', 'physics_dispatch_to_avoid_spill_Haukeli', 'physics_profitable_remaining_fraction_Haukeli', 'physics_head_Songa', 'physics_pool_fill_Songa', 'physics_supply_fraction_Songa', 'physics_catchment_inflow_fraction_Songa', 'physics_dispatch_to_avoid_spill_Songa', 'physics_profitable_remaining_fraction_Songa', 'physics_head_Kjela', 'physics_pool_fill_Kjela', 'physics_supply_fraction_Kjela', 'physics_catchment_inflow_fraction_Kjela', 'physics_dispatch_to_avoid_spill_Kjela', 'physics_profitable_remaining_fraction_Kjela', 'physics_head_Vesle Kjela', 'physics_pool_fill_Vesle Kjela', 'physics_supply_fraction_Vesle Kjela', 'physics_catchment_inflow_fraction_Vesle Kjela', 'physics_dispatch_to_avoid_spill_Vesle Kjela', 'physics_profitable_remaining_fraction_Vesle Kjela', 'context_rank_remaining', 'lp_qmin_ratio_Hogga_G1', 'lp_power_fraction_Hogga_G1', 'lp_economic_margin_Hogga_G1', 'lp_qmin_ratio_Tokke_G1', 'lp_power_fraction_Tokke_G1', 'lp_economic_margin_Tokke_G1', 'lp_qmin_ratio_Tokke_G2', 'lp_power_fraction_Tokke_G2', 'lp_economic_margin_Tokke_G2', 'lp_qmin_ratio_Tokke_G3', 'lp_power_fraction_Tokke_G3', 'lp_economic_margin_Tokke_G3', 'lp_qmin_ratio_Tokke_G4', 'lp_power_fraction_Tokke_G4', 'lp_economic_margin_Tokke_G4', 'lp_qmin_ratio_Lio_G1', 'lp_power_fraction_Lio_G1', 'lp_economic_margin_Lio_G1', 'lp_qmin_ratio_Byrte_G1', 'lp_power_fraction_Byrte_G1', 'lp_economic_margin_Byrte_G1', 'lp_qmin_ratio_Vinje_G1', 'lp_power_fraction_Vinje_G1', 'lp_economic_margin_Vinje_G1', 'lp_qmin_ratio_Vinje_G2', 'lp_power_fraction_Vinje_G2', 'lp_economic_margin_Vinje_G2', 'lp_qmin_ratio_Vinje_G3', 'lp_power_fraction_Vinje_G3', 'lp_economic_margin_Vinje_G3', 'lp_qmin_ratio_Haukeli_G1', 'lp_power_fraction_Haukeli_G1', 'lp_economic_margin_Haukeli_G1', 'lp_qmin_ratio_Songa_G1', 'lp_power_fraction_Songa_G1', 'lp_economic_margin_Songa_G1', 'lp_qmin_ratio_Kjela_G1', 'lp_power_fraction_Kjela_G1', 'lp_economic_margin_Kjela_G1', 'lp_qmin_ratio_Vesle Kjela_G1', 'lp_power_fraction_Vesle Kjela_G1', 'lp_economic_margin_Vesle Kjela_G1', 'lp_plant_power_fraction_Hogga', 'lp_storage_fill_Hogga', 'lp_storage_min_slack_Hogga', 'lp_storage_max_slack_Hogga', 'lp_plant_power_fraction_Tokke', 'lp_storage_fill_Tokke', 'lp_storage_min_slack_Tokke', 'lp_storage_max_slack_Tokke', 'lp_plant_power_fraction_Lio', 'lp_storage_fill_Lio', 'lp_storage_min_slack_Lio', 'lp_storage_max_slack_Lio', 'lp_plant_power_fraction_Byrte', 'lp_storage_fill_Byrte', 'lp_storage_min_slack_Byrte', 'lp_storage_max_slack_Byrte', 'lp_plant_power_fraction_Vinje', 'lp_storage_fill_Vinje', 'lp_storage_min_slack_Vinje', 'lp_storage_max_slack_Vinje', 'lp_plant_power_fraction_Haukeli', 'lp_storage_fill_Haukeli', 'lp_storage_min_slack_Haukeli', 'lp_storage_max_slack_Haukeli', 'lp_plant_power_fraction_Songa', 'lp_storage_fill_Songa', 'lp_storage_min_slack_Songa', 'lp_storage_max_slack_Songa', 'lp_plant_power_fraction_Kjela', 'lp_storage_fill_Kjela', 'lp_storage_min_slack_Kjela', 'lp_storage_max_slack_Kjela', 'lp_plant_power_fraction_Vesle Kjela', 'lp_storage_fill_Vesle Kjela', 'lp_storage_min_slack_Vesle Kjela', 'lp_storage_max_slack_Vesle Kjela', 'lp_context_qmin_previous_1h_Hogga_G1', 'lp_context_qmin_next_1h_Hogga_G1', 'lp_context_qmin_previous_3h_Hogga_G1', 'lp_context_qmin_next_3h_Hogga_G1', 'lp_context_qmin_previous_6h_mean_Hogga_G1', 'lp_context_qmin_next_6h_mean_Hogga_G1', 'lp_context_qmin_week_mean_Hogga_G1', 'lp_context_qmin_week_at_least_one_fraction_Hogga_G1', 'lp_context_qmin_week_positive_fraction_Hogga_G1', 'lp_context_qmin_previous_1h_Tokke_G1', 'lp_context_qmin_next_1h_Tokke_G1', 'lp_context_qmin_previous_3h_Tokke_G1', 'lp_context_qmin_next_3h_Tokke_G1', 'lp_context_qmin_previous_6h_mean_Tokke_G1', 'lp_context_qmin_next_6h_mean_Tokke_G1', 'lp_context_qmin_week_mean_Tokke_G1', 'lp_context_qmin_week_at_least_one_fraction_Tokke_G1', 'lp_context_qmin_week_positive_fraction_Tokke_G1', 'lp_context_qmin_previous_1h_Tokke_G2', 'lp_context_qmin_next_1h_Tokke_G2', 'lp_context_qmin_previous_3h_Tokke_G2', 'lp_context_qmin_next_3h_Tokke_G2', 'lp_context_qmin_previous_6h_mean_Tokke_G2', 'lp_context_qmin_next_6h_mean_Tokke_G2', 'lp_context_qmin_week_mean_Tokke_G2', 'lp_context_qmin_week_at_least_one_fraction_Tokke_G2', 'lp_context_qmin_week_positive_fraction_Tokke_G2', 'lp_context_qmin_previous_1h_Tokke_G3', 'lp_context_qmin_next_1h_Tokke_G3', 'lp_context_qmin_previous_3h_Tokke_G3', 'lp_context_qmin_next_3h_Tokke_G3', 'lp_context_qmin_previous_6h_mean_Tokke_G3', 'lp_context_qmin_next_6h_mean_Tokke_G3', 'lp_context_qmin_week_mean_Tokke_G3', 'lp_context_qmin_week_at_least_one_fraction_Tokke_G3', 'lp_context_qmin_week_positive_fraction_Tokke_G3', 'lp_context_qmin_previous_1h_Tokke_G4', 'lp_context_qmin_next_1h_Tokke_G4', 'lp_context_qmin_previous_3h_Tokke_G4', 'lp_context_qmin_next_3h_Tokke_G4', 'lp_context_qmin_previous_6h_mean_Tokke_G4', 'lp_context_qmin_next_6h_mean_Tokke_G4', 'lp_context_qmin_week_mean_Tokke_G4', 'lp_context_qmin_week_at_least_one_fraction_Tokke_G4', 'lp_context_qmin_week_positive_fraction_Tokke_G4', 'lp_context_qmin_previous_1h_Lio_G1', 'lp_context_qmin_next_1h_Lio_G1', 'lp_context_qmin_previous_3h_Lio_G1', 'lp_context_qmin_next_3h_Lio_G1', 'lp_context_qmin_previous_6h_mean_Lio_G1', 'lp_context_qmin_next_6h_mean_Lio_G1', 'lp_context_qmin_week_mean_Lio_G1', 'lp_context_qmin_week_at_least_one_fraction_Lio_G1', 'lp_context_qmin_week_positive_fraction_Lio_G1', 'lp_context_qmin_previous_1h_Byrte_G1', 'lp_context_qmin_next_1h_Byrte_G1', 'lp_context_qmin_previous_3h_Byrte_G1', 'lp_context_qmin_next_3h_Byrte_G1', 'lp_context_qmin_previous_6h_mean_Byrte_G1', 'lp_context_qmin_next_6h_mean_Byrte_G1', 'lp_context_qmin_week_mean_Byrte_G1', 'lp_context_qmin_week_at_least_one_fraction_Byrte_G1', 'lp_context_qmin_week_positive_fraction_Byrte_G1', 'lp_context_qmin_previous_1h_Vinje_G1', 'lp_context_qmin_next_1h_Vinje_G1', 'lp_context_qmin_previous_3h_Vinje_G1', 'lp_context_qmin_next_3h_Vinje_G1', 'lp_context_qmin_previous_6h_mean_Vinje_G1', 'lp_context_qmin_next_6h_mean_Vinje_G1', 'lp_context_qmin_week_mean_Vinje_G1', 'lp_context_qmin_week_at_least_one_fraction_Vinje_G1', 'lp_context_qmin_week_positive_fraction_Vinje_G1', 'lp_context_qmin_previous_1h_Vinje_G2', 'lp_context_qmin_next_1h_Vinje_G2', 'lp_context_qmin_previous_3h_Vinje_G2', 'lp_context_qmin_next_3h_Vinje_G2', 'lp_context_qmin_previous_6h_mean_Vinje_G2', 'lp_context_qmin_next_6h_mean_Vinje_G2', 'lp_context_qmin_week_mean_Vinje_G2', 'lp_context_qmin_week_at_least_one_fraction_Vinje_G2', 'lp_context_qmin_week_positive_fraction_Vinje_G2', 'lp_context_qmin_previous_1h_Vinje_G3', 'lp_context_qmin_next_1h_Vinje_G3', 'lp_context_qmin_previous_3h_Vinje_G3', 'lp_context_qmin_next_3h_Vinje_G3', 'lp_context_qmin_previous_6h_mean_Vinje_G3', 'lp_context_qmin_next_6h_mean_Vinje_G3', 'lp_context_qmin_week_mean_Vinje_G3', 'lp_context_qmin_week_at_least_one_fraction_Vinje_G3', 'lp_context_qmin_week_positive_fraction_Vinje_G3', 'lp_context_qmin_previous_1h_Haukeli_G1', 'lp_context_qmin_next_1h_Haukeli_G1', 'lp_context_qmin_previous_3h_Haukeli_G1', 'lp_context_qmin_next_3h_Haukeli_G1', 'lp_context_qmin_previous_6h_mean_Haukeli_G1', 'lp_context_qmin_next_6h_mean_Haukeli_G1', 'lp_context_qmin_week_mean_Haukeli_G1', 'lp_context_qmin_week_at_least_one_fraction_Haukeli_G1', 'lp_context_qmin_week_positive_fraction_Haukeli_G1', 'lp_context_qmin_previous_1h_Songa_G1', 'lp_context_qmin_next_1h_Songa_G1', 'lp_context_qmin_previous_3h_Songa_G1', 'lp_context_qmin_next_3h_Songa_G1', 'lp_context_qmin_previous_6h_mean_Songa_G1', 'lp_context_qmin_next_6h_mean_Songa_G1', 'lp_context_qmin_week_mean_Songa_G1', 'lp_context_qmin_week_at_least_one_fraction_Songa_G1', 'lp_context_qmin_week_positive_fraction_Songa_G1', 'lp_context_qmin_previous_1h_Kjela_G1', 'lp_context_qmin_next_1h_Kjela_G1', 'lp_context_qmin_previous_3h_Kjela_G1', 'lp_context_qmin_next_3h_Kjela_G1', 'lp_context_qmin_previous_6h_mean_Kjela_G1', 'lp_context_qmin_next_6h_mean_Kjela_G1', 'lp_context_qmin_week_mean_Kjela_G1', 'lp_context_qmin_week_at_least_one_fraction_Kjela_G1', 'lp_context_qmin_week_positive_fraction_Kjela_G1', 'lp_context_qmin_previous_1h_Vesle Kjela_G1', 'lp_context_qmin_next_1h_Vesle Kjela_G1', 'lp_context_qmin_previous_3h_Vesle Kjela_G1', 'lp_context_qmin_next_3h_Vesle Kjela_G1', 'lp_context_qmin_previous_6h_mean_Vesle Kjela_G1', 'lp_context_qmin_next_6h_mean_Vesle Kjela_G1', 'lp_context_qmin_week_mean_Vesle Kjela_G1', 'lp_context_qmin_week_at_least_one_fraction_Vesle Kjela_G1', 'lp_context_qmin_week_positive_fraction_Vesle Kjela_G1', 'norm_price', 'norm_price_offset_-24', 'norm_price_offset_-12', 'norm_price_offset_-6', 'norm_price_offset_-3', 'norm_price_offset_-1', 'norm_price_offset_+1', 'norm_price_offset_+3', 'norm_price_offset_+6', 'norm_price_offset_+12', 'norm_price_offset_+24', 'norm_price_week_mean', 'norm_price_week_std', 'norm_price_week_min', 'norm_price_week_max', 'norm_price_week_q10', 'norm_price_week_q25', 'norm_price_week_q50', 'norm_price_week_q75', 'norm_price_week_q90', 'norm_price_minus_week_mean', 'norm_price_day_mean', 'norm_price_day_min', 'norm_price_day_max', 'norm_price_day_std', 'norm_price_remaining_mean', 'norm_price_remaining_max', 'norm_water_value_Kjelavatn', 'norm_price_minus_water_Kjelavatn', 'norm_water_value_Staavatn', 'norm_price_minus_water_Staavatn', 'norm_water_value_Langesae', 'norm_price_minus_water_Langesae', 'norm_water_value_Foersvatn', 'norm_price_minus_water_Foersvatn', 'norm_water_value_Venemo', 'norm_price_minus_water_Venemo', 'norm_water_value_Botnedalsvatn', 'norm_price_minus_water_Botnedalsvatn', 'norm_water_value_Bitdalsvatn', 'norm_price_minus_water_Bitdalsvatn', 'norm_water_value_Vaamarvatn', 'norm_price_minus_water_Vaamarvatn', 'norm_water_value_Langeidvatn', 'norm_price_minus_water_Langeidvatn', 'norm_water_value_Totak', 'norm_price_minus_water_Totak', 'norm_water_value_Songavatn', 'norm_price_minus_water_Songavatn', 'norm_water_value_Vinjevatn', 'norm_price_minus_water_Vinjevatn', 'norm_water_value_Bandak', 'norm_price_minus_water_Bandak', 'norm_water_value_Bordalsvatn', 'norm_price_minus_water_Bordalsvatn', 'norm_water_value_Byrtevatn', 'norm_price_minus_water_Byrtevatn', 'norm_water_value_Hyljelihyl', 'norm_price_minus_water_Hyljelihyl', 'norm_water_value_Vatjern', 'norm_price_minus_water_Vatjern', 'norm_physics_switch_margin_minimum_Hogga', 'norm_physics_switch_margin_maximum_Hogga', 'norm_physics_switch_margin_minimum_Tokke', 'norm_physics_switch_margin_maximum_Tokke', 'norm_physics_switch_margin_minimum_Lio', 'norm_physics_switch_margin_maximum_Lio', 'norm_physics_switch_margin_minimum_Byrte', 'norm_physics_switch_margin_maximum_Byrte', 'norm_physics_switch_margin_minimum_Vinje', 'norm_physics_switch_margin_maximum_Vinje', 'norm_physics_switch_margin_minimum_Haukeli', 'norm_physics_switch_margin_maximum_Haukeli', 'norm_physics_switch_margin_minimum_Songa', 'norm_physics_switch_margin_maximum_Songa', 'norm_physics_switch_margin_minimum_Kjela', 'norm_physics_switch_margin_maximum_Kjela', 'norm_physics_switch_margin_minimum_Vesle Kjela', 'norm_physics_switch_margin_maximum_Vesle Kjela', 'norm_price_previous_mean_6h', 'norm_price_minus_previous_mean_6h', 'norm_price_next_mean_6h', 'norm_price_minus_next_mean_6h', 'norm_price_previous_mean_12h', 'norm_price_minus_previous_mean_12h', 'norm_price_next_mean_12h', 'norm_price_minus_next_mean_12h', 'norm_price_previous_mean_24h', 'norm_price_minus_previous_mean_24h', 'norm_price_next_mean_24h', 'norm_price_minus_next_mean_24h', 'monetary_scale_inverse'], 'joint': ['horizon_hour', 'horizon_day', 'hour_of_day', 'hour_sin', 'hour_cos', 'weekday', 'month', 'day_of_year', 'season_sin', 'season_cos', 'price_week_zscore', 'price_week_rank', 'volume_Kjelavatn', 'fill_Kjelavatn', 'price_over_water_Kjelavatn', 'headroom_after_inflow_Kjelavatn', 'volume_Staavatn', 'fill_Staavatn', 'price_over_water_Staavatn', 'headroom_after_inflow_Staavatn', 'volume_Langesae', 'fill_Langesae', 'price_over_water_Langesae', 'headroom_after_inflow_Langesae', 'volume_Foersvatn', 'fill_Foersvatn', 'price_over_water_Foersvatn', 'headroom_after_inflow_Foersvatn', 'volume_Venemo', 'fill_Venemo', 'price_over_water_Venemo', 'headroom_after_inflow_Venemo', 'volume_Botnedalsvatn', 'fill_Botnedalsvatn', 'price_over_water_Botnedalsvatn', 'headroom_after_inflow_Botnedalsvatn', 'volume_Bitdalsvatn', 'fill_Bitdalsvatn', 'price_over_water_Bitdalsvatn', 'headroom_after_inflow_Bitdalsvatn', 'volume_Vaamarvatn', 'fill_Vaamarvatn', 'price_over_water_Vaamarvatn', 'headroom_after_inflow_Vaamarvatn', 'volume_Langeidvatn', 'fill_Langeidvatn', 'price_over_water_Langeidvatn', 'headroom_after_inflow_Langeidvatn', 'volume_Totak', 'fill_Totak', 'price_over_water_Totak', 'headroom_after_inflow_Totak', 'volume_Songavatn', 'fill_Songavatn', 'price_over_water_Songavatn', 'headroom_after_inflow_Songavatn', 'volume_Vinjevatn', 'fill_Vinjevatn', 'price_over_water_Vinjevatn', 'headroom_after_inflow_Vinjevatn', 'volume_Bandak', 'fill_Bandak', 'price_over_water_Bandak', 'headroom_after_inflow_Bandak', 'volume_Bordalsvatn', 'fill_Bordalsvatn', 'price_over_water_Bordalsvatn', 'headroom_after_inflow_Bordalsvatn', 'volume_Byrtevatn', 'fill_Byrtevatn', 'price_over_water_Byrtevatn', 'headroom_after_inflow_Byrtevatn', 'volume_Hyljelihyl', 'fill_Hyljelihyl', 'price_over_water_Hyljelihyl', 'headroom_after_inflow_Hyljelihyl', 'volume_Vatjern', 'fill_Vatjern', 'price_over_water_Vatjern', 'headroom_after_inflow_Vatjern', 'inflow_Langesae', 'inflow_week_mean_Langesae', 'inflow_elapsed_mm3_Langesae', 'inflow_remaining_mm3_Langesae', 'inflow_Staavatn', 'inflow_week_mean_Staavatn', 'inflow_elapsed_mm3_Staavatn', 'inflow_remaining_mm3_Staavatn', 'inflow_Kjelavatn', 'inflow_week_mean_Kjelavatn', 'inflow_elapsed_mm3_Kjelavatn', 'inflow_remaining_mm3_Kjelavatn', 'inflow_Bordalsvatn', 'inflow_week_mean_Bordalsvatn', 'inflow_elapsed_mm3_Bordalsvatn', 'inflow_remaining_mm3_Bordalsvatn', 'inflow_Foersvatn', 'inflow_week_mean_Foersvatn', 'inflow_elapsed_mm3_Foersvatn', 'inflow_remaining_mm3_Foersvatn', 'inflow_Hyljelihyl', 'inflow_week_mean_Hyljelihyl', 'inflow_elapsed_mm3_Hyljelihyl', 'inflow_remaining_mm3_Hyljelihyl', 'inflow_Venemo', 'inflow_week_mean_Venemo', 'inflow_elapsed_mm3_Venemo', 'inflow_remaining_mm3_Venemo', 'inflow_Bitdalsvatn', 'inflow_week_mean_Bitdalsvatn', 'inflow_elapsed_mm3_Bitdalsvatn', 'inflow_remaining_mm3_Bitdalsvatn', 'inflow_Songavatn', 'inflow_week_mean_Songavatn', 'inflow_elapsed_mm3_Songavatn', 'inflow_remaining_mm3_Songavatn', 'inflow_Totak', 'inflow_week_mean_Totak', 'inflow_elapsed_mm3_Totak', 'inflow_remaining_mm3_Totak', 'inflow_Vaamarvatn', 'inflow_week_mean_Vaamarvatn', 'inflow_elapsed_mm3_Vaamarvatn', 'inflow_remaining_mm3_Vaamarvatn', 'inflow_Langeidvatn', 'inflow_week_mean_Langeidvatn', 'inflow_elapsed_mm3_Langeidvatn', 'inflow_remaining_mm3_Langeidvatn', 'inflow_Vatjern', 'inflow_week_mean_Vatjern', 'inflow_elapsed_mm3_Vatjern', 'inflow_remaining_mm3_Vatjern', 'inflow_Vinjevatn', 'inflow_week_mean_Vinjevatn', 'inflow_elapsed_mm3_Vinjevatn', 'inflow_remaining_mm3_Vinjevatn', 'inflow_Botnedalsvatn', 'inflow_week_mean_Botnedalsvatn', 'inflow_elapsed_mm3_Botnedalsvatn', 'inflow_remaining_mm3_Botnedalsvatn', 'inflow_Byrtevatn', 'inflow_week_mean_Byrtevatn', 'inflow_elapsed_mm3_Byrtevatn', 'inflow_remaining_mm3_Byrtevatn', 'inflow_Bandak', 'inflow_week_mean_Bandak', 'inflow_elapsed_mm3_Bandak', 'inflow_remaining_mm3_Bandak', 'inflow_r_Vest_Vassdraget', 'inflow_week_mean_r_Vest_Vassdraget', 'inflow_elapsed_mm3_r_Vest_Vassdraget', 'inflow_remaining_mm3_r_Vest_Vassdraget', 'min_volume_Totak', 'min_volume_week_max_Totak', 'min_volume_Byrtevatn', 'min_volume_week_max_Byrtevatn', 'min_volume_Staavatn', 'min_volume_week_max_Staavatn', 'min_flow_b_Kjelavatn', 'min_flow_week_max_b_Kjelavatn', 'min_flow_b_Hyljelihyl', 'min_flow_week_max_b_Hyljelihyl', 'min_flow_b_Vinjevatn', 'min_flow_week_max_b_Vinjevatn', 'min_flow_b_Byrtevatn', 'min_flow_week_max_b_Byrtevatn', 'min_flow_b_Bandak', 'min_flow_week_max_b_Bandak', 'min_flow_r_Vest_Vassdraget', 'min_flow_week_max_r_Vest_Vassdraget', 'physics_head_Hogga', 'physics_pool_fill_Hogga', 'physics_supply_fraction_Hogga', 'physics_catchment_inflow_fraction_Hogga', 'physics_dispatch_to_avoid_spill_Hogga', 'physics_profitable_remaining_fraction_Hogga', 'physics_head_Tokke', 'physics_pool_fill_Tokke', 'physics_supply_fraction_Tokke', 'physics_catchment_inflow_fraction_Tokke', 'physics_dispatch_to_avoid_spill_Tokke', 'physics_profitable_remaining_fraction_Tokke', 'physics_head_Lio', 'physics_pool_fill_Lio', 'physics_supply_fraction_Lio', 'physics_catchment_inflow_fraction_Lio', 'physics_dispatch_to_avoid_spill_Lio', 'physics_profitable_remaining_fraction_Lio', 'physics_head_Byrte', 'physics_pool_fill_Byrte', 'physics_supply_fraction_Byrte', 'physics_catchment_inflow_fraction_Byrte', 'physics_dispatch_to_avoid_spill_Byrte', 'physics_profitable_remaining_fraction_Byrte', 'physics_head_Vinje', 'physics_pool_fill_Vinje', 'physics_supply_fraction_Vinje', 'physics_catchment_inflow_fraction_Vinje', 'physics_dispatch_to_avoid_spill_Vinje', 'physics_profitable_remaining_fraction_Vinje', 'physics_head_Haukeli', 'physics_pool_fill_Haukeli', 'physics_supply_fraction_Haukeli', 'physics_catchment_inflow_fraction_Haukeli', 'physics_dispatch_to_avoid_spill_Haukeli', 'physics_profitable_remaining_fraction_Haukeli', 'physics_head_Songa', 'physics_pool_fill_Songa', 'physics_supply_fraction_Songa', 'physics_catchment_inflow_fraction_Songa', 'physics_dispatch_to_avoid_spill_Songa', 'physics_profitable_remaining_fraction_Songa', 'physics_head_Kjela', 'physics_pool_fill_Kjela', 'physics_supply_fraction_Kjela', 'physics_catchment_inflow_fraction_Kjela', 'physics_dispatch_to_avoid_spill_Kjela', 'physics_profitable_remaining_fraction_Kjela', 'physics_head_Vesle Kjela', 'physics_pool_fill_Vesle Kjela', 'physics_supply_fraction_Vesle Kjela', 'physics_catchment_inflow_fraction_Vesle Kjela', 'physics_dispatch_to_avoid_spill_Vesle Kjela', 'physics_profitable_remaining_fraction_Vesle Kjela', 'context_rank_remaining', 'lpjoint_qmin_ratio_Hogga_G1', 'lpjoint_power_fraction_Hogga_G1', 'lpjoint_economic_margin_Hogga_G1', 'lpjoint_qmin_ratio_Tokke_G1', 'lpjoint_power_fraction_Tokke_G1', 'lpjoint_economic_margin_Tokke_G1', 'lpjoint_qmin_ratio_Tokke_G2', 'lpjoint_power_fraction_Tokke_G2', 'lpjoint_economic_margin_Tokke_G2', 'lpjoint_qmin_ratio_Tokke_G3', 'lpjoint_power_fraction_Tokke_G3', 'lpjoint_economic_margin_Tokke_G3', 'lpjoint_qmin_ratio_Tokke_G4', 'lpjoint_power_fraction_Tokke_G4', 'lpjoint_economic_margin_Tokke_G4', 'lpjoint_qmin_ratio_Lio_G1', 'lpjoint_power_fraction_Lio_G1', 'lpjoint_economic_margin_Lio_G1', 'lpjoint_qmin_ratio_Byrte_G1', 'lpjoint_power_fraction_Byrte_G1', 'lpjoint_economic_margin_Byrte_G1', 'lpjoint_qmin_ratio_Vinje_G1', 'lpjoint_power_fraction_Vinje_G1', 'lpjoint_economic_margin_Vinje_G1', 'lpjoint_qmin_ratio_Vinje_G2', 'lpjoint_power_fraction_Vinje_G2', 'lpjoint_economic_margin_Vinje_G2', 'lpjoint_qmin_ratio_Vinje_G3', 'lpjoint_power_fraction_Vinje_G3', 'lpjoint_economic_margin_Vinje_G3', 'lpjoint_qmin_ratio_Haukeli_G1', 'lpjoint_power_fraction_Haukeli_G1', 'lpjoint_economic_margin_Haukeli_G1', 'lpjoint_qmin_ratio_Songa_G1', 'lpjoint_power_fraction_Songa_G1', 'lpjoint_economic_margin_Songa_G1', 'lpjoint_qmin_ratio_Kjela_G1', 'lpjoint_power_fraction_Kjela_G1', 'lpjoint_economic_margin_Kjela_G1', 'lpjoint_qmin_ratio_Vesle Kjela_G1', 'lpjoint_power_fraction_Vesle Kjela_G1', 'lpjoint_economic_margin_Vesle Kjela_G1', 'lpjoint_plant_power_fraction_Hogga', 'lpjoint_storage_fill_Hogga', 'lpjoint_storage_min_slack_Hogga', 'lpjoint_storage_max_slack_Hogga', 'lpjoint_plant_power_fraction_Tokke', 'lpjoint_storage_fill_Tokke', 'lpjoint_storage_min_slack_Tokke', 'lpjoint_storage_max_slack_Tokke', 'lpjoint_plant_power_fraction_Lio', 'lpjoint_storage_fill_Lio', 'lpjoint_storage_min_slack_Lio', 'lpjoint_storage_max_slack_Lio', 'lpjoint_plant_power_fraction_Byrte', 'lpjoint_storage_fill_Byrte', 'lpjoint_storage_min_slack_Byrte', 'lpjoint_storage_max_slack_Byrte', 'lpjoint_plant_power_fraction_Vinje', 'lpjoint_storage_fill_Vinje', 'lpjoint_storage_min_slack_Vinje', 'lpjoint_storage_max_slack_Vinje', 'lpjoint_plant_power_fraction_Haukeli', 'lpjoint_storage_fill_Haukeli', 'lpjoint_storage_min_slack_Haukeli', 'lpjoint_storage_max_slack_Haukeli', 'lpjoint_plant_power_fraction_Songa', 'lpjoint_storage_fill_Songa', 'lpjoint_storage_min_slack_Songa', 'lpjoint_storage_max_slack_Songa', 'lpjoint_plant_power_fraction_Kjela', 'lpjoint_storage_fill_Kjela', 'lpjoint_storage_min_slack_Kjela', 'lpjoint_storage_max_slack_Kjela', 'lpjoint_plant_power_fraction_Vesle Kjela', 'lpjoint_storage_fill_Vesle Kjela', 'lpjoint_storage_min_slack_Vesle Kjela', 'lpjoint_storage_max_slack_Vesle Kjela', 'lpjoint_on_fraction_Tokke_G1', 'lpjoint_on_fraction_Tokke_G2', 'lpjoint_on_fraction_Tokke_G3', 'lpjoint_on_fraction_Tokke_G4', 'lpjoint_on_fraction_Vinje_G1', 'lpjoint_on_fraction_Vinje_G2', 'lpjoint_on_fraction_Vinje_G3', 'lpjoint_active_unit_count_Tokke', 'lpjoint_active_unit_count_Vinje', 'norm_price', 'norm_price_offset_-24', 'norm_price_offset_-12', 'norm_price_offset_-6', 'norm_price_offset_-3', 'norm_price_offset_-1', 'norm_price_offset_+1', 'norm_price_offset_+3', 'norm_price_offset_+6', 'norm_price_offset_+12', 'norm_price_offset_+24', 'norm_price_week_mean', 'norm_price_week_std', 'norm_price_week_min', 'norm_price_week_max', 'norm_price_week_q10', 'norm_price_week_q25', 'norm_price_week_q50', 'norm_price_week_q75', 'norm_price_week_q90', 'norm_price_minus_week_mean', 'norm_price_day_mean', 'norm_price_day_min', 'norm_price_day_max', 'norm_price_day_std', 'norm_price_remaining_mean', 'norm_price_remaining_max', 'norm_water_value_Kjelavatn', 'norm_price_minus_water_Kjelavatn', 'norm_water_value_Staavatn', 'norm_price_minus_water_Staavatn', 'norm_water_value_Langesae', 'norm_price_minus_water_Langesae', 'norm_water_value_Foersvatn', 'norm_price_minus_water_Foersvatn', 'norm_water_value_Venemo', 'norm_price_minus_water_Venemo', 'norm_water_value_Botnedalsvatn', 'norm_price_minus_water_Botnedalsvatn', 'norm_water_value_Bitdalsvatn', 'norm_price_minus_water_Bitdalsvatn', 'norm_water_value_Vaamarvatn', 'norm_price_minus_water_Vaamarvatn', 'norm_water_value_Langeidvatn', 'norm_price_minus_water_Langeidvatn', 'norm_water_value_Totak', 'norm_price_minus_water_Totak', 'norm_water_value_Songavatn', 'norm_price_minus_water_Songavatn', 'norm_water_value_Vinjevatn', 'norm_price_minus_water_Vinjevatn', 'norm_water_value_Bandak', 'norm_price_minus_water_Bandak', 'norm_water_value_Bordalsvatn', 'norm_price_minus_water_Bordalsvatn', 'norm_water_value_Byrtevatn', 'norm_price_minus_water_Byrtevatn', 'norm_water_value_Hyljelihyl', 'norm_price_minus_water_Hyljelihyl', 'norm_water_value_Vatjern', 'norm_price_minus_water_Vatjern', 'norm_physics_switch_margin_minimum_Hogga', 'norm_physics_switch_margin_maximum_Hogga', 'norm_physics_switch_margin_minimum_Tokke', 'norm_physics_switch_margin_maximum_Tokke', 'norm_physics_switch_margin_minimum_Lio', 'norm_physics_switch_margin_maximum_Lio', 'norm_physics_switch_margin_minimum_Byrte', 'norm_physics_switch_margin_maximum_Byrte', 'norm_physics_switch_margin_minimum_Vinje', 'norm_physics_switch_margin_maximum_Vinje', 'norm_physics_switch_margin_minimum_Haukeli', 'norm_physics_switch_margin_maximum_Haukeli', 'norm_physics_switch_margin_minimum_Songa', 'norm_physics_switch_margin_maximum_Songa', 'norm_physics_switch_margin_minimum_Kjela', 'norm_physics_switch_margin_maximum_Kjela', 'norm_physics_switch_margin_minimum_Vesle Kjela', 'norm_physics_switch_margin_maximum_Vesle Kjela', 'norm_price_previous_mean_6h', 'norm_price_minus_previous_mean_6h', 'norm_price_next_mean_6h', 'norm_price_minus_next_mean_6h', 'norm_price_previous_mean_12h', 'norm_price_minus_previous_mean_12h', 'norm_price_next_mean_12h', 'norm_price_minus_next_mean_12h', 'norm_price_previous_mean_24h', 'norm_price_minus_previous_mean_24h', 'norm_price_next_mean_24h', 'norm_price_minus_next_mean_24h', 'monetary_scale_inverse'], 'context_joint': ['horizon_hour', 'horizon_day', 'hour_of_day', 'hour_sin', 'hour_cos', 'weekday', 'month', 'day_of_year', 'season_sin', 'season_cos', 'price_week_zscore', 'price_week_rank', 'volume_Kjelavatn', 'fill_Kjelavatn', 'price_over_water_Kjelavatn', 'headroom_after_inflow_Kjelavatn', 'volume_Staavatn', 'fill_Staavatn', 'price_over_water_Staavatn', 'headroom_after_inflow_Staavatn', 'volume_Langesae', 'fill_Langesae', 'price_over_water_Langesae', 'headroom_after_inflow_Langesae', 'volume_Foersvatn', 'fill_Foersvatn', 'price_over_water_Foersvatn', 'headroom_after_inflow_Foersvatn', 'volume_Venemo', 'fill_Venemo', 'price_over_water_Venemo', 'headroom_after_inflow_Venemo', 'volume_Botnedalsvatn', 'fill_Botnedalsvatn', 'price_over_water_Botnedalsvatn', 'headroom_after_inflow_Botnedalsvatn', 'volume_Bitdalsvatn', 'fill_Bitdalsvatn', 'price_over_water_Bitdalsvatn', 'headroom_after_inflow_Bitdalsvatn', 'volume_Vaamarvatn', 'fill_Vaamarvatn', 'price_over_water_Vaamarvatn', 'headroom_after_inflow_Vaamarvatn', 'volume_Langeidvatn', 'fill_Langeidvatn', 'price_over_water_Langeidvatn', 'headroom_after_inflow_Langeidvatn', 'volume_Totak', 'fill_Totak', 'price_over_water_Totak', 'headroom_after_inflow_Totak', 'volume_Songavatn', 'fill_Songavatn', 'price_over_water_Songavatn', 'headroom_after_inflow_Songavatn', 'volume_Vinjevatn', 'fill_Vinjevatn', 'price_over_water_Vinjevatn', 'headroom_after_inflow_Vinjevatn', 'volume_Bandak', 'fill_Bandak', 'price_over_water_Bandak', 'headroom_after_inflow_Bandak', 'volume_Bordalsvatn', 'fill_Bordalsvatn', 'price_over_water_Bordalsvatn', 'headroom_after_inflow_Bordalsvatn', 'volume_Byrtevatn', 'fill_Byrtevatn', 'price_over_water_Byrtevatn', 'headroom_after_inflow_Byrtevatn', 'volume_Hyljelihyl', 'fill_Hyljelihyl', 'price_over_water_Hyljelihyl', 'headroom_after_inflow_Hyljelihyl', 'volume_Vatjern', 'fill_Vatjern', 'price_over_water_Vatjern', 'headroom_after_inflow_Vatjern', 'inflow_Langesae', 'inflow_week_mean_Langesae', 'inflow_elapsed_mm3_Langesae', 'inflow_remaining_mm3_Langesae', 'inflow_Staavatn', 'inflow_week_mean_Staavatn', 'inflow_elapsed_mm3_Staavatn', 'inflow_remaining_mm3_Staavatn', 'inflow_Kjelavatn', 'inflow_week_mean_Kjelavatn', 'inflow_elapsed_mm3_Kjelavatn', 'inflow_remaining_mm3_Kjelavatn', 'inflow_Bordalsvatn', 'inflow_week_mean_Bordalsvatn', 'inflow_elapsed_mm3_Bordalsvatn', 'inflow_remaining_mm3_Bordalsvatn', 'inflow_Foersvatn', 'inflow_week_mean_Foersvatn', 'inflow_elapsed_mm3_Foersvatn', 'inflow_remaining_mm3_Foersvatn', 'inflow_Hyljelihyl', 'inflow_week_mean_Hyljelihyl', 'inflow_elapsed_mm3_Hyljelihyl', 'inflow_remaining_mm3_Hyljelihyl', 'inflow_Venemo', 'inflow_week_mean_Venemo', 'inflow_elapsed_mm3_Venemo', 'inflow_remaining_mm3_Venemo', 'inflow_Bitdalsvatn', 'inflow_week_mean_Bitdalsvatn', 'inflow_elapsed_mm3_Bitdalsvatn', 'inflow_remaining_mm3_Bitdalsvatn', 'inflow_Songavatn', 'inflow_week_mean_Songavatn', 'inflow_elapsed_mm3_Songavatn', 'inflow_remaining_mm3_Songavatn', 'inflow_Totak', 'inflow_week_mean_Totak', 'inflow_elapsed_mm3_Totak', 'inflow_remaining_mm3_Totak', 'inflow_Vaamarvatn', 'inflow_week_mean_Vaamarvatn', 'inflow_elapsed_mm3_Vaamarvatn', 'inflow_remaining_mm3_Vaamarvatn', 'inflow_Langeidvatn', 'inflow_week_mean_Langeidvatn', 'inflow_elapsed_mm3_Langeidvatn', 'inflow_remaining_mm3_Langeidvatn', 'inflow_Vatjern', 'inflow_week_mean_Vatjern', 'inflow_elapsed_mm3_Vatjern', 'inflow_remaining_mm3_Vatjern', 'inflow_Vinjevatn', 'inflow_week_mean_Vinjevatn', 'inflow_elapsed_mm3_Vinjevatn', 'inflow_remaining_mm3_Vinjevatn', 'inflow_Botnedalsvatn', 'inflow_week_mean_Botnedalsvatn', 'inflow_elapsed_mm3_Botnedalsvatn', 'inflow_remaining_mm3_Botnedalsvatn', 'inflow_Byrtevatn', 'inflow_week_mean_Byrtevatn', 'inflow_elapsed_mm3_Byrtevatn', 'inflow_remaining_mm3_Byrtevatn', 'inflow_Bandak', 'inflow_week_mean_Bandak', 'inflow_elapsed_mm3_Bandak', 'inflow_remaining_mm3_Bandak', 'inflow_r_Vest_Vassdraget', 'inflow_week_mean_r_Vest_Vassdraget', 'inflow_elapsed_mm3_r_Vest_Vassdraget', 'inflow_remaining_mm3_r_Vest_Vassdraget', 'min_volume_Totak', 'min_volume_week_max_Totak', 'min_volume_Byrtevatn', 'min_volume_week_max_Byrtevatn', 'min_volume_Staavatn', 'min_volume_week_max_Staavatn', 'min_flow_b_Kjelavatn', 'min_flow_week_max_b_Kjelavatn', 'min_flow_b_Hyljelihyl', 'min_flow_week_max_b_Hyljelihyl', 'min_flow_b_Vinjevatn', 'min_flow_week_max_b_Vinjevatn', 'min_flow_b_Byrtevatn', 'min_flow_week_max_b_Byrtevatn', 'min_flow_b_Bandak', 'min_flow_week_max_b_Bandak', 'min_flow_r_Vest_Vassdraget', 'min_flow_week_max_r_Vest_Vassdraget', 'physics_head_Hogga', 'physics_pool_fill_Hogga', 'physics_supply_fraction_Hogga', 'physics_catchment_inflow_fraction_Hogga', 'physics_dispatch_to_avoid_spill_Hogga', 'physics_profitable_remaining_fraction_Hogga', 'physics_head_Tokke', 'physics_pool_fill_Tokke', 'physics_supply_fraction_Tokke', 'physics_catchment_inflow_fraction_Tokke', 'physics_dispatch_to_avoid_spill_Tokke', 'physics_profitable_remaining_fraction_Tokke', 'physics_head_Lio', 'physics_pool_fill_Lio', 'physics_supply_fraction_Lio', 'physics_catchment_inflow_fraction_Lio', 'physics_dispatch_to_avoid_spill_Lio', 'physics_profitable_remaining_fraction_Lio', 'physics_head_Byrte', 'physics_pool_fill_Byrte', 'physics_supply_fraction_Byrte', 'physics_catchment_inflow_fraction_Byrte', 'physics_dispatch_to_avoid_spill_Byrte', 'physics_profitable_remaining_fraction_Byrte', 'physics_head_Vinje', 'physics_pool_fill_Vinje', 'physics_supply_fraction_Vinje', 'physics_catchment_inflow_fraction_Vinje', 'physics_dispatch_to_avoid_spill_Vinje', 'physics_profitable_remaining_fraction_Vinje', 'physics_head_Haukeli', 'physics_pool_fill_Haukeli', 'physics_supply_fraction_Haukeli', 'physics_catchment_inflow_fraction_Haukeli', 'physics_dispatch_to_avoid_spill_Haukeli', 'physics_profitable_remaining_fraction_Haukeli', 'physics_head_Songa', 'physics_pool_fill_Songa', 'physics_supply_fraction_Songa', 'physics_catchment_inflow_fraction_Songa', 'physics_dispatch_to_avoid_spill_Songa', 'physics_profitable_remaining_fraction_Songa', 'physics_head_Kjela', 'physics_pool_fill_Kjela', 'physics_supply_fraction_Kjela', 'physics_catchment_inflow_fraction_Kjela', 'physics_dispatch_to_avoid_spill_Kjela', 'physics_profitable_remaining_fraction_Kjela', 'physics_head_Vesle Kjela', 'physics_pool_fill_Vesle Kjela', 'physics_supply_fraction_Vesle Kjela', 'physics_catchment_inflow_fraction_Vesle Kjela', 'physics_dispatch_to_avoid_spill_Vesle Kjela', 'physics_profitable_remaining_fraction_Vesle Kjela', 'context_rank_remaining', 'lp_qmin_ratio_Hogga_G1', 'lp_power_fraction_Hogga_G1', 'lp_economic_margin_Hogga_G1', 'lp_qmin_ratio_Tokke_G1', 'lp_power_fraction_Tokke_G1', 'lp_economic_margin_Tokke_G1', 'lp_qmin_ratio_Tokke_G2', 'lp_power_fraction_Tokke_G2', 'lp_economic_margin_Tokke_G2', 'lp_qmin_ratio_Tokke_G3', 'lp_power_fraction_Tokke_G3', 'lp_economic_margin_Tokke_G3', 'lp_qmin_ratio_Tokke_G4', 'lp_power_fraction_Tokke_G4', 'lp_economic_margin_Tokke_G4', 'lp_qmin_ratio_Lio_G1', 'lp_power_fraction_Lio_G1', 'lp_economic_margin_Lio_G1', 'lp_qmin_ratio_Byrte_G1', 'lp_power_fraction_Byrte_G1', 'lp_economic_margin_Byrte_G1', 'lp_qmin_ratio_Vinje_G1', 'lp_power_fraction_Vinje_G1', 'lp_economic_margin_Vinje_G1', 'lp_qmin_ratio_Vinje_G2', 'lp_power_fraction_Vinje_G2', 'lp_economic_margin_Vinje_G2', 'lp_qmin_ratio_Vinje_G3', 'lp_power_fraction_Vinje_G3', 'lp_economic_margin_Vinje_G3', 'lp_qmin_ratio_Haukeli_G1', 'lp_power_fraction_Haukeli_G1', 'lp_economic_margin_Haukeli_G1', 'lp_qmin_ratio_Songa_G1', 'lp_power_fraction_Songa_G1', 'lp_economic_margin_Songa_G1', 'lp_qmin_ratio_Kjela_G1', 'lp_power_fraction_Kjela_G1', 'lp_economic_margin_Kjela_G1', 'lp_qmin_ratio_Vesle Kjela_G1', 'lp_power_fraction_Vesle Kjela_G1', 'lp_economic_margin_Vesle Kjela_G1', 'lp_plant_power_fraction_Hogga', 'lp_storage_fill_Hogga', 'lp_storage_min_slack_Hogga', 'lp_storage_max_slack_Hogga', 'lp_plant_power_fraction_Tokke', 'lp_storage_fill_Tokke', 'lp_storage_min_slack_Tokke', 'lp_storage_max_slack_Tokke', 'lp_plant_power_fraction_Lio', 'lp_storage_fill_Lio', 'lp_storage_min_slack_Lio', 'lp_storage_max_slack_Lio', 'lp_plant_power_fraction_Byrte', 'lp_storage_fill_Byrte', 'lp_storage_min_slack_Byrte', 'lp_storage_max_slack_Byrte', 'lp_plant_power_fraction_Vinje', 'lp_storage_fill_Vinje', 'lp_storage_min_slack_Vinje', 'lp_storage_max_slack_Vinje', 'lp_plant_power_fraction_Haukeli', 'lp_storage_fill_Haukeli', 'lp_storage_min_slack_Haukeli', 'lp_storage_max_slack_Haukeli', 'lp_plant_power_fraction_Songa', 'lp_storage_fill_Songa', 'lp_storage_min_slack_Songa', 'lp_storage_max_slack_Songa', 'lp_plant_power_fraction_Kjela', 'lp_storage_fill_Kjela', 'lp_storage_min_slack_Kjela', 'lp_storage_max_slack_Kjela', 'lp_plant_power_fraction_Vesle Kjela', 'lp_storage_fill_Vesle Kjela', 'lp_storage_min_slack_Vesle Kjela', 'lp_storage_max_slack_Vesle Kjela', 'lp_context_qmin_previous_1h_Hogga_G1', 'lp_context_qmin_next_1h_Hogga_G1', 'lp_context_qmin_previous_3h_Hogga_G1', 'lp_context_qmin_next_3h_Hogga_G1', 'lp_context_qmin_previous_6h_mean_Hogga_G1', 'lp_context_qmin_next_6h_mean_Hogga_G1', 'lp_context_qmin_week_mean_Hogga_G1', 'lp_context_qmin_week_at_least_one_fraction_Hogga_G1', 'lp_context_qmin_week_positive_fraction_Hogga_G1', 'lp_context_qmin_previous_1h_Tokke_G1', 'lp_context_qmin_next_1h_Tokke_G1', 'lp_context_qmin_previous_3h_Tokke_G1', 'lp_context_qmin_next_3h_Tokke_G1', 'lp_context_qmin_previous_6h_mean_Tokke_G1', 'lp_context_qmin_next_6h_mean_Tokke_G1', 'lp_context_qmin_week_mean_Tokke_G1', 'lp_context_qmin_week_at_least_one_fraction_Tokke_G1', 'lp_context_qmin_week_positive_fraction_Tokke_G1', 'lp_context_qmin_previous_1h_Tokke_G2', 'lp_context_qmin_next_1h_Tokke_G2', 'lp_context_qmin_previous_3h_Tokke_G2', 'lp_context_qmin_next_3h_Tokke_G2', 'lp_context_qmin_previous_6h_mean_Tokke_G2', 'lp_context_qmin_next_6h_mean_Tokke_G2', 'lp_context_qmin_week_mean_Tokke_G2', 'lp_context_qmin_week_at_least_one_fraction_Tokke_G2', 'lp_context_qmin_week_positive_fraction_Tokke_G2', 'lp_context_qmin_previous_1h_Tokke_G3', 'lp_context_qmin_next_1h_Tokke_G3', 'lp_context_qmin_previous_3h_Tokke_G3', 'lp_context_qmin_next_3h_Tokke_G3', 'lp_context_qmin_previous_6h_mean_Tokke_G3', 'lp_context_qmin_next_6h_mean_Tokke_G3', 'lp_context_qmin_week_mean_Tokke_G3', 'lp_context_qmin_week_at_least_one_fraction_Tokke_G3', 'lp_context_qmin_week_positive_fraction_Tokke_G3', 'lp_context_qmin_previous_1h_Tokke_G4', 'lp_context_qmin_next_1h_Tokke_G4', 'lp_context_qmin_previous_3h_Tokke_G4', 'lp_context_qmin_next_3h_Tokke_G4', 'lp_context_qmin_previous_6h_mean_Tokke_G4', 'lp_context_qmin_next_6h_mean_Tokke_G4', 'lp_context_qmin_week_mean_Tokke_G4', 'lp_context_qmin_week_at_least_one_fraction_Tokke_G4', 'lp_context_qmin_week_positive_fraction_Tokke_G4', 'lp_context_qmin_previous_1h_Lio_G1', 'lp_context_qmin_next_1h_Lio_G1', 'lp_context_qmin_previous_3h_Lio_G1', 'lp_context_qmin_next_3h_Lio_G1', 'lp_context_qmin_previous_6h_mean_Lio_G1', 'lp_context_qmin_next_6h_mean_Lio_G1', 'lp_context_qmin_week_mean_Lio_G1', 'lp_context_qmin_week_at_least_one_fraction_Lio_G1', 'lp_context_qmin_week_positive_fraction_Lio_G1', 'lp_context_qmin_previous_1h_Byrte_G1', 'lp_context_qmin_next_1h_Byrte_G1', 'lp_context_qmin_previous_3h_Byrte_G1', 'lp_context_qmin_next_3h_Byrte_G1', 'lp_context_qmin_previous_6h_mean_Byrte_G1', 'lp_context_qmin_next_6h_mean_Byrte_G1', 'lp_context_qmin_week_mean_Byrte_G1', 'lp_context_qmin_week_at_least_one_fraction_Byrte_G1', 'lp_context_qmin_week_positive_fraction_Byrte_G1', 'lp_context_qmin_previous_1h_Vinje_G1', 'lp_context_qmin_next_1h_Vinje_G1', 'lp_context_qmin_previous_3h_Vinje_G1', 'lp_context_qmin_next_3h_Vinje_G1', 'lp_context_qmin_previous_6h_mean_Vinje_G1', 'lp_context_qmin_next_6h_mean_Vinje_G1', 'lp_context_qmin_week_mean_Vinje_G1', 'lp_context_qmin_week_at_least_one_fraction_Vinje_G1', 'lp_context_qmin_week_positive_fraction_Vinje_G1', 'lp_context_qmin_previous_1h_Vinje_G2', 'lp_context_qmin_next_1h_Vinje_G2', 'lp_context_qmin_previous_3h_Vinje_G2', 'lp_context_qmin_next_3h_Vinje_G2', 'lp_context_qmin_previous_6h_mean_Vinje_G2', 'lp_context_qmin_next_6h_mean_Vinje_G2', 'lp_context_qmin_week_mean_Vinje_G2', 'lp_context_qmin_week_at_least_one_fraction_Vinje_G2', 'lp_context_qmin_week_positive_fraction_Vinje_G2', 'lp_context_qmin_previous_1h_Vinje_G3', 'lp_context_qmin_next_1h_Vinje_G3', 'lp_context_qmin_previous_3h_Vinje_G3', 'lp_context_qmin_next_3h_Vinje_G3', 'lp_context_qmin_previous_6h_mean_Vinje_G3', 'lp_context_qmin_next_6h_mean_Vinje_G3', 'lp_context_qmin_week_mean_Vinje_G3', 'lp_context_qmin_week_at_least_one_fraction_Vinje_G3', 'lp_context_qmin_week_positive_fraction_Vinje_G3', 'lp_context_qmin_previous_1h_Haukeli_G1', 'lp_context_qmin_next_1h_Haukeli_G1', 'lp_context_qmin_previous_3h_Haukeli_G1', 'lp_context_qmin_next_3h_Haukeli_G1', 'lp_context_qmin_previous_6h_mean_Haukeli_G1', 'lp_context_qmin_next_6h_mean_Haukeli_G1', 'lp_context_qmin_week_mean_Haukeli_G1', 'lp_context_qmin_week_at_least_one_fraction_Haukeli_G1', 'lp_context_qmin_week_positive_fraction_Haukeli_G1', 'lp_context_qmin_previous_1h_Songa_G1', 'lp_context_qmin_next_1h_Songa_G1', 'lp_context_qmin_previous_3h_Songa_G1', 'lp_context_qmin_next_3h_Songa_G1', 'lp_context_qmin_previous_6h_mean_Songa_G1', 'lp_context_qmin_next_6h_mean_Songa_G1', 'lp_context_qmin_week_mean_Songa_G1', 'lp_context_qmin_week_at_least_one_fraction_Songa_G1', 'lp_context_qmin_week_positive_fraction_Songa_G1', 'lp_context_qmin_previous_1h_Kjela_G1', 'lp_context_qmin_next_1h_Kjela_G1', 'lp_context_qmin_previous_3h_Kjela_G1', 'lp_context_qmin_next_3h_Kjela_G1', 'lp_context_qmin_previous_6h_mean_Kjela_G1', 'lp_context_qmin_next_6h_mean_Kjela_G1', 'lp_context_qmin_week_mean_Kjela_G1', 'lp_context_qmin_week_at_least_one_fraction_Kjela_G1', 'lp_context_qmin_week_positive_fraction_Kjela_G1', 'lp_context_qmin_previous_1h_Vesle Kjela_G1', 'lp_context_qmin_next_1h_Vesle Kjela_G1', 'lp_context_qmin_previous_3h_Vesle Kjela_G1', 'lp_context_qmin_next_3h_Vesle Kjela_G1', 'lp_context_qmin_previous_6h_mean_Vesle Kjela_G1', 'lp_context_qmin_next_6h_mean_Vesle Kjela_G1', 'lp_context_qmin_week_mean_Vesle Kjela_G1', 'lp_context_qmin_week_at_least_one_fraction_Vesle Kjela_G1', 'lp_context_qmin_week_positive_fraction_Vesle Kjela_G1', 'norm_price', 'norm_price_offset_-24', 'norm_price_offset_-12', 'norm_price_offset_-6', 'norm_price_offset_-3', 'norm_price_offset_-1', 'norm_price_offset_+1', 'norm_price_offset_+3', 'norm_price_offset_+6', 'norm_price_offset_+12', 'norm_price_offset_+24', 'norm_price_week_mean', 'norm_price_week_std', 'norm_price_week_min', 'norm_price_week_max', 'norm_price_week_q10', 'norm_price_week_q25', 'norm_price_week_q50', 'norm_price_week_q75', 'norm_price_week_q90', 'norm_price_minus_week_mean', 'norm_price_day_mean', 'norm_price_day_min', 'norm_price_day_max', 'norm_price_day_std', 'norm_price_remaining_mean', 'norm_price_remaining_max', 'norm_water_value_Kjelavatn', 'norm_price_minus_water_Kjelavatn', 'norm_water_value_Staavatn', 'norm_price_minus_water_Staavatn', 'norm_water_value_Langesae', 'norm_price_minus_water_Langesae', 'norm_water_value_Foersvatn', 'norm_price_minus_water_Foersvatn', 'norm_water_value_Venemo', 'norm_price_minus_water_Venemo', 'norm_water_value_Botnedalsvatn', 'norm_price_minus_water_Botnedalsvatn', 'norm_water_value_Bitdalsvatn', 'norm_price_minus_water_Bitdalsvatn', 'norm_water_value_Vaamarvatn', 'norm_price_minus_water_Vaamarvatn', 'norm_water_value_Langeidvatn', 'norm_price_minus_water_Langeidvatn', 'norm_water_value_Totak', 'norm_price_minus_water_Totak', 'norm_water_value_Songavatn', 'norm_price_minus_water_Songavatn', 'norm_water_value_Vinjevatn', 'norm_price_minus_water_Vinjevatn', 'norm_water_value_Bandak', 'norm_price_minus_water_Bandak', 'norm_water_value_Bordalsvatn', 'norm_price_minus_water_Bordalsvatn', 'norm_water_value_Byrtevatn', 'norm_price_minus_water_Byrtevatn', 'norm_water_value_Hyljelihyl', 'norm_price_minus_water_Hyljelihyl', 'norm_water_value_Vatjern', 'norm_price_minus_water_Vatjern', 'norm_physics_switch_margin_minimum_Hogga', 'norm_physics_switch_margin_maximum_Hogga', 'norm_physics_switch_margin_minimum_Tokke', 'norm_physics_switch_margin_maximum_Tokke', 'norm_physics_switch_margin_minimum_Lio', 'norm_physics_switch_margin_maximum_Lio', 'norm_physics_switch_margin_minimum_Byrte', 'norm_physics_switch_margin_maximum_Byrte', 'norm_physics_switch_margin_minimum_Vinje', 'norm_physics_switch_margin_maximum_Vinje', 'norm_physics_switch_margin_minimum_Haukeli', 'norm_physics_switch_margin_maximum_Haukeli', 'norm_physics_switch_margin_minimum_Songa', 'norm_physics_switch_margin_maximum_Songa', 'norm_physics_switch_margin_minimum_Kjela', 'norm_physics_switch_margin_maximum_Kjela', 'norm_physics_switch_margin_minimum_Vesle Kjela', 'norm_physics_switch_margin_maximum_Vesle Kjela', 'norm_price_previous_mean_6h', 'norm_price_minus_previous_mean_6h', 'norm_price_next_mean_6h', 'norm_price_minus_next_mean_6h', 'norm_price_previous_mean_12h', 'norm_price_minus_previous_mean_12h', 'norm_price_next_mean_12h', 'norm_price_minus_next_mean_12h', 'norm_price_previous_mean_24h', 'norm_price_minus_previous_mean_24h', 'norm_price_next_mean_24h', 'norm_price_minus_next_mean_24h', 'monetary_scale_inverse', 'lpjoint_qmin_ratio_Hogga_G1', 'lpjoint_power_fraction_Hogga_G1', 'lpjoint_economic_margin_Hogga_G1', 'lpjoint_qmin_ratio_Tokke_G1', 'lpjoint_power_fraction_Tokke_G1', 'lpjoint_economic_margin_Tokke_G1', 'lpjoint_qmin_ratio_Tokke_G2', 'lpjoint_power_fraction_Tokke_G2', 'lpjoint_economic_margin_Tokke_G2', 'lpjoint_qmin_ratio_Tokke_G3', 'lpjoint_power_fraction_Tokke_G3', 'lpjoint_economic_margin_Tokke_G3', 'lpjoint_qmin_ratio_Tokke_G4', 'lpjoint_power_fraction_Tokke_G4', 'lpjoint_economic_margin_Tokke_G4', 'lpjoint_qmin_ratio_Lio_G1', 'lpjoint_power_fraction_Lio_G1', 'lpjoint_economic_margin_Lio_G1', 'lpjoint_qmin_ratio_Byrte_G1', 'lpjoint_power_fraction_Byrte_G1', 'lpjoint_economic_margin_Byrte_G1', 'lpjoint_qmin_ratio_Vinje_G1', 'lpjoint_power_fraction_Vinje_G1', 'lpjoint_economic_margin_Vinje_G1', 'lpjoint_qmin_ratio_Vinje_G2', 'lpjoint_power_fraction_Vinje_G2', 'lpjoint_economic_margin_Vinje_G2', 'lpjoint_qmin_ratio_Vinje_G3', 'lpjoint_power_fraction_Vinje_G3', 'lpjoint_economic_margin_Vinje_G3', 'lpjoint_qmin_ratio_Haukeli_G1', 'lpjoint_power_fraction_Haukeli_G1', 'lpjoint_economic_margin_Haukeli_G1', 'lpjoint_qmin_ratio_Songa_G1', 'lpjoint_power_fraction_Songa_G1', 'lpjoint_economic_margin_Songa_G1', 'lpjoint_qmin_ratio_Kjela_G1', 'lpjoint_power_fraction_Kjela_G1', 'lpjoint_economic_margin_Kjela_G1', 'lpjoint_qmin_ratio_Vesle Kjela_G1', 'lpjoint_power_fraction_Vesle Kjela_G1', 'lpjoint_economic_margin_Vesle Kjela_G1', 'lpjoint_plant_power_fraction_Hogga', 'lpjoint_storage_fill_Hogga', 'lpjoint_storage_min_slack_Hogga', 'lpjoint_storage_max_slack_Hogga', 'lpjoint_plant_power_fraction_Tokke', 'lpjoint_storage_fill_Tokke', 'lpjoint_storage_min_slack_Tokke', 'lpjoint_storage_max_slack_Tokke', 'lpjoint_plant_power_fraction_Lio', 'lpjoint_storage_fill_Lio', 'lpjoint_storage_min_slack_Lio', 'lpjoint_storage_max_slack_Lio', 'lpjoint_plant_power_fraction_Byrte', 'lpjoint_storage_fill_Byrte', 'lpjoint_storage_min_slack_Byrte', 'lpjoint_storage_max_slack_Byrte', 'lpjoint_plant_power_fraction_Vinje', 'lpjoint_storage_fill_Vinje', 'lpjoint_storage_min_slack_Vinje', 'lpjoint_storage_max_slack_Vinje', 'lpjoint_plant_power_fraction_Haukeli', 'lpjoint_storage_fill_Haukeli', 'lpjoint_storage_min_slack_Haukeli', 'lpjoint_storage_max_slack_Haukeli', 'lpjoint_plant_power_fraction_Songa', 'lpjoint_storage_fill_Songa', 'lpjoint_storage_min_slack_Songa', 'lpjoint_storage_max_slack_Songa', 'lpjoint_plant_power_fraction_Kjela', 'lpjoint_storage_fill_Kjela', 'lpjoint_storage_min_slack_Kjela', 'lpjoint_storage_max_slack_Kjela', 'lpjoint_plant_power_fraction_Vesle Kjela', 'lpjoint_storage_fill_Vesle Kjela', 'lpjoint_storage_min_slack_Vesle Kjela', 'lpjoint_storage_max_slack_Vesle Kjela', 'lpjoint_on_fraction_Tokke_G1', 'lpjoint_on_fraction_Tokke_G2', 'lpjoint_on_fraction_Tokke_G3', 'lpjoint_on_fraction_Tokke_G4', 'lpjoint_on_fraction_Vinje_G1', 'lpjoint_on_fraction_Vinje_G2', 'lpjoint_on_fraction_Vinje_G3', 'lpjoint_active_unit_count_Tokke', 'lpjoint_active_unit_count_Vinje'], 'gpu608': ['horizon_hour', 'horizon_day', 'hour_of_day', 'hour_sin', 'hour_cos', 'weekday', 'month', 'day_of_year', 'season_sin', 'season_cos', 'price_week_zscore', 'price_week_rank', 'volume_Kjelavatn', 'fill_Kjelavatn', 'price_over_water_Kjelavatn', 'headroom_after_inflow_Kjelavatn', 'volume_Staavatn', 'fill_Staavatn', 'price_over_water_Staavatn', 'headroom_after_inflow_Staavatn', 'volume_Langesae', 'fill_Langesae', 'price_over_water_Langesae', 'headroom_after_inflow_Langesae', 'volume_Foersvatn', 'fill_Foersvatn', 'price_over_water_Foersvatn', 'headroom_after_inflow_Foersvatn', 'volume_Venemo', 'fill_Venemo', 'price_over_water_Venemo', 'headroom_after_inflow_Venemo', 'volume_Botnedalsvatn', 'fill_Botnedalsvatn', 'price_over_water_Botnedalsvatn', 'headroom_after_inflow_Botnedalsvatn', 'volume_Bitdalsvatn', 'fill_Bitdalsvatn', 'price_over_water_Bitdalsvatn', 'headroom_after_inflow_Bitdalsvatn', 'volume_Vaamarvatn', 'fill_Vaamarvatn', 'price_over_water_Vaamarvatn', 'headroom_after_inflow_Vaamarvatn', 'volume_Langeidvatn', 'fill_Langeidvatn', 'price_over_water_Langeidvatn', 'headroom_after_inflow_Langeidvatn', 'volume_Totak', 'fill_Totak', 'price_over_water_Totak', 'headroom_after_inflow_Totak', 'volume_Songavatn', 'fill_Songavatn', 'price_over_water_Songavatn', 'headroom_after_inflow_Songavatn', 'volume_Vinjevatn', 'fill_Vinjevatn', 'price_over_water_Vinjevatn', 'headroom_after_inflow_Vinjevatn', 'volume_Bandak', 'fill_Bandak', 'price_over_water_Bandak', 'headroom_after_inflow_Bandak', 'volume_Bordalsvatn', 'fill_Bordalsvatn', 'price_over_water_Bordalsvatn', 'headroom_after_inflow_Bordalsvatn', 'volume_Byrtevatn', 'fill_Byrtevatn', 'price_over_water_Byrtevatn', 'headroom_after_inflow_Byrtevatn', 'volume_Hyljelihyl', 'fill_Hyljelihyl', 'price_over_water_Hyljelihyl', 'headroom_after_inflow_Hyljelihyl', 'volume_Vatjern', 'fill_Vatjern', 'price_over_water_Vatjern', 'headroom_after_inflow_Vatjern', 'inflow_Langesae', 'inflow_week_mean_Langesae', 'inflow_elapsed_mm3_Langesae', 'inflow_remaining_mm3_Langesae', 'inflow_Staavatn', 'inflow_week_mean_Staavatn', 'inflow_elapsed_mm3_Staavatn', 'inflow_remaining_mm3_Staavatn', 'inflow_Kjelavatn', 'inflow_week_mean_Kjelavatn', 'inflow_elapsed_mm3_Kjelavatn', 'inflow_remaining_mm3_Kjelavatn', 'inflow_Bordalsvatn', 'inflow_week_mean_Bordalsvatn', 'inflow_elapsed_mm3_Bordalsvatn', 'inflow_remaining_mm3_Bordalsvatn', 'inflow_Foersvatn', 'inflow_week_mean_Foersvatn', 'inflow_elapsed_mm3_Foersvatn', 'inflow_remaining_mm3_Foersvatn', 'inflow_Hyljelihyl', 'inflow_week_mean_Hyljelihyl', 'inflow_elapsed_mm3_Hyljelihyl', 'inflow_remaining_mm3_Hyljelihyl', 'inflow_Venemo', 'inflow_week_mean_Venemo', 'inflow_elapsed_mm3_Venemo', 'inflow_remaining_mm3_Venemo', 'inflow_Bitdalsvatn', 'inflow_week_mean_Bitdalsvatn', 'inflow_elapsed_mm3_Bitdalsvatn', 'inflow_remaining_mm3_Bitdalsvatn', 'inflow_Songavatn', 'inflow_week_mean_Songavatn', 'inflow_elapsed_mm3_Songavatn', 'inflow_remaining_mm3_Songavatn', 'inflow_Totak', 'inflow_week_mean_Totak', 'inflow_elapsed_mm3_Totak', 'inflow_remaining_mm3_Totak', 'inflow_Vaamarvatn', 'inflow_week_mean_Vaamarvatn', 'inflow_elapsed_mm3_Vaamarvatn', 'inflow_remaining_mm3_Vaamarvatn', 'inflow_Langeidvatn', 'inflow_week_mean_Langeidvatn', 'inflow_elapsed_mm3_Langeidvatn', 'inflow_remaining_mm3_Langeidvatn', 'inflow_Vatjern', 'inflow_week_mean_Vatjern', 'inflow_elapsed_mm3_Vatjern', 'inflow_remaining_mm3_Vatjern', 'inflow_Vinjevatn', 'inflow_week_mean_Vinjevatn', 'inflow_elapsed_mm3_Vinjevatn', 'inflow_remaining_mm3_Vinjevatn', 'inflow_Botnedalsvatn', 'inflow_week_mean_Botnedalsvatn', 'inflow_elapsed_mm3_Botnedalsvatn', 'inflow_remaining_mm3_Botnedalsvatn', 'inflow_Byrtevatn', 'inflow_week_mean_Byrtevatn', 'inflow_elapsed_mm3_Byrtevatn', 'inflow_remaining_mm3_Byrtevatn', 'inflow_Bandak', 'inflow_week_mean_Bandak', 'inflow_elapsed_mm3_Bandak', 'inflow_remaining_mm3_Bandak', 'inflow_r_Vest_Vassdraget', 'inflow_week_mean_r_Vest_Vassdraget', 'inflow_elapsed_mm3_r_Vest_Vassdraget', 'inflow_remaining_mm3_r_Vest_Vassdraget', 'min_volume_Totak', 'min_volume_week_max_Totak', 'min_volume_Byrtevatn', 'min_volume_week_max_Byrtevatn', 'min_volume_Staavatn', 'min_volume_week_max_Staavatn', 'min_flow_b_Kjelavatn', 'min_flow_week_max_b_Kjelavatn', 'min_flow_b_Hyljelihyl', 'min_flow_week_max_b_Hyljelihyl', 'min_flow_b_Vinjevatn', 'min_flow_week_max_b_Vinjevatn', 'min_flow_b_Byrtevatn', 'min_flow_week_max_b_Byrtevatn', 'min_flow_b_Bandak', 'min_flow_week_max_b_Bandak', 'min_flow_r_Vest_Vassdraget', 'min_flow_week_max_r_Vest_Vassdraget', 'physics_head_Hogga', 'physics_pool_fill_Hogga', 'physics_supply_fraction_Hogga', 'physics_catchment_inflow_fraction_Hogga', 'physics_dispatch_to_avoid_spill_Hogga', 'physics_profitable_remaining_fraction_Hogga', 'physics_head_Tokke', 'physics_pool_fill_Tokke', 'physics_supply_fraction_Tokke', 'physics_catchment_inflow_fraction_Tokke', 'physics_dispatch_to_avoid_spill_Tokke', 'physics_profitable_remaining_fraction_Tokke', 'physics_head_Lio', 'physics_pool_fill_Lio', 'physics_supply_fraction_Lio', 'physics_catchment_inflow_fraction_Lio', 'physics_dispatch_to_avoid_spill_Lio', 'physics_profitable_remaining_fraction_Lio', 'physics_head_Byrte', 'physics_pool_fill_Byrte', 'physics_supply_fraction_Byrte', 'physics_catchment_inflow_fraction_Byrte', 'physics_dispatch_to_avoid_spill_Byrte', 'physics_profitable_remaining_fraction_Byrte', 'physics_head_Vinje', 'physics_pool_fill_Vinje', 'physics_supply_fraction_Vinje', 'physics_catchment_inflow_fraction_Vinje', 'physics_dispatch_to_avoid_spill_Vinje', 'physics_profitable_remaining_fraction_Vinje', 'physics_head_Haukeli', 'physics_pool_fill_Haukeli', 'physics_supply_fraction_Haukeli', 'physics_catchment_inflow_fraction_Haukeli', 'physics_dispatch_to_avoid_spill_Haukeli', 'physics_profitable_remaining_fraction_Haukeli', 'physics_head_Songa', 'physics_pool_fill_Songa', 'physics_supply_fraction_Songa', 'physics_catchment_inflow_fraction_Songa', 'physics_dispatch_to_avoid_spill_Songa', 'physics_profitable_remaining_fraction_Songa', 'physics_head_Kjela', 'physics_pool_fill_Kjela', 'physics_supply_fraction_Kjela', 'physics_catchment_inflow_fraction_Kjela', 'physics_dispatch_to_avoid_spill_Kjela', 'physics_profitable_remaining_fraction_Kjela', 'physics_head_Vesle Kjela', 'physics_pool_fill_Vesle Kjela', 'physics_supply_fraction_Vesle Kjela', 'physics_catchment_inflow_fraction_Vesle Kjela', 'physics_dispatch_to_avoid_spill_Vesle Kjela', 'physics_profitable_remaining_fraction_Vesle Kjela', 'context_rank_remaining', 'lp_qmin_ratio_Hogga_G1', 'lp_power_fraction_Hogga_G1', 'lp_economic_margin_Hogga_G1', 'lp_qmin_ratio_Tokke_G1', 'lp_power_fraction_Tokke_G1', 'lp_economic_margin_Tokke_G1', 'lp_qmin_ratio_Tokke_G2', 'lp_power_fraction_Tokke_G2', 'lp_economic_margin_Tokke_G2', 'lp_qmin_ratio_Tokke_G3', 'lp_power_fraction_Tokke_G3', 'lp_economic_margin_Tokke_G3', 'lp_qmin_ratio_Tokke_G4', 'lp_power_fraction_Tokke_G4', 'lp_economic_margin_Tokke_G4', 'lp_qmin_ratio_Lio_G1', 'lp_power_fraction_Lio_G1', 'lp_economic_margin_Lio_G1', 'lp_qmin_ratio_Byrte_G1', 'lp_power_fraction_Byrte_G1', 'lp_economic_margin_Byrte_G1', 'lp_qmin_ratio_Vinje_G1', 'lp_power_fraction_Vinje_G1', 'lp_economic_margin_Vinje_G1', 'lp_qmin_ratio_Vinje_G2', 'lp_power_fraction_Vinje_G2', 'lp_economic_margin_Vinje_G2', 'lp_qmin_ratio_Vinje_G3', 'lp_power_fraction_Vinje_G3', 'lp_economic_margin_Vinje_G3', 'lp_qmin_ratio_Haukeli_G1', 'lp_power_fraction_Haukeli_G1', 'lp_economic_margin_Haukeli_G1', 'lp_qmin_ratio_Songa_G1', 'lp_power_fraction_Songa_G1', 'lp_economic_margin_Songa_G1', 'lp_qmin_ratio_Kjela_G1', 'lp_power_fraction_Kjela_G1', 'lp_economic_margin_Kjela_G1', 'lp_qmin_ratio_Vesle Kjela_G1', 'lp_power_fraction_Vesle Kjela_G1', 'lp_economic_margin_Vesle Kjela_G1', 'lp_plant_power_fraction_Hogga', 'lp_storage_fill_Hogga', 'lp_storage_min_slack_Hogga', 'lp_storage_max_slack_Hogga', 'lp_plant_power_fraction_Tokke', 'lp_storage_fill_Tokke', 'lp_storage_min_slack_Tokke', 'lp_storage_max_slack_Tokke', 'lp_plant_power_fraction_Lio', 'lp_storage_fill_Lio', 'lp_storage_min_slack_Lio', 'lp_storage_max_slack_Lio', 'lp_plant_power_fraction_Byrte', 'lp_storage_fill_Byrte', 'lp_storage_min_slack_Byrte', 'lp_storage_max_slack_Byrte', 'lp_plant_power_fraction_Vinje', 'lp_storage_fill_Vinje', 'lp_storage_min_slack_Vinje', 'lp_storage_max_slack_Vinje', 'lp_plant_power_fraction_Haukeli', 'lp_storage_fill_Haukeli', 'lp_storage_min_slack_Haukeli', 'lp_storage_max_slack_Haukeli', 'lp_plant_power_fraction_Songa', 'lp_storage_fill_Songa', 'lp_storage_min_slack_Songa', 'lp_storage_max_slack_Songa', 'lp_plant_power_fraction_Kjela', 'lp_storage_fill_Kjela', 'lp_storage_min_slack_Kjela', 'lp_storage_max_slack_Kjela', 'lp_plant_power_fraction_Vesle Kjela', 'lp_storage_fill_Vesle Kjela', 'lp_storage_min_slack_Vesle Kjela', 'lp_storage_max_slack_Vesle Kjela', 'lp_context_qmin_previous_1h_Hogga_G1', 'lp_context_qmin_next_1h_Hogga_G1', 'lp_context_qmin_previous_3h_Hogga_G1', 'lp_context_qmin_next_3h_Hogga_G1', 'lp_context_qmin_previous_6h_mean_Hogga_G1', 'lp_context_qmin_next_6h_mean_Hogga_G1', 'lp_context_qmin_week_mean_Hogga_G1', 'lp_context_qmin_week_at_least_one_fraction_Hogga_G1', 'lp_context_qmin_week_positive_fraction_Hogga_G1', 'lp_context_qmin_previous_1h_Tokke_G1', 'lp_context_qmin_next_1h_Tokke_G1', 'lp_context_qmin_previous_3h_Tokke_G1', 'lp_context_qmin_next_3h_Tokke_G1', 'lp_context_qmin_previous_6h_mean_Tokke_G1', 'lp_context_qmin_next_6h_mean_Tokke_G1', 'lp_context_qmin_week_mean_Tokke_G1', 'lp_context_qmin_week_at_least_one_fraction_Tokke_G1', 'lp_context_qmin_week_positive_fraction_Tokke_G1', 'lp_context_qmin_previous_1h_Tokke_G2', 'lp_context_qmin_next_1h_Tokke_G2', 'lp_context_qmin_previous_3h_Tokke_G2', 'lp_context_qmin_next_3h_Tokke_G2', 'lp_context_qmin_previous_6h_mean_Tokke_G2', 'lp_context_qmin_next_6h_mean_Tokke_G2', 'lp_context_qmin_week_mean_Tokke_G2', 'lp_context_qmin_week_at_least_one_fraction_Tokke_G2', 'lp_context_qmin_week_positive_fraction_Tokke_G2', 'lp_context_qmin_previous_1h_Tokke_G3', 'lp_context_qmin_next_1h_Tokke_G3', 'lp_context_qmin_previous_3h_Tokke_G3', 'lp_context_qmin_next_3h_Tokke_G3', 'lp_context_qmin_previous_6h_mean_Tokke_G3', 'lp_context_qmin_next_6h_mean_Tokke_G3', 'lp_context_qmin_week_mean_Tokke_G3', 'lp_context_qmin_week_at_least_one_fraction_Tokke_G3', 'lp_context_qmin_week_positive_fraction_Tokke_G3', 'lp_context_qmin_previous_1h_Tokke_G4', 'lp_context_qmin_next_1h_Tokke_G4', 'lp_context_qmin_previous_3h_Tokke_G4', 'lp_context_qmin_next_3h_Tokke_G4', 'lp_context_qmin_previous_6h_mean_Tokke_G4', 'lp_context_qmin_next_6h_mean_Tokke_G4', 'lp_context_qmin_week_mean_Tokke_G4', 'lp_context_qmin_week_at_least_one_fraction_Tokke_G4', 'lp_context_qmin_week_positive_fraction_Tokke_G4', 'lp_context_qmin_previous_1h_Lio_G1', 'lp_context_qmin_next_1h_Lio_G1', 'lp_context_qmin_previous_3h_Lio_G1', 'lp_context_qmin_next_3h_Lio_G1', 'lp_context_qmin_previous_6h_mean_Lio_G1', 'lp_context_qmin_next_6h_mean_Lio_G1', 'lp_context_qmin_week_mean_Lio_G1', 'lp_context_qmin_week_at_least_one_fraction_Lio_G1', 'lp_context_qmin_week_positive_fraction_Lio_G1', 'lp_context_qmin_previous_1h_Byrte_G1', 'lp_context_qmin_next_1h_Byrte_G1', 'lp_context_qmin_previous_3h_Byrte_G1', 'lp_context_qmin_next_3h_Byrte_G1', 'lp_context_qmin_previous_6h_mean_Byrte_G1', 'lp_context_qmin_next_6h_mean_Byrte_G1', 'lp_context_qmin_week_mean_Byrte_G1', 'lp_context_qmin_week_at_least_one_fraction_Byrte_G1', 'lp_context_qmin_week_positive_fraction_Byrte_G1', 'lp_context_qmin_previous_1h_Vinje_G1', 'lp_context_qmin_next_1h_Vinje_G1', 'lp_context_qmin_previous_3h_Vinje_G1', 'lp_context_qmin_next_3h_Vinje_G1', 'lp_context_qmin_previous_6h_mean_Vinje_G1', 'lp_context_qmin_next_6h_mean_Vinje_G1', 'lp_context_qmin_week_mean_Vinje_G1', 'lp_context_qmin_week_at_least_one_fraction_Vinje_G1', 'lp_context_qmin_week_positive_fraction_Vinje_G1', 'lp_context_qmin_previous_1h_Vinje_G2', 'lp_context_qmin_next_1h_Vinje_G2', 'lp_context_qmin_previous_3h_Vinje_G2', 'lp_context_qmin_next_3h_Vinje_G2', 'lp_context_qmin_previous_6h_mean_Vinje_G2', 'lp_context_qmin_next_6h_mean_Vinje_G2', 'lp_context_qmin_week_mean_Vinje_G2', 'lp_context_qmin_week_at_least_one_fraction_Vinje_G2', 'lp_context_qmin_week_positive_fraction_Vinje_G2', 'lp_context_qmin_previous_1h_Vinje_G3', 'lp_context_qmin_next_1h_Vinje_G3', 'lp_context_qmin_previous_3h_Vinje_G3', 'lp_context_qmin_next_3h_Vinje_G3', 'lp_context_qmin_previous_6h_mean_Vinje_G3', 'lp_context_qmin_next_6h_mean_Vinje_G3', 'lp_context_qmin_week_mean_Vinje_G3', 'lp_context_qmin_week_at_least_one_fraction_Vinje_G3', 'lp_context_qmin_week_positive_fraction_Vinje_G3', 'lp_context_qmin_previous_1h_Haukeli_G1', 'lp_context_qmin_next_1h_Haukeli_G1', 'lp_context_qmin_previous_3h_Haukeli_G1', 'lp_context_qmin_next_3h_Haukeli_G1', 'lp_context_qmin_previous_6h_mean_Haukeli_G1', 'lp_context_qmin_next_6h_mean_Haukeli_G1', 'lp_context_qmin_week_mean_Haukeli_G1', 'lp_context_qmin_week_at_least_one_fraction_Haukeli_G1', 'lp_context_qmin_week_positive_fraction_Haukeli_G1', 'lp_context_qmin_previous_1h_Songa_G1', 'lp_context_qmin_next_1h_Songa_G1', 'lp_context_qmin_previous_3h_Songa_G1', 'lp_context_qmin_next_3h_Songa_G1', 'lp_context_qmin_previous_6h_mean_Songa_G1', 'lp_context_qmin_next_6h_mean_Songa_G1', 'lp_context_qmin_week_mean_Songa_G1', 'lp_context_qmin_week_at_least_one_fraction_Songa_G1', 'lp_context_qmin_week_positive_fraction_Songa_G1', 'lp_context_qmin_previous_1h_Kjela_G1', 'lp_context_qmin_next_1h_Kjela_G1', 'lp_context_qmin_previous_3h_Kjela_G1', 'lp_context_qmin_next_3h_Kjela_G1', 'lp_context_qmin_previous_6h_mean_Kjela_G1', 'lp_context_qmin_next_6h_mean_Kjela_G1', 'lp_context_qmin_week_mean_Kjela_G1', 'lp_context_qmin_week_at_least_one_fraction_Kjela_G1', 'lp_context_qmin_week_positive_fraction_Kjela_G1', 'lp_context_qmin_previous_1h_Vesle Kjela_G1', 'lp_context_qmin_next_1h_Vesle Kjela_G1', 'lp_context_qmin_previous_3h_Vesle Kjela_G1', 'lp_context_qmin_next_3h_Vesle Kjela_G1', 'lp_context_qmin_previous_6h_mean_Vesle Kjela_G1', 'lp_context_qmin_next_6h_mean_Vesle Kjela_G1', 'lp_context_qmin_week_mean_Vesle Kjela_G1', 'lp_context_qmin_week_at_least_one_fraction_Vesle Kjela_G1', 'lp_context_qmin_week_positive_fraction_Vesle Kjela_G1', 'norm_price', 'norm_price_offset_-24', 'norm_price_offset_-12', 'norm_price_offset_-6', 'norm_price_offset_-3', 'norm_price_offset_-1', 'norm_price_offset_+1', 'norm_price_offset_+3', 'norm_price_offset_+6', 'norm_price_offset_+12', 'norm_price_offset_+24', 'norm_price_week_mean', 'norm_price_week_std', 'norm_price_week_min', 'norm_price_week_max', 'norm_price_week_q10', 'norm_price_week_q25', 'norm_price_week_q50', 'norm_price_week_q75', 'norm_price_week_q90', 'norm_price_minus_week_mean', 'norm_price_day_mean', 'norm_price_day_min', 'norm_price_day_max', 'norm_price_day_std', 'norm_price_remaining_mean', 'norm_price_remaining_max', 'norm_water_value_Kjelavatn', 'norm_price_minus_water_Kjelavatn', 'norm_water_value_Staavatn', 'norm_price_minus_water_Staavatn', 'norm_water_value_Langesae', 'norm_price_minus_water_Langesae', 'norm_water_value_Foersvatn', 'norm_price_minus_water_Foersvatn', 'norm_water_value_Venemo', 'norm_price_minus_water_Venemo', 'norm_water_value_Botnedalsvatn', 'norm_price_minus_water_Botnedalsvatn', 'norm_water_value_Bitdalsvatn', 'norm_price_minus_water_Bitdalsvatn', 'norm_water_value_Vaamarvatn', 'norm_price_minus_water_Vaamarvatn', 'norm_water_value_Langeidvatn', 'norm_price_minus_water_Langeidvatn', 'norm_water_value_Totak', 'norm_price_minus_water_Totak', 'norm_water_value_Songavatn', 'norm_price_minus_water_Songavatn', 'norm_water_value_Vinjevatn', 'norm_price_minus_water_Vinjevatn', 'norm_water_value_Bandak', 'norm_price_minus_water_Bandak', 'norm_water_value_Bordalsvatn', 'norm_price_minus_water_Bordalsvatn', 'norm_water_value_Byrtevatn', 'norm_price_minus_water_Byrtevatn', 'norm_water_value_Hyljelihyl', 'norm_price_minus_water_Hyljelihyl', 'norm_water_value_Vatjern', 'norm_price_minus_water_Vatjern', 'norm_physics_switch_margin_minimum_Hogga', 'norm_physics_switch_margin_maximum_Hogga', 'norm_physics_switch_margin_minimum_Tokke', 'norm_physics_switch_margin_maximum_Tokke', 'norm_physics_switch_margin_minimum_Lio', 'norm_physics_switch_margin_maximum_Lio', 'norm_physics_switch_margin_minimum_Byrte', 'norm_physics_switch_margin_maximum_Byrte', 'norm_physics_switch_margin_minimum_Vinje', 'norm_physics_switch_margin_maximum_Vinje', 'norm_physics_switch_margin_minimum_Haukeli', 'norm_physics_switch_margin_maximum_Haukeli', 'norm_physics_switch_margin_minimum_Songa', 'norm_physics_switch_margin_maximum_Songa', 'norm_physics_switch_margin_minimum_Kjela', 'norm_physics_switch_margin_maximum_Kjela', 'norm_physics_switch_margin_minimum_Vesle Kjela', 'norm_physics_switch_margin_maximum_Vesle Kjela', 'norm_price_previous_mean_6h', 'norm_price_minus_previous_mean_6h', 'norm_price_next_mean_6h', 'norm_price_minus_next_mean_6h', 'norm_price_previous_mean_12h', 'norm_price_minus_previous_mean_12h', 'norm_price_next_mean_12h', 'norm_price_minus_next_mean_12h', 'norm_price_previous_mean_24h', 'norm_price_minus_previous_mean_24h', 'norm_price_next_mean_24h', 'norm_price_minus_next_mean_24h', 'monetary_scale_inverse', 'lpjoint_qmin_ratio_Hogga_G1', 'lpjoint_power_fraction_Hogga_G1', 'lpjoint_economic_margin_Hogga_G1', 'lpjoint_qmin_ratio_Tokke_G1', 'lpjoint_power_fraction_Tokke_G1', 'lpjoint_economic_margin_Tokke_G1', 'lpjoint_qmin_ratio_Tokke_G2', 'lpjoint_power_fraction_Tokke_G2', 'lpjoint_economic_margin_Tokke_G2', 'lpjoint_qmin_ratio_Tokke_G3', 'lpjoint_power_fraction_Tokke_G3', 'lpjoint_economic_margin_Tokke_G3', 'lpjoint_qmin_ratio_Tokke_G4', 'lpjoint_power_fraction_Tokke_G4', 'lpjoint_economic_margin_Tokke_G4', 'lpjoint_qmin_ratio_Lio_G1', 'lpjoint_power_fraction_Lio_G1', 'lpjoint_economic_margin_Lio_G1', 'lpjoint_qmin_ratio_Byrte_G1', 'lpjoint_power_fraction_Byrte_G1', 'lpjoint_economic_margin_Byrte_G1', 'lpjoint_qmin_ratio_Vinje_G1', 'lpjoint_power_fraction_Vinje_G1', 'lpjoint_economic_margin_Vinje_G1', 'lpjoint_qmin_ratio_Vinje_G2', 'lpjoint_power_fraction_Vinje_G2', 'lpjoint_economic_margin_Vinje_G2', 'lpjoint_qmin_ratio_Vinje_G3', 'lpjoint_power_fraction_Vinje_G3', 'lpjoint_economic_margin_Vinje_G3', 'lpjoint_qmin_ratio_Haukeli_G1', 'lpjoint_power_fraction_Haukeli_G1', 'lpjoint_economic_margin_Haukeli_G1', 'lpjoint_qmin_ratio_Songa_G1', 'lpjoint_power_fraction_Songa_G1', 'lpjoint_economic_margin_Songa_G1', 'lpjoint_qmin_ratio_Kjela_G1', 'lpjoint_power_fraction_Kjela_G1', 'lpjoint_economic_margin_Kjela_G1', 'lpjoint_qmin_ratio_Vesle Kjela_G1', 'lpjoint_power_fraction_Vesle Kjela_G1', 'lpjoint_economic_margin_Vesle Kjela_G1', 'lpjoint_plant_power_fraction_Hogga', 'lpjoint_storage_fill_Hogga', 'lpjoint_storage_min_slack_Hogga', 'lpjoint_storage_max_slack_Hogga', 'lpjoint_plant_power_fraction_Tokke', 'lpjoint_storage_fill_Tokke', 'lpjoint_storage_min_slack_Tokke', 'lpjoint_storage_max_slack_Tokke', 'lpjoint_plant_power_fraction_Lio', 'lpjoint_storage_fill_Lio', 'lpjoint_storage_min_slack_Lio', 'lpjoint_storage_max_slack_Lio', 'lpjoint_plant_power_fraction_Byrte', 'lpjoint_storage_fill_Byrte', 'lpjoint_storage_min_slack_Byrte', 'lpjoint_storage_max_slack_Byrte', 'lpjoint_plant_power_fraction_Vinje', 'lpjoint_storage_fill_Vinje', 'lpjoint_storage_min_slack_Vinje', 'lpjoint_storage_max_slack_Vinje', 'lpjoint_plant_power_fraction_Haukeli', 'lpjoint_storage_fill_Haukeli', 'lpjoint_storage_min_slack_Haukeli', 'lpjoint_storage_max_slack_Haukeli', 'lpjoint_plant_power_fraction_Songa', 'lpjoint_storage_fill_Songa', 'lpjoint_storage_min_slack_Songa', 'lpjoint_storage_max_slack_Songa', 'lpjoint_plant_power_fraction_Kjela', 'lpjoint_storage_fill_Kjela', 'lpjoint_storage_min_slack_Kjela', 'lpjoint_storage_max_slack_Kjela', 'lpjoint_plant_power_fraction_Vesle Kjela', 'lpjoint_storage_fill_Vesle Kjela', 'lpjoint_storage_min_slack_Vesle Kjela', 'lpjoint_storage_max_slack_Vesle Kjela', 'lpjoint_on_fraction_Tokke_G1', 'lpjoint_on_fraction_Tokke_G2', 'lpjoint_on_fraction_Tokke_G3', 'lpjoint_on_fraction_Tokke_G4', 'lpjoint_on_fraction_Vinje_G1', 'lpjoint_on_fraction_Vinje_G2', 'lpjoint_on_fraction_Vinje_G3', 'lpjoint_active_unit_count_Tokke', 'lpjoint_active_unit_count_Vinje'], 'gpu664': ['horizon_hour', 'horizon_day', 'hour_of_day', 'hour_sin', 'hour_cos', 'weekday', 'month', 'day_of_year', 'season_sin', 'season_cos', 'price_week_zscore', 'price_week_rank', 'volume_Kjelavatn', 'fill_Kjelavatn', 'price_over_water_Kjelavatn', 'headroom_after_inflow_Kjelavatn', 'volume_Staavatn', 'fill_Staavatn', 'price_over_water_Staavatn', 'headroom_after_inflow_Staavatn', 'volume_Langesae', 'fill_Langesae', 'price_over_water_Langesae', 'headroom_after_inflow_Langesae', 'volume_Foersvatn', 'fill_Foersvatn', 'price_over_water_Foersvatn', 'headroom_after_inflow_Foersvatn', 'volume_Venemo', 'fill_Venemo', 'price_over_water_Venemo', 'headroom_after_inflow_Venemo', 'volume_Botnedalsvatn', 'fill_Botnedalsvatn', 'price_over_water_Botnedalsvatn', 'headroom_after_inflow_Botnedalsvatn', 'volume_Bitdalsvatn', 'fill_Bitdalsvatn', 'price_over_water_Bitdalsvatn', 'headroom_after_inflow_Bitdalsvatn', 'volume_Vaamarvatn', 'fill_Vaamarvatn', 'price_over_water_Vaamarvatn', 'headroom_after_inflow_Vaamarvatn', 'volume_Langeidvatn', 'fill_Langeidvatn', 'price_over_water_Langeidvatn', 'headroom_after_inflow_Langeidvatn', 'volume_Totak', 'fill_Totak', 'price_over_water_Totak', 'headroom_after_inflow_Totak', 'volume_Songavatn', 'fill_Songavatn', 'price_over_water_Songavatn', 'headroom_after_inflow_Songavatn', 'volume_Vinjevatn', 'fill_Vinjevatn', 'price_over_water_Vinjevatn', 'headroom_after_inflow_Vinjevatn', 'volume_Bandak', 'fill_Bandak', 'price_over_water_Bandak', 'headroom_after_inflow_Bandak', 'volume_Bordalsvatn', 'fill_Bordalsvatn', 'price_over_water_Bordalsvatn', 'headroom_after_inflow_Bordalsvatn', 'volume_Byrtevatn', 'fill_Byrtevatn', 'price_over_water_Byrtevatn', 'headroom_after_inflow_Byrtevatn', 'volume_Hyljelihyl', 'fill_Hyljelihyl', 'price_over_water_Hyljelihyl', 'headroom_after_inflow_Hyljelihyl', 'volume_Vatjern', 'fill_Vatjern', 'price_over_water_Vatjern', 'headroom_after_inflow_Vatjern', 'inflow_Langesae', 'inflow_week_mean_Langesae', 'inflow_elapsed_mm3_Langesae', 'inflow_remaining_mm3_Langesae', 'inflow_Staavatn', 'inflow_week_mean_Staavatn', 'inflow_elapsed_mm3_Staavatn', 'inflow_remaining_mm3_Staavatn', 'inflow_Kjelavatn', 'inflow_week_mean_Kjelavatn', 'inflow_elapsed_mm3_Kjelavatn', 'inflow_remaining_mm3_Kjelavatn', 'inflow_Bordalsvatn', 'inflow_week_mean_Bordalsvatn', 'inflow_elapsed_mm3_Bordalsvatn', 'inflow_remaining_mm3_Bordalsvatn', 'inflow_Foersvatn', 'inflow_week_mean_Foersvatn', 'inflow_elapsed_mm3_Foersvatn', 'inflow_remaining_mm3_Foersvatn', 'inflow_Hyljelihyl', 'inflow_week_mean_Hyljelihyl', 'inflow_elapsed_mm3_Hyljelihyl', 'inflow_remaining_mm3_Hyljelihyl', 'inflow_Venemo', 'inflow_week_mean_Venemo', 'inflow_elapsed_mm3_Venemo', 'inflow_remaining_mm3_Venemo', 'inflow_Bitdalsvatn', 'inflow_week_mean_Bitdalsvatn', 'inflow_elapsed_mm3_Bitdalsvatn', 'inflow_remaining_mm3_Bitdalsvatn', 'inflow_Songavatn', 'inflow_week_mean_Songavatn', 'inflow_elapsed_mm3_Songavatn', 'inflow_remaining_mm3_Songavatn', 'inflow_Totak', 'inflow_week_mean_Totak', 'inflow_elapsed_mm3_Totak', 'inflow_remaining_mm3_Totak', 'inflow_Vaamarvatn', 'inflow_week_mean_Vaamarvatn', 'inflow_elapsed_mm3_Vaamarvatn', 'inflow_remaining_mm3_Vaamarvatn', 'inflow_Langeidvatn', 'inflow_week_mean_Langeidvatn', 'inflow_elapsed_mm3_Langeidvatn', 'inflow_remaining_mm3_Langeidvatn', 'inflow_Vatjern', 'inflow_week_mean_Vatjern', 'inflow_elapsed_mm3_Vatjern', 'inflow_remaining_mm3_Vatjern', 'inflow_Vinjevatn', 'inflow_week_mean_Vinjevatn', 'inflow_elapsed_mm3_Vinjevatn', 'inflow_remaining_mm3_Vinjevatn', 'inflow_Botnedalsvatn', 'inflow_week_mean_Botnedalsvatn', 'inflow_elapsed_mm3_Botnedalsvatn', 'inflow_remaining_mm3_Botnedalsvatn', 'inflow_Byrtevatn', 'inflow_week_mean_Byrtevatn', 'inflow_elapsed_mm3_Byrtevatn', 'inflow_remaining_mm3_Byrtevatn', 'inflow_Bandak', 'inflow_week_mean_Bandak', 'inflow_elapsed_mm3_Bandak', 'inflow_remaining_mm3_Bandak', 'inflow_r_Vest_Vassdraget', 'inflow_week_mean_r_Vest_Vassdraget', 'inflow_elapsed_mm3_r_Vest_Vassdraget', 'inflow_remaining_mm3_r_Vest_Vassdraget', 'min_volume_Totak', 'min_volume_week_max_Totak', 'min_volume_Byrtevatn', 'min_volume_week_max_Byrtevatn', 'min_volume_Staavatn', 'min_volume_week_max_Staavatn', 'min_flow_b_Kjelavatn', 'min_flow_week_max_b_Kjelavatn', 'min_flow_b_Hyljelihyl', 'min_flow_week_max_b_Hyljelihyl', 'min_flow_b_Vinjevatn', 'min_flow_week_max_b_Vinjevatn', 'min_flow_b_Byrtevatn', 'min_flow_week_max_b_Byrtevatn', 'min_flow_b_Bandak', 'min_flow_week_max_b_Bandak', 'min_flow_r_Vest_Vassdraget', 'min_flow_week_max_r_Vest_Vassdraget', 'physics_head_Hogga', 'physics_pool_fill_Hogga', 'physics_supply_fraction_Hogga', 'physics_catchment_inflow_fraction_Hogga', 'physics_dispatch_to_avoid_spill_Hogga', 'physics_profitable_remaining_fraction_Hogga', 'physics_head_Tokke', 'physics_pool_fill_Tokke', 'physics_supply_fraction_Tokke', 'physics_catchment_inflow_fraction_Tokke', 'physics_dispatch_to_avoid_spill_Tokke', 'physics_profitable_remaining_fraction_Tokke', 'physics_head_Lio', 'physics_pool_fill_Lio', 'physics_supply_fraction_Lio', 'physics_catchment_inflow_fraction_Lio', 'physics_dispatch_to_avoid_spill_Lio', 'physics_profitable_remaining_fraction_Lio', 'physics_head_Byrte', 'physics_pool_fill_Byrte', 'physics_supply_fraction_Byrte', 'physics_catchment_inflow_fraction_Byrte', 'physics_dispatch_to_avoid_spill_Byrte', 'physics_profitable_remaining_fraction_Byrte', 'physics_head_Vinje', 'physics_pool_fill_Vinje', 'physics_supply_fraction_Vinje', 'physics_catchment_inflow_fraction_Vinje', 'physics_dispatch_to_avoid_spill_Vinje', 'physics_profitable_remaining_fraction_Vinje', 'physics_head_Haukeli', 'physics_pool_fill_Haukeli', 'physics_supply_fraction_Haukeli', 'physics_catchment_inflow_fraction_Haukeli', 'physics_dispatch_to_avoid_spill_Haukeli', 'physics_profitable_remaining_fraction_Haukeli', 'physics_head_Songa', 'physics_pool_fill_Songa', 'physics_supply_fraction_Songa', 'physics_catchment_inflow_fraction_Songa', 'physics_dispatch_to_avoid_spill_Songa', 'physics_profitable_remaining_fraction_Songa', 'physics_head_Kjela', 'physics_pool_fill_Kjela', 'physics_supply_fraction_Kjela', 'physics_catchment_inflow_fraction_Kjela', 'physics_dispatch_to_avoid_spill_Kjela', 'physics_profitable_remaining_fraction_Kjela', 'physics_head_Vesle Kjela', 'physics_pool_fill_Vesle Kjela', 'physics_supply_fraction_Vesle Kjela', 'physics_catchment_inflow_fraction_Vesle Kjela', 'physics_dispatch_to_avoid_spill_Vesle Kjela', 'physics_profitable_remaining_fraction_Vesle Kjela', 'context_rank_remaining', 'lp_qmin_ratio_Hogga_G1', 'lp_power_fraction_Hogga_G1', 'lp_economic_margin_Hogga_G1', 'lp_qmin_ratio_Tokke_G1', 'lp_power_fraction_Tokke_G1', 'lp_economic_margin_Tokke_G1', 'lp_qmin_ratio_Tokke_G2', 'lp_power_fraction_Tokke_G2', 'lp_economic_margin_Tokke_G2', 'lp_qmin_ratio_Tokke_G3', 'lp_power_fraction_Tokke_G3', 'lp_economic_margin_Tokke_G3', 'lp_qmin_ratio_Tokke_G4', 'lp_power_fraction_Tokke_G4', 'lp_economic_margin_Tokke_G4', 'lp_qmin_ratio_Lio_G1', 'lp_power_fraction_Lio_G1', 'lp_economic_margin_Lio_G1', 'lp_qmin_ratio_Byrte_G1', 'lp_power_fraction_Byrte_G1', 'lp_economic_margin_Byrte_G1', 'lp_qmin_ratio_Vinje_G1', 'lp_power_fraction_Vinje_G1', 'lp_economic_margin_Vinje_G1', 'lp_qmin_ratio_Vinje_G2', 'lp_power_fraction_Vinje_G2', 'lp_economic_margin_Vinje_G2', 'lp_qmin_ratio_Vinje_G3', 'lp_power_fraction_Vinje_G3', 'lp_economic_margin_Vinje_G3', 'lp_qmin_ratio_Haukeli_G1', 'lp_power_fraction_Haukeli_G1', 'lp_economic_margin_Haukeli_G1', 'lp_qmin_ratio_Songa_G1', 'lp_power_fraction_Songa_G1', 'lp_economic_margin_Songa_G1', 'lp_qmin_ratio_Kjela_G1', 'lp_power_fraction_Kjela_G1', 'lp_economic_margin_Kjela_G1', 'lp_qmin_ratio_Vesle Kjela_G1', 'lp_power_fraction_Vesle Kjela_G1', 'lp_economic_margin_Vesle Kjela_G1', 'lp_plant_power_fraction_Hogga', 'lp_storage_fill_Hogga', 'lp_storage_min_slack_Hogga', 'lp_storage_max_slack_Hogga', 'lp_plant_power_fraction_Tokke', 'lp_storage_fill_Tokke', 'lp_storage_min_slack_Tokke', 'lp_storage_max_slack_Tokke', 'lp_plant_power_fraction_Lio', 'lp_storage_fill_Lio', 'lp_storage_min_slack_Lio', 'lp_storage_max_slack_Lio', 'lp_plant_power_fraction_Byrte', 'lp_storage_fill_Byrte', 'lp_storage_min_slack_Byrte', 'lp_storage_max_slack_Byrte', 'lp_plant_power_fraction_Vinje', 'lp_storage_fill_Vinje', 'lp_storage_min_slack_Vinje', 'lp_storage_max_slack_Vinje', 'lp_plant_power_fraction_Haukeli', 'lp_storage_fill_Haukeli', 'lp_storage_min_slack_Haukeli', 'lp_storage_max_slack_Haukeli', 'lp_plant_power_fraction_Songa', 'lp_storage_fill_Songa', 'lp_storage_min_slack_Songa', 'lp_storage_max_slack_Songa', 'lp_plant_power_fraction_Kjela', 'lp_storage_fill_Kjela', 'lp_storage_min_slack_Kjela', 'lp_storage_max_slack_Kjela', 'lp_plant_power_fraction_Vesle Kjela', 'lp_storage_fill_Vesle Kjela', 'lp_storage_min_slack_Vesle Kjela', 'lp_storage_max_slack_Vesle Kjela', 'lp_context_qmin_previous_1h_Hogga_G1', 'lp_context_qmin_next_1h_Hogga_G1', 'lp_context_qmin_previous_3h_Hogga_G1', 'lp_context_qmin_next_3h_Hogga_G1', 'lp_context_qmin_previous_6h_mean_Hogga_G1', 'lp_context_qmin_next_6h_mean_Hogga_G1', 'lp_context_qmin_week_mean_Hogga_G1', 'lp_context_qmin_week_at_least_one_fraction_Hogga_G1', 'lp_context_qmin_week_positive_fraction_Hogga_G1', 'lp_context_qmin_previous_1h_Tokke_G1', 'lp_context_qmin_next_1h_Tokke_G1', 'lp_context_qmin_previous_3h_Tokke_G1', 'lp_context_qmin_next_3h_Tokke_G1', 'lp_context_qmin_previous_6h_mean_Tokke_G1', 'lp_context_qmin_next_6h_mean_Tokke_G1', 'lp_context_qmin_week_mean_Tokke_G1', 'lp_context_qmin_week_at_least_one_fraction_Tokke_G1', 'lp_context_qmin_week_positive_fraction_Tokke_G1', 'lp_context_qmin_previous_1h_Tokke_G2', 'lp_context_qmin_next_1h_Tokke_G2', 'lp_context_qmin_previous_3h_Tokke_G2', 'lp_context_qmin_next_3h_Tokke_G2', 'lp_context_qmin_previous_6h_mean_Tokke_G2', 'lp_context_qmin_next_6h_mean_Tokke_G2', 'lp_context_qmin_week_mean_Tokke_G2', 'lp_context_qmin_week_at_least_one_fraction_Tokke_G2', 'lp_context_qmin_week_positive_fraction_Tokke_G2', 'lp_context_qmin_previous_1h_Tokke_G3', 'lp_context_qmin_next_1h_Tokke_G3', 'lp_context_qmin_previous_3h_Tokke_G3', 'lp_context_qmin_next_3h_Tokke_G3', 'lp_context_qmin_previous_6h_mean_Tokke_G3', 'lp_context_qmin_next_6h_mean_Tokke_G3', 'lp_context_qmin_week_mean_Tokke_G3', 'lp_context_qmin_week_at_least_one_fraction_Tokke_G3', 'lp_context_qmin_week_positive_fraction_Tokke_G3', 'lp_context_qmin_previous_1h_Tokke_G4', 'lp_context_qmin_next_1h_Tokke_G4', 'lp_context_qmin_previous_3h_Tokke_G4', 'lp_context_qmin_next_3h_Tokke_G4', 'lp_context_qmin_previous_6h_mean_Tokke_G4', 'lp_context_qmin_next_6h_mean_Tokke_G4', 'lp_context_qmin_week_mean_Tokke_G4', 'lp_context_qmin_week_at_least_one_fraction_Tokke_G4', 'lp_context_qmin_week_positive_fraction_Tokke_G4', 'lp_context_qmin_previous_1h_Lio_G1', 'lp_context_qmin_next_1h_Lio_G1', 'lp_context_qmin_previous_3h_Lio_G1', 'lp_context_qmin_next_3h_Lio_G1', 'lp_context_qmin_previous_6h_mean_Lio_G1', 'lp_context_qmin_next_6h_mean_Lio_G1', 'lp_context_qmin_week_mean_Lio_G1', 'lp_context_qmin_week_at_least_one_fraction_Lio_G1', 'lp_context_qmin_week_positive_fraction_Lio_G1', 'lp_context_qmin_previous_1h_Byrte_G1', 'lp_context_qmin_next_1h_Byrte_G1', 'lp_context_qmin_previous_3h_Byrte_G1', 'lp_context_qmin_next_3h_Byrte_G1', 'lp_context_qmin_previous_6h_mean_Byrte_G1', 'lp_context_qmin_next_6h_mean_Byrte_G1', 'lp_context_qmin_week_mean_Byrte_G1', 'lp_context_qmin_week_at_least_one_fraction_Byrte_G1', 'lp_context_qmin_week_positive_fraction_Byrte_G1', 'lp_context_qmin_previous_1h_Vinje_G1', 'lp_context_qmin_next_1h_Vinje_G1', 'lp_context_qmin_previous_3h_Vinje_G1', 'lp_context_qmin_next_3h_Vinje_G1', 'lp_context_qmin_previous_6h_mean_Vinje_G1', 'lp_context_qmin_next_6h_mean_Vinje_G1', 'lp_context_qmin_week_mean_Vinje_G1', 'lp_context_qmin_week_at_least_one_fraction_Vinje_G1', 'lp_context_qmin_week_positive_fraction_Vinje_G1', 'lp_context_qmin_previous_1h_Vinje_G2', 'lp_context_qmin_next_1h_Vinje_G2', 'lp_context_qmin_previous_3h_Vinje_G2', 'lp_context_qmin_next_3h_Vinje_G2', 'lp_context_qmin_previous_6h_mean_Vinje_G2', 'lp_context_qmin_next_6h_mean_Vinje_G2', 'lp_context_qmin_week_mean_Vinje_G2', 'lp_context_qmin_week_at_least_one_fraction_Vinje_G2', 'lp_context_qmin_week_positive_fraction_Vinje_G2', 'lp_context_qmin_previous_1h_Vinje_G3', 'lp_context_qmin_next_1h_Vinje_G3', 'lp_context_qmin_previous_3h_Vinje_G3', 'lp_context_qmin_next_3h_Vinje_G3', 'lp_context_qmin_previous_6h_mean_Vinje_G3', 'lp_context_qmin_next_6h_mean_Vinje_G3', 'lp_context_qmin_week_mean_Vinje_G3', 'lp_context_qmin_week_at_least_one_fraction_Vinje_G3', 'lp_context_qmin_week_positive_fraction_Vinje_G3', 'lp_context_qmin_previous_1h_Haukeli_G1', 'lp_context_qmin_next_1h_Haukeli_G1', 'lp_context_qmin_previous_3h_Haukeli_G1', 'lp_context_qmin_next_3h_Haukeli_G1', 'lp_context_qmin_previous_6h_mean_Haukeli_G1', 'lp_context_qmin_next_6h_mean_Haukeli_G1', 'lp_context_qmin_week_mean_Haukeli_G1', 'lp_context_qmin_week_at_least_one_fraction_Haukeli_G1', 'lp_context_qmin_week_positive_fraction_Haukeli_G1', 'lp_context_qmin_previous_1h_Songa_G1', 'lp_context_qmin_next_1h_Songa_G1', 'lp_context_qmin_previous_3h_Songa_G1', 'lp_context_qmin_next_3h_Songa_G1', 'lp_context_qmin_previous_6h_mean_Songa_G1', 'lp_context_qmin_next_6h_mean_Songa_G1', 'lp_context_qmin_week_mean_Songa_G1', 'lp_context_qmin_week_at_least_one_fraction_Songa_G1', 'lp_context_qmin_week_positive_fraction_Songa_G1', 'lp_context_qmin_previous_1h_Kjela_G1', 'lp_context_qmin_next_1h_Kjela_G1', 'lp_context_qmin_previous_3h_Kjela_G1', 'lp_context_qmin_next_3h_Kjela_G1', 'lp_context_qmin_previous_6h_mean_Kjela_G1', 'lp_context_qmin_next_6h_mean_Kjela_G1', 'lp_context_qmin_week_mean_Kjela_G1', 'lp_context_qmin_week_at_least_one_fraction_Kjela_G1', 'lp_context_qmin_week_positive_fraction_Kjela_G1', 'lp_context_qmin_previous_1h_Vesle Kjela_G1', 'lp_context_qmin_next_1h_Vesle Kjela_G1', 'lp_context_qmin_previous_3h_Vesle Kjela_G1', 'lp_context_qmin_next_3h_Vesle Kjela_G1', 'lp_context_qmin_previous_6h_mean_Vesle Kjela_G1', 'lp_context_qmin_next_6h_mean_Vesle Kjela_G1', 'lp_context_qmin_week_mean_Vesle Kjela_G1', 'lp_context_qmin_week_at_least_one_fraction_Vesle Kjela_G1', 'lp_context_qmin_week_positive_fraction_Vesle Kjela_G1', 'norm_price', 'norm_price_offset_-24', 'norm_price_offset_-12', 'norm_price_offset_-6', 'norm_price_offset_-3', 'norm_price_offset_-1', 'norm_price_offset_+1', 'norm_price_offset_+3', 'norm_price_offset_+6', 'norm_price_offset_+12', 'norm_price_offset_+24', 'norm_price_week_mean', 'norm_price_week_std', 'norm_price_week_min', 'norm_price_week_max', 'norm_price_week_q10', 'norm_price_week_q25', 'norm_price_week_q50', 'norm_price_week_q75', 'norm_price_week_q90', 'norm_price_minus_week_mean', 'norm_price_day_mean', 'norm_price_day_min', 'norm_price_day_max', 'norm_price_day_std', 'norm_price_remaining_mean', 'norm_price_remaining_max', 'norm_water_value_Kjelavatn', 'norm_price_minus_water_Kjelavatn', 'norm_water_value_Staavatn', 'norm_price_minus_water_Staavatn', 'norm_water_value_Langesae', 'norm_price_minus_water_Langesae', 'norm_water_value_Foersvatn', 'norm_price_minus_water_Foersvatn', 'norm_water_value_Venemo', 'norm_price_minus_water_Venemo', 'norm_water_value_Botnedalsvatn', 'norm_price_minus_water_Botnedalsvatn', 'norm_water_value_Bitdalsvatn', 'norm_price_minus_water_Bitdalsvatn', 'norm_water_value_Vaamarvatn', 'norm_price_minus_water_Vaamarvatn', 'norm_water_value_Langeidvatn', 'norm_price_minus_water_Langeidvatn', 'norm_water_value_Totak', 'norm_price_minus_water_Totak', 'norm_water_value_Songavatn', 'norm_price_minus_water_Songavatn', 'norm_water_value_Vinjevatn', 'norm_price_minus_water_Vinjevatn', 'norm_water_value_Bandak', 'norm_price_minus_water_Bandak', 'norm_water_value_Bordalsvatn', 'norm_price_minus_water_Bordalsvatn', 'norm_water_value_Byrtevatn', 'norm_price_minus_water_Byrtevatn', 'norm_water_value_Hyljelihyl', 'norm_price_minus_water_Hyljelihyl', 'norm_water_value_Vatjern', 'norm_price_minus_water_Vatjern', 'norm_physics_switch_margin_minimum_Hogga', 'norm_physics_switch_margin_maximum_Hogga', 'norm_physics_switch_margin_minimum_Tokke', 'norm_physics_switch_margin_maximum_Tokke', 'norm_physics_switch_margin_minimum_Lio', 'norm_physics_switch_margin_maximum_Lio', 'norm_physics_switch_margin_minimum_Byrte', 'norm_physics_switch_margin_maximum_Byrte', 'norm_physics_switch_margin_minimum_Vinje', 'norm_physics_switch_margin_maximum_Vinje', 'norm_physics_switch_margin_minimum_Haukeli', 'norm_physics_switch_margin_maximum_Haukeli', 'norm_physics_switch_margin_minimum_Songa', 'norm_physics_switch_margin_maximum_Songa', 'norm_physics_switch_margin_minimum_Kjela', 'norm_physics_switch_margin_maximum_Kjela', 'norm_physics_switch_margin_minimum_Vesle Kjela', 'norm_physics_switch_margin_maximum_Vesle Kjela', 'norm_price_previous_mean_6h', 'norm_price_minus_previous_mean_6h', 'norm_price_next_mean_6h', 'norm_price_minus_next_mean_6h', 'norm_price_previous_mean_12h', 'norm_price_minus_previous_mean_12h', 'norm_price_next_mean_12h', 'norm_price_minus_next_mean_12h', 'norm_price_previous_mean_24h', 'norm_price_minus_previous_mean_24h', 'norm_price_next_mean_24h', 'norm_price_minus_next_mean_24h', 'monetary_scale_inverse', 'lpjoint_qmin_ratio_Hogga_G1', 'lpjoint_power_fraction_Hogga_G1', 'lpjoint_economic_margin_Hogga_G1', 'lpjoint_qmin_ratio_Tokke_G1', 'lpjoint_power_fraction_Tokke_G1', 'lpjoint_economic_margin_Tokke_G1', 'lpjoint_qmin_ratio_Tokke_G2', 'lpjoint_power_fraction_Tokke_G2', 'lpjoint_economic_margin_Tokke_G2', 'lpjoint_qmin_ratio_Tokke_G3', 'lpjoint_power_fraction_Tokke_G3', 'lpjoint_economic_margin_Tokke_G3', 'lpjoint_qmin_ratio_Tokke_G4', 'lpjoint_power_fraction_Tokke_G4', 'lpjoint_economic_margin_Tokke_G4', 'lpjoint_qmin_ratio_Lio_G1', 'lpjoint_power_fraction_Lio_G1', 'lpjoint_economic_margin_Lio_G1', 'lpjoint_qmin_ratio_Byrte_G1', 'lpjoint_power_fraction_Byrte_G1', 'lpjoint_economic_margin_Byrte_G1', 'lpjoint_qmin_ratio_Vinje_G1', 'lpjoint_power_fraction_Vinje_G1', 'lpjoint_economic_margin_Vinje_G1', 'lpjoint_qmin_ratio_Vinje_G2', 'lpjoint_power_fraction_Vinje_G2', 'lpjoint_economic_margin_Vinje_G2', 'lpjoint_qmin_ratio_Vinje_G3', 'lpjoint_power_fraction_Vinje_G3', 'lpjoint_economic_margin_Vinje_G3', 'lpjoint_qmin_ratio_Haukeli_G1', 'lpjoint_power_fraction_Haukeli_G1', 'lpjoint_economic_margin_Haukeli_G1', 'lpjoint_qmin_ratio_Songa_G1', 'lpjoint_power_fraction_Songa_G1', 'lpjoint_economic_margin_Songa_G1', 'lpjoint_qmin_ratio_Kjela_G1', 'lpjoint_power_fraction_Kjela_G1', 'lpjoint_economic_margin_Kjela_G1', 'lpjoint_qmin_ratio_Vesle Kjela_G1', 'lpjoint_power_fraction_Vesle Kjela_G1', 'lpjoint_economic_margin_Vesle Kjela_G1', 'lpjoint_plant_power_fraction_Hogga', 'lpjoint_storage_fill_Hogga', 'lpjoint_storage_min_slack_Hogga', 'lpjoint_storage_max_slack_Hogga', 'lpjoint_plant_power_fraction_Tokke', 'lpjoint_storage_fill_Tokke', 'lpjoint_storage_min_slack_Tokke', 'lpjoint_storage_max_slack_Tokke', 'lpjoint_plant_power_fraction_Lio', 'lpjoint_storage_fill_Lio', 'lpjoint_storage_min_slack_Lio', 'lpjoint_storage_max_slack_Lio', 'lpjoint_plant_power_fraction_Byrte', 'lpjoint_storage_fill_Byrte', 'lpjoint_storage_min_slack_Byrte', 'lpjoint_storage_max_slack_Byrte', 'lpjoint_plant_power_fraction_Vinje', 'lpjoint_storage_fill_Vinje', 'lpjoint_storage_min_slack_Vinje', 'lpjoint_storage_max_slack_Vinje', 'lpjoint_plant_power_fraction_Haukeli', 'lpjoint_storage_fill_Haukeli', 'lpjoint_storage_min_slack_Haukeli', 'lpjoint_storage_max_slack_Haukeli', 'lpjoint_plant_power_fraction_Songa', 'lpjoint_storage_fill_Songa', 'lpjoint_storage_min_slack_Songa', 'lpjoint_storage_max_slack_Songa', 'lpjoint_plant_power_fraction_Kjela', 'lpjoint_storage_fill_Kjela', 'lpjoint_storage_min_slack_Kjela', 'lpjoint_storage_max_slack_Kjela', 'lpjoint_plant_power_fraction_Vesle Kjela', 'lpjoint_storage_fill_Vesle Kjela', 'lpjoint_storage_min_slack_Vesle Kjela', 'lpjoint_storage_max_slack_Vesle Kjela', 'lpjoint_on_fraction_Tokke_G1', 'lpjoint_on_fraction_Tokke_G2', 'lpjoint_on_fraction_Tokke_G3', 'lpjoint_on_fraction_Tokke_G4', 'lpjoint_on_fraction_Vinje_G1', 'lpjoint_on_fraction_Vinje_G2', 'lpjoint_on_fraction_Vinje_G3', 'lpjoint_active_unit_count_Tokke', 'lpjoint_active_unit_count_Vinje', 'shop_state_Hogga_G1', 'shop_delta_qmin_ratio_Hogga_G1', 'shop_delta_power_fraction_Hogga_G1', 'shop_delta_economic_margin_Hogga_G1', 'shop_state_Tokke_G1', 'shop_delta_qmin_ratio_Tokke_G1', 'shop_delta_power_fraction_Tokke_G1', 'shop_delta_economic_margin_Tokke_G1', 'shop_state_Tokke_G2', 'shop_delta_qmin_ratio_Tokke_G2', 'shop_delta_power_fraction_Tokke_G2', 'shop_delta_economic_margin_Tokke_G2', 'shop_state_Tokke_G3', 'shop_delta_qmin_ratio_Tokke_G3', 'shop_delta_power_fraction_Tokke_G3', 'shop_delta_economic_margin_Tokke_G3', 'shop_state_Tokke_G4', 'shop_delta_qmin_ratio_Tokke_G4', 'shop_delta_power_fraction_Tokke_G4', 'shop_delta_economic_margin_Tokke_G4', 'shop_state_Lio_G1', 'shop_delta_qmin_ratio_Lio_G1', 'shop_delta_power_fraction_Lio_G1', 'shop_delta_economic_margin_Lio_G1', 'shop_state_Byrte_G1', 'shop_delta_qmin_ratio_Byrte_G1', 'shop_delta_power_fraction_Byrte_G1', 'shop_delta_economic_margin_Byrte_G1', 'shop_state_Vinje_G1', 'shop_delta_qmin_ratio_Vinje_G1', 'shop_delta_power_fraction_Vinje_G1', 'shop_delta_economic_margin_Vinje_G1', 'shop_state_Vinje_G2', 'shop_delta_qmin_ratio_Vinje_G2', 'shop_delta_power_fraction_Vinje_G2', 'shop_delta_economic_margin_Vinje_G2', 'shop_state_Vinje_G3', 'shop_delta_qmin_ratio_Vinje_G3', 'shop_delta_power_fraction_Vinje_G3', 'shop_delta_economic_margin_Vinje_G3', 'shop_state_Haukeli_G1', 'shop_delta_qmin_ratio_Haukeli_G1', 'shop_delta_power_fraction_Haukeli_G1', 'shop_delta_economic_margin_Haukeli_G1', 'shop_state_Songa_G1', 'shop_delta_qmin_ratio_Songa_G1', 'shop_delta_power_fraction_Songa_G1', 'shop_delta_economic_margin_Songa_G1', 'shop_state_Kjela_G1', 'shop_delta_qmin_ratio_Kjela_G1', 'shop_delta_power_fraction_Kjela_G1', 'shop_delta_economic_margin_Kjela_G1', 'shop_state_Vesle Kjela_G1', 'shop_delta_qmin_ratio_Vesle Kjela_G1', 'shop_delta_power_fraction_Vesle Kjela_G1', 'shop_delta_economic_margin_Vesle Kjela_G1'], 'lgb664': ['horizon_hour', 'horizon_day', 'hour_of_day', 'hour_sin', 'hour_cos', 'weekday', 'month', 'day_of_year', 'season_sin', 'season_cos', 'price_week_zscore', 'price_week_rank', 'volume_Kjelavatn', 'fill_Kjelavatn', 'price_over_water_Kjelavatn', 'headroom_after_inflow_Kjelavatn', 'volume_Staavatn', 'fill_Staavatn', 'price_over_water_Staavatn', 'headroom_after_inflow_Staavatn', 'volume_Langesae', 'fill_Langesae', 'price_over_water_Langesae', 'headroom_after_inflow_Langesae', 'volume_Foersvatn', 'fill_Foersvatn', 'price_over_water_Foersvatn', 'headroom_after_inflow_Foersvatn', 'volume_Venemo', 'fill_Venemo', 'price_over_water_Venemo', 'headroom_after_inflow_Venemo', 'volume_Botnedalsvatn', 'fill_Botnedalsvatn', 'price_over_water_Botnedalsvatn', 'headroom_after_inflow_Botnedalsvatn', 'volume_Bitdalsvatn', 'fill_Bitdalsvatn', 'price_over_water_Bitdalsvatn', 'headroom_after_inflow_Bitdalsvatn', 'volume_Vaamarvatn', 'fill_Vaamarvatn', 'price_over_water_Vaamarvatn', 'headroom_after_inflow_Vaamarvatn', 'volume_Langeidvatn', 'fill_Langeidvatn', 'price_over_water_Langeidvatn', 'headroom_after_inflow_Langeidvatn', 'volume_Totak', 'fill_Totak', 'price_over_water_Totak', 'headroom_after_inflow_Totak', 'volume_Songavatn', 'fill_Songavatn', 'price_over_water_Songavatn', 'headroom_after_inflow_Songavatn', 'volume_Vinjevatn', 'fill_Vinjevatn', 'price_over_water_Vinjevatn', 'headroom_after_inflow_Vinjevatn', 'volume_Bandak', 'fill_Bandak', 'price_over_water_Bandak', 'headroom_after_inflow_Bandak', 'volume_Bordalsvatn', 'fill_Bordalsvatn', 'price_over_water_Bordalsvatn', 'headroom_after_inflow_Bordalsvatn', 'volume_Byrtevatn', 'fill_Byrtevatn', 'price_over_water_Byrtevatn', 'headroom_after_inflow_Byrtevatn', 'volume_Hyljelihyl', 'fill_Hyljelihyl', 'price_over_water_Hyljelihyl', 'headroom_after_inflow_Hyljelihyl', 'volume_Vatjern', 'fill_Vatjern', 'price_over_water_Vatjern', 'headroom_after_inflow_Vatjern', 'inflow_Langesae', 'inflow_week_mean_Langesae', 'inflow_elapsed_mm3_Langesae', 'inflow_remaining_mm3_Langesae', 'inflow_Staavatn', 'inflow_week_mean_Staavatn', 'inflow_elapsed_mm3_Staavatn', 'inflow_remaining_mm3_Staavatn', 'inflow_Kjelavatn', 'inflow_week_mean_Kjelavatn', 'inflow_elapsed_mm3_Kjelavatn', 'inflow_remaining_mm3_Kjelavatn', 'inflow_Bordalsvatn', 'inflow_week_mean_Bordalsvatn', 'inflow_elapsed_mm3_Bordalsvatn', 'inflow_remaining_mm3_Bordalsvatn', 'inflow_Foersvatn', 'inflow_week_mean_Foersvatn', 'inflow_elapsed_mm3_Foersvatn', 'inflow_remaining_mm3_Foersvatn', 'inflow_Hyljelihyl', 'inflow_week_mean_Hyljelihyl', 'inflow_elapsed_mm3_Hyljelihyl', 'inflow_remaining_mm3_Hyljelihyl', 'inflow_Venemo', 'inflow_week_mean_Venemo', 'inflow_elapsed_mm3_Venemo', 'inflow_remaining_mm3_Venemo', 'inflow_Bitdalsvatn', 'inflow_week_mean_Bitdalsvatn', 'inflow_elapsed_mm3_Bitdalsvatn', 'inflow_remaining_mm3_Bitdalsvatn', 'inflow_Songavatn', 'inflow_week_mean_Songavatn', 'inflow_elapsed_mm3_Songavatn', 'inflow_remaining_mm3_Songavatn', 'inflow_Totak', 'inflow_week_mean_Totak', 'inflow_elapsed_mm3_Totak', 'inflow_remaining_mm3_Totak', 'inflow_Vaamarvatn', 'inflow_week_mean_Vaamarvatn', 'inflow_elapsed_mm3_Vaamarvatn', 'inflow_remaining_mm3_Vaamarvatn', 'inflow_Langeidvatn', 'inflow_week_mean_Langeidvatn', 'inflow_elapsed_mm3_Langeidvatn', 'inflow_remaining_mm3_Langeidvatn', 'inflow_Vatjern', 'inflow_week_mean_Vatjern', 'inflow_elapsed_mm3_Vatjern', 'inflow_remaining_mm3_Vatjern', 'inflow_Vinjevatn', 'inflow_week_mean_Vinjevatn', 'inflow_elapsed_mm3_Vinjevatn', 'inflow_remaining_mm3_Vinjevatn', 'inflow_Botnedalsvatn', 'inflow_week_mean_Botnedalsvatn', 'inflow_elapsed_mm3_Botnedalsvatn', 'inflow_remaining_mm3_Botnedalsvatn', 'inflow_Byrtevatn', 'inflow_week_mean_Byrtevatn', 'inflow_elapsed_mm3_Byrtevatn', 'inflow_remaining_mm3_Byrtevatn', 'inflow_Bandak', 'inflow_week_mean_Bandak', 'inflow_elapsed_mm3_Bandak', 'inflow_remaining_mm3_Bandak', 'inflow_r_Vest_Vassdraget', 'inflow_week_mean_r_Vest_Vassdraget', 'inflow_elapsed_mm3_r_Vest_Vassdraget', 'inflow_remaining_mm3_r_Vest_Vassdraget', 'min_volume_Totak', 'min_volume_week_max_Totak', 'min_volume_Byrtevatn', 'min_volume_week_max_Byrtevatn', 'min_volume_Staavatn', 'min_volume_week_max_Staavatn', 'min_flow_b_Kjelavatn', 'min_flow_week_max_b_Kjelavatn', 'min_flow_b_Hyljelihyl', 'min_flow_week_max_b_Hyljelihyl', 'min_flow_b_Vinjevatn', 'min_flow_week_max_b_Vinjevatn', 'min_flow_b_Byrtevatn', 'min_flow_week_max_b_Byrtevatn', 'min_flow_b_Bandak', 'min_flow_week_max_b_Bandak', 'min_flow_r_Vest_Vassdraget', 'min_flow_week_max_r_Vest_Vassdraget', 'physics_head_Hogga', 'physics_pool_fill_Hogga', 'physics_supply_fraction_Hogga', 'physics_catchment_inflow_fraction_Hogga', 'physics_dispatch_to_avoid_spill_Hogga', 'physics_profitable_remaining_fraction_Hogga', 'physics_head_Tokke', 'physics_pool_fill_Tokke', 'physics_supply_fraction_Tokke', 'physics_catchment_inflow_fraction_Tokke', 'physics_dispatch_to_avoid_spill_Tokke', 'physics_profitable_remaining_fraction_Tokke', 'physics_head_Lio', 'physics_pool_fill_Lio', 'physics_supply_fraction_Lio', 'physics_catchment_inflow_fraction_Lio', 'physics_dispatch_to_avoid_spill_Lio', 'physics_profitable_remaining_fraction_Lio', 'physics_head_Byrte', 'physics_pool_fill_Byrte', 'physics_supply_fraction_Byrte', 'physics_catchment_inflow_fraction_Byrte', 'physics_dispatch_to_avoid_spill_Byrte', 'physics_profitable_remaining_fraction_Byrte', 'physics_head_Vinje', 'physics_pool_fill_Vinje', 'physics_supply_fraction_Vinje', 'physics_catchment_inflow_fraction_Vinje', 'physics_dispatch_to_avoid_spill_Vinje', 'physics_profitable_remaining_fraction_Vinje', 'physics_head_Haukeli', 'physics_pool_fill_Haukeli', 'physics_supply_fraction_Haukeli', 'physics_catchment_inflow_fraction_Haukeli', 'physics_dispatch_to_avoid_spill_Haukeli', 'physics_profitable_remaining_fraction_Haukeli', 'physics_head_Songa', 'physics_pool_fill_Songa', 'physics_supply_fraction_Songa', 'physics_catchment_inflow_fraction_Songa', 'physics_dispatch_to_avoid_spill_Songa', 'physics_profitable_remaining_fraction_Songa', 'physics_head_Kjela', 'physics_pool_fill_Kjela', 'physics_supply_fraction_Kjela', 'physics_catchment_inflow_fraction_Kjela', 'physics_dispatch_to_avoid_spill_Kjela', 'physics_profitable_remaining_fraction_Kjela', 'physics_head_Vesle Kjela', 'physics_pool_fill_Vesle Kjela', 'physics_supply_fraction_Vesle Kjela', 'physics_catchment_inflow_fraction_Vesle Kjela', 'physics_dispatch_to_avoid_spill_Vesle Kjela', 'physics_profitable_remaining_fraction_Vesle Kjela', 'context_rank_remaining', 'lp_qmin_ratio_Hogga_G1', 'lp_power_fraction_Hogga_G1', 'lp_economic_margin_Hogga_G1', 'lp_qmin_ratio_Tokke_G1', 'lp_power_fraction_Tokke_G1', 'lp_economic_margin_Tokke_G1', 'lp_qmin_ratio_Tokke_G2', 'lp_power_fraction_Tokke_G2', 'lp_economic_margin_Tokke_G2', 'lp_qmin_ratio_Tokke_G3', 'lp_power_fraction_Tokke_G3', 'lp_economic_margin_Tokke_G3', 'lp_qmin_ratio_Tokke_G4', 'lp_power_fraction_Tokke_G4', 'lp_economic_margin_Tokke_G4', 'lp_qmin_ratio_Lio_G1', 'lp_power_fraction_Lio_G1', 'lp_economic_margin_Lio_G1', 'lp_qmin_ratio_Byrte_G1', 'lp_power_fraction_Byrte_G1', 'lp_economic_margin_Byrte_G1', 'lp_qmin_ratio_Vinje_G1', 'lp_power_fraction_Vinje_G1', 'lp_economic_margin_Vinje_G1', 'lp_qmin_ratio_Vinje_G2', 'lp_power_fraction_Vinje_G2', 'lp_economic_margin_Vinje_G2', 'lp_qmin_ratio_Vinje_G3', 'lp_power_fraction_Vinje_G3', 'lp_economic_margin_Vinje_G3', 'lp_qmin_ratio_Haukeli_G1', 'lp_power_fraction_Haukeli_G1', 'lp_economic_margin_Haukeli_G1', 'lp_qmin_ratio_Songa_G1', 'lp_power_fraction_Songa_G1', 'lp_economic_margin_Songa_G1', 'lp_qmin_ratio_Kjela_G1', 'lp_power_fraction_Kjela_G1', 'lp_economic_margin_Kjela_G1', 'lp_qmin_ratio_Vesle Kjela_G1', 'lp_power_fraction_Vesle Kjela_G1', 'lp_economic_margin_Vesle Kjela_G1', 'lp_plant_power_fraction_Hogga', 'lp_storage_fill_Hogga', 'lp_storage_min_slack_Hogga', 'lp_storage_max_slack_Hogga', 'lp_plant_power_fraction_Tokke', 'lp_storage_fill_Tokke', 'lp_storage_min_slack_Tokke', 'lp_storage_max_slack_Tokke', 'lp_plant_power_fraction_Lio', 'lp_storage_fill_Lio', 'lp_storage_min_slack_Lio', 'lp_storage_max_slack_Lio', 'lp_plant_power_fraction_Byrte', 'lp_storage_fill_Byrte', 'lp_storage_min_slack_Byrte', 'lp_storage_max_slack_Byrte', 'lp_plant_power_fraction_Vinje', 'lp_storage_fill_Vinje', 'lp_storage_min_slack_Vinje', 'lp_storage_max_slack_Vinje', 'lp_plant_power_fraction_Haukeli', 'lp_storage_fill_Haukeli', 'lp_storage_min_slack_Haukeli', 'lp_storage_max_slack_Haukeli', 'lp_plant_power_fraction_Songa', 'lp_storage_fill_Songa', 'lp_storage_min_slack_Songa', 'lp_storage_max_slack_Songa', 'lp_plant_power_fraction_Kjela', 'lp_storage_fill_Kjela', 'lp_storage_min_slack_Kjela', 'lp_storage_max_slack_Kjela', 'lp_plant_power_fraction_Vesle Kjela', 'lp_storage_fill_Vesle Kjela', 'lp_storage_min_slack_Vesle Kjela', 'lp_storage_max_slack_Vesle Kjela', 'lp_context_qmin_previous_1h_Hogga_G1', 'lp_context_qmin_next_1h_Hogga_G1', 'lp_context_qmin_previous_3h_Hogga_G1', 'lp_context_qmin_next_3h_Hogga_G1', 'lp_context_qmin_previous_6h_mean_Hogga_G1', 'lp_context_qmin_next_6h_mean_Hogga_G1', 'lp_context_qmin_week_mean_Hogga_G1', 'lp_context_qmin_week_at_least_one_fraction_Hogga_G1', 'lp_context_qmin_week_positive_fraction_Hogga_G1', 'lp_context_qmin_previous_1h_Tokke_G1', 'lp_context_qmin_next_1h_Tokke_G1', 'lp_context_qmin_previous_3h_Tokke_G1', 'lp_context_qmin_next_3h_Tokke_G1', 'lp_context_qmin_previous_6h_mean_Tokke_G1', 'lp_context_qmin_next_6h_mean_Tokke_G1', 'lp_context_qmin_week_mean_Tokke_G1', 'lp_context_qmin_week_at_least_one_fraction_Tokke_G1', 'lp_context_qmin_week_positive_fraction_Tokke_G1', 'lp_context_qmin_previous_1h_Tokke_G2', 'lp_context_qmin_next_1h_Tokke_G2', 'lp_context_qmin_previous_3h_Tokke_G2', 'lp_context_qmin_next_3h_Tokke_G2', 'lp_context_qmin_previous_6h_mean_Tokke_G2', 'lp_context_qmin_next_6h_mean_Tokke_G2', 'lp_context_qmin_week_mean_Tokke_G2', 'lp_context_qmin_week_at_least_one_fraction_Tokke_G2', 'lp_context_qmin_week_positive_fraction_Tokke_G2', 'lp_context_qmin_previous_1h_Tokke_G3', 'lp_context_qmin_next_1h_Tokke_G3', 'lp_context_qmin_previous_3h_Tokke_G3', 'lp_context_qmin_next_3h_Tokke_G3', 'lp_context_qmin_previous_6h_mean_Tokke_G3', 'lp_context_qmin_next_6h_mean_Tokke_G3', 'lp_context_qmin_week_mean_Tokke_G3', 'lp_context_qmin_week_at_least_one_fraction_Tokke_G3', 'lp_context_qmin_week_positive_fraction_Tokke_G3', 'lp_context_qmin_previous_1h_Tokke_G4', 'lp_context_qmin_next_1h_Tokke_G4', 'lp_context_qmin_previous_3h_Tokke_G4', 'lp_context_qmin_next_3h_Tokke_G4', 'lp_context_qmin_previous_6h_mean_Tokke_G4', 'lp_context_qmin_next_6h_mean_Tokke_G4', 'lp_context_qmin_week_mean_Tokke_G4', 'lp_context_qmin_week_at_least_one_fraction_Tokke_G4', 'lp_context_qmin_week_positive_fraction_Tokke_G4', 'lp_context_qmin_previous_1h_Lio_G1', 'lp_context_qmin_next_1h_Lio_G1', 'lp_context_qmin_previous_3h_Lio_G1', 'lp_context_qmin_next_3h_Lio_G1', 'lp_context_qmin_previous_6h_mean_Lio_G1', 'lp_context_qmin_next_6h_mean_Lio_G1', 'lp_context_qmin_week_mean_Lio_G1', 'lp_context_qmin_week_at_least_one_fraction_Lio_G1', 'lp_context_qmin_week_positive_fraction_Lio_G1', 'lp_context_qmin_previous_1h_Byrte_G1', 'lp_context_qmin_next_1h_Byrte_G1', 'lp_context_qmin_previous_3h_Byrte_G1', 'lp_context_qmin_next_3h_Byrte_G1', 'lp_context_qmin_previous_6h_mean_Byrte_G1', 'lp_context_qmin_next_6h_mean_Byrte_G1', 'lp_context_qmin_week_mean_Byrte_G1', 'lp_context_qmin_week_at_least_one_fraction_Byrte_G1', 'lp_context_qmin_week_positive_fraction_Byrte_G1', 'lp_context_qmin_previous_1h_Vinje_G1', 'lp_context_qmin_next_1h_Vinje_G1', 'lp_context_qmin_previous_3h_Vinje_G1', 'lp_context_qmin_next_3h_Vinje_G1', 'lp_context_qmin_previous_6h_mean_Vinje_G1', 'lp_context_qmin_next_6h_mean_Vinje_G1', 'lp_context_qmin_week_mean_Vinje_G1', 'lp_context_qmin_week_at_least_one_fraction_Vinje_G1', 'lp_context_qmin_week_positive_fraction_Vinje_G1', 'lp_context_qmin_previous_1h_Vinje_G2', 'lp_context_qmin_next_1h_Vinje_G2', 'lp_context_qmin_previous_3h_Vinje_G2', 'lp_context_qmin_next_3h_Vinje_G2', 'lp_context_qmin_previous_6h_mean_Vinje_G2', 'lp_context_qmin_next_6h_mean_Vinje_G2', 'lp_context_qmin_week_mean_Vinje_G2', 'lp_context_qmin_week_at_least_one_fraction_Vinje_G2', 'lp_context_qmin_week_positive_fraction_Vinje_G2', 'lp_context_qmin_previous_1h_Vinje_G3', 'lp_context_qmin_next_1h_Vinje_G3', 'lp_context_qmin_previous_3h_Vinje_G3', 'lp_context_qmin_next_3h_Vinje_G3', 'lp_context_qmin_previous_6h_mean_Vinje_G3', 'lp_context_qmin_next_6h_mean_Vinje_G3', 'lp_context_qmin_week_mean_Vinje_G3', 'lp_context_qmin_week_at_least_one_fraction_Vinje_G3', 'lp_context_qmin_week_positive_fraction_Vinje_G3', 'lp_context_qmin_previous_1h_Haukeli_G1', 'lp_context_qmin_next_1h_Haukeli_G1', 'lp_context_qmin_previous_3h_Haukeli_G1', 'lp_context_qmin_next_3h_Haukeli_G1', 'lp_context_qmin_previous_6h_mean_Haukeli_G1', 'lp_context_qmin_next_6h_mean_Haukeli_G1', 'lp_context_qmin_week_mean_Haukeli_G1', 'lp_context_qmin_week_at_least_one_fraction_Haukeli_G1', 'lp_context_qmin_week_positive_fraction_Haukeli_G1', 'lp_context_qmin_previous_1h_Songa_G1', 'lp_context_qmin_next_1h_Songa_G1', 'lp_context_qmin_previous_3h_Songa_G1', 'lp_context_qmin_next_3h_Songa_G1', 'lp_context_qmin_previous_6h_mean_Songa_G1', 'lp_context_qmin_next_6h_mean_Songa_G1', 'lp_context_qmin_week_mean_Songa_G1', 'lp_context_qmin_week_at_least_one_fraction_Songa_G1', 'lp_context_qmin_week_positive_fraction_Songa_G1', 'lp_context_qmin_previous_1h_Kjela_G1', 'lp_context_qmin_next_1h_Kjela_G1', 'lp_context_qmin_previous_3h_Kjela_G1', 'lp_context_qmin_next_3h_Kjela_G1', 'lp_context_qmin_previous_6h_mean_Kjela_G1', 'lp_context_qmin_next_6h_mean_Kjela_G1', 'lp_context_qmin_week_mean_Kjela_G1', 'lp_context_qmin_week_at_least_one_fraction_Kjela_G1', 'lp_context_qmin_week_positive_fraction_Kjela_G1', 'lp_context_qmin_previous_1h_Vesle Kjela_G1', 'lp_context_qmin_next_1h_Vesle Kjela_G1', 'lp_context_qmin_previous_3h_Vesle Kjela_G1', 'lp_context_qmin_next_3h_Vesle Kjela_G1', 'lp_context_qmin_previous_6h_mean_Vesle Kjela_G1', 'lp_context_qmin_next_6h_mean_Vesle Kjela_G1', 'lp_context_qmin_week_mean_Vesle Kjela_G1', 'lp_context_qmin_week_at_least_one_fraction_Vesle Kjela_G1', 'lp_context_qmin_week_positive_fraction_Vesle Kjela_G1', 'norm_price', 'norm_price_offset_-24', 'norm_price_offset_-12', 'norm_price_offset_-6', 'norm_price_offset_-3', 'norm_price_offset_-1', 'norm_price_offset_+1', 'norm_price_offset_+3', 'norm_price_offset_+6', 'norm_price_offset_+12', 'norm_price_offset_+24', 'norm_price_week_mean', 'norm_price_week_std', 'norm_price_week_min', 'norm_price_week_max', 'norm_price_week_q10', 'norm_price_week_q25', 'norm_price_week_q50', 'norm_price_week_q75', 'norm_price_week_q90', 'norm_price_minus_week_mean', 'norm_price_day_mean', 'norm_price_day_min', 'norm_price_day_max', 'norm_price_day_std', 'norm_price_remaining_mean', 'norm_price_remaining_max', 'norm_water_value_Kjelavatn', 'norm_price_minus_water_Kjelavatn', 'norm_water_value_Staavatn', 'norm_price_minus_water_Staavatn', 'norm_water_value_Langesae', 'norm_price_minus_water_Langesae', 'norm_water_value_Foersvatn', 'norm_price_minus_water_Foersvatn', 'norm_water_value_Venemo', 'norm_price_minus_water_Venemo', 'norm_water_value_Botnedalsvatn', 'norm_price_minus_water_Botnedalsvatn', 'norm_water_value_Bitdalsvatn', 'norm_price_minus_water_Bitdalsvatn', 'norm_water_value_Vaamarvatn', 'norm_price_minus_water_Vaamarvatn', 'norm_water_value_Langeidvatn', 'norm_price_minus_water_Langeidvatn', 'norm_water_value_Totak', 'norm_price_minus_water_Totak', 'norm_water_value_Songavatn', 'norm_price_minus_water_Songavatn', 'norm_water_value_Vinjevatn', 'norm_price_minus_water_Vinjevatn', 'norm_water_value_Bandak', 'norm_price_minus_water_Bandak', 'norm_water_value_Bordalsvatn', 'norm_price_minus_water_Bordalsvatn', 'norm_water_value_Byrtevatn', 'norm_price_minus_water_Byrtevatn', 'norm_water_value_Hyljelihyl', 'norm_price_minus_water_Hyljelihyl', 'norm_water_value_Vatjern', 'norm_price_minus_water_Vatjern', 'norm_physics_switch_margin_minimum_Hogga', 'norm_physics_switch_margin_maximum_Hogga', 'norm_physics_switch_margin_minimum_Tokke', 'norm_physics_switch_margin_maximum_Tokke', 'norm_physics_switch_margin_minimum_Lio', 'norm_physics_switch_margin_maximum_Lio', 'norm_physics_switch_margin_minimum_Byrte', 'norm_physics_switch_margin_maximum_Byrte', 'norm_physics_switch_margin_minimum_Vinje', 'norm_physics_switch_margin_maximum_Vinje', 'norm_physics_switch_margin_minimum_Haukeli', 'norm_physics_switch_margin_maximum_Haukeli', 'norm_physics_switch_margin_minimum_Songa', 'norm_physics_switch_margin_maximum_Songa', 'norm_physics_switch_margin_minimum_Kjela', 'norm_physics_switch_margin_maximum_Kjela', 'norm_physics_switch_margin_minimum_Vesle Kjela', 'norm_physics_switch_margin_maximum_Vesle Kjela', 'norm_price_previous_mean_6h', 'norm_price_minus_previous_mean_6h', 'norm_price_next_mean_6h', 'norm_price_minus_next_mean_6h', 'norm_price_previous_mean_12h', 'norm_price_minus_previous_mean_12h', 'norm_price_next_mean_12h', 'norm_price_minus_next_mean_12h', 'norm_price_previous_mean_24h', 'norm_price_minus_previous_mean_24h', 'norm_price_next_mean_24h', 'norm_price_minus_next_mean_24h', 'monetary_scale_inverse', 'lpjoint_qmin_ratio_Hogga_G1', 'lpjoint_power_fraction_Hogga_G1', 'lpjoint_economic_margin_Hogga_G1', 'lpjoint_qmin_ratio_Tokke_G1', 'lpjoint_power_fraction_Tokke_G1', 'lpjoint_economic_margin_Tokke_G1', 'lpjoint_qmin_ratio_Tokke_G2', 'lpjoint_power_fraction_Tokke_G2', 'lpjoint_economic_margin_Tokke_G2', 'lpjoint_qmin_ratio_Tokke_G3', 'lpjoint_power_fraction_Tokke_G3', 'lpjoint_economic_margin_Tokke_G3', 'lpjoint_qmin_ratio_Tokke_G4', 'lpjoint_power_fraction_Tokke_G4', 'lpjoint_economic_margin_Tokke_G4', 'lpjoint_qmin_ratio_Lio_G1', 'lpjoint_power_fraction_Lio_G1', 'lpjoint_economic_margin_Lio_G1', 'lpjoint_qmin_ratio_Byrte_G1', 'lpjoint_power_fraction_Byrte_G1', 'lpjoint_economic_margin_Byrte_G1', 'lpjoint_qmin_ratio_Vinje_G1', 'lpjoint_power_fraction_Vinje_G1', 'lpjoint_economic_margin_Vinje_G1', 'lpjoint_qmin_ratio_Vinje_G2', 'lpjoint_power_fraction_Vinje_G2', 'lpjoint_economic_margin_Vinje_G2', 'lpjoint_qmin_ratio_Vinje_G3', 'lpjoint_power_fraction_Vinje_G3', 'lpjoint_economic_margin_Vinje_G3', 'lpjoint_qmin_ratio_Haukeli_G1', 'lpjoint_power_fraction_Haukeli_G1', 'lpjoint_economic_margin_Haukeli_G1', 'lpjoint_qmin_ratio_Songa_G1', 'lpjoint_power_fraction_Songa_G1', 'lpjoint_economic_margin_Songa_G1', 'lpjoint_qmin_ratio_Kjela_G1', 'lpjoint_power_fraction_Kjela_G1', 'lpjoint_economic_margin_Kjela_G1', 'lpjoint_qmin_ratio_Vesle Kjela_G1', 'lpjoint_power_fraction_Vesle Kjela_G1', 'lpjoint_economic_margin_Vesle Kjela_G1', 'lpjoint_plant_power_fraction_Hogga', 'lpjoint_storage_fill_Hogga', 'lpjoint_storage_min_slack_Hogga', 'lpjoint_storage_max_slack_Hogga', 'lpjoint_plant_power_fraction_Tokke', 'lpjoint_storage_fill_Tokke', 'lpjoint_storage_min_slack_Tokke', 'lpjoint_storage_max_slack_Tokke', 'lpjoint_plant_power_fraction_Lio', 'lpjoint_storage_fill_Lio', 'lpjoint_storage_min_slack_Lio', 'lpjoint_storage_max_slack_Lio', 'lpjoint_plant_power_fraction_Byrte', 'lpjoint_storage_fill_Byrte', 'lpjoint_storage_min_slack_Byrte', 'lpjoint_storage_max_slack_Byrte', 'lpjoint_plant_power_fraction_Vinje', 'lpjoint_storage_fill_Vinje', 'lpjoint_storage_min_slack_Vinje', 'lpjoint_storage_max_slack_Vinje', 'lpjoint_plant_power_fraction_Haukeli', 'lpjoint_storage_fill_Haukeli', 'lpjoint_storage_min_slack_Haukeli', 'lpjoint_storage_max_slack_Haukeli', 'lpjoint_plant_power_fraction_Songa', 'lpjoint_storage_fill_Songa', 'lpjoint_storage_min_slack_Songa', 'lpjoint_storage_max_slack_Songa', 'lpjoint_plant_power_fraction_Kjela', 'lpjoint_storage_fill_Kjela', 'lpjoint_storage_min_slack_Kjela', 'lpjoint_storage_max_slack_Kjela', 'lpjoint_plant_power_fraction_Vesle Kjela', 'lpjoint_storage_fill_Vesle Kjela', 'lpjoint_storage_min_slack_Vesle Kjela', 'lpjoint_storage_max_slack_Vesle Kjela', 'lpjoint_on_fraction_Tokke_G1', 'lpjoint_on_fraction_Tokke_G2', 'lpjoint_on_fraction_Tokke_G3', 'lpjoint_on_fraction_Tokke_G4', 'lpjoint_on_fraction_Vinje_G1', 'lpjoint_on_fraction_Vinje_G2', 'lpjoint_on_fraction_Vinje_G3', 'lpjoint_active_unit_count_Tokke', 'lpjoint_active_unit_count_Vinje', 'shop_state_Hogga_G1', 'shop_delta_qmin_ratio_Hogga_G1', 'shop_delta_power_fraction_Hogga_G1', 'shop_delta_economic_margin_Hogga_G1', 'shop_state_Tokke_G1', 'shop_delta_qmin_ratio_Tokke_G1', 'shop_delta_power_fraction_Tokke_G1', 'shop_delta_economic_margin_Tokke_G1', 'shop_state_Tokke_G2', 'shop_delta_qmin_ratio_Tokke_G2', 'shop_delta_power_fraction_Tokke_G2', 'shop_delta_economic_margin_Tokke_G2', 'shop_state_Tokke_G3', 'shop_delta_qmin_ratio_Tokke_G3', 'shop_delta_power_fraction_Tokke_G3', 'shop_delta_economic_margin_Tokke_G3', 'shop_state_Tokke_G4', 'shop_delta_qmin_ratio_Tokke_G4', 'shop_delta_power_fraction_Tokke_G4', 'shop_delta_economic_margin_Tokke_G4', 'shop_state_Lio_G1', 'shop_delta_qmin_ratio_Lio_G1', 'shop_delta_power_fraction_Lio_G1', 'shop_delta_economic_margin_Lio_G1', 'shop_state_Byrte_G1', 'shop_delta_qmin_ratio_Byrte_G1', 'shop_delta_power_fraction_Byrte_G1', 'shop_delta_economic_margin_Byrte_G1', 'shop_state_Vinje_G1', 'shop_delta_qmin_ratio_Vinje_G1', 'shop_delta_power_fraction_Vinje_G1', 'shop_delta_economic_margin_Vinje_G1', 'shop_state_Vinje_G2', 'shop_delta_qmin_ratio_Vinje_G2', 'shop_delta_power_fraction_Vinje_G2', 'shop_delta_economic_margin_Vinje_G2', 'shop_state_Vinje_G3', 'shop_delta_qmin_ratio_Vinje_G3', 'shop_delta_power_fraction_Vinje_G3', 'shop_delta_economic_margin_Vinje_G3', 'shop_state_Haukeli_G1', 'shop_delta_qmin_ratio_Haukeli_G1', 'shop_delta_power_fraction_Haukeli_G1', 'shop_delta_economic_margin_Haukeli_G1', 'shop_state_Songa_G1', 'shop_delta_qmin_ratio_Songa_G1', 'shop_delta_power_fraction_Songa_G1', 'shop_delta_economic_margin_Songa_G1', 'shop_state_Kjela_G1', 'shop_delta_qmin_ratio_Kjela_G1', 'shop_delta_power_fraction_Kjela_G1', 'shop_delta_economic_margin_Kjela_G1', 'shop_state_Vesle Kjela_G1', 'shop_delta_qmin_ratio_Vesle Kjela_G1', 'shop_delta_power_fraction_Vesle Kjela_G1', 'shop_delta_economic_margin_Vesle Kjela_G1']}


## 5. Fit seven component groups and apply each Vinje projection

In [ ]:
predictions = {}
for name in ("lp", "context", "joint", "context_joint", "gpu608", "gpu664", "lgb664"):
    frame = feature_table(name)
    assert frame.columns.tolist() == EXPECTED_COLUMNS[name], f"Feature order differs: {name}"
    assert np.isfinite(frame.to_numpy()).all()
    xt, xv = frame.iloc[sample], frame.iloc[NTRAIN:]
    params = dict(COMPONENT_PARAMS[name])
    params["n_jobs"] = min(THREADS, 2) if name.startswith("gpu") or name == "lgb664" else min(THREADS, 3)
    if name.startswith("gpu"):
        params["device"] = XGB_DEVICE
    pred = np.empty((len(xv), 14), dtype=np.float32)
    for j, unit in enumerate(UNITS):
        model = xgb.XGBClassifier(**params) if name.startswith("gpu") else lgb.LGBMClassifier(**params)
        model.fit(xt, Y[sample, j])
        if name.startswith("gpu"):
            actual = json.loads(model.get_booster().save_config())["learner"]["generic_param"]["device"]
            assert actual.startswith("cuda"), "XGBoost fell back to CPU; stop instead of changing the recipe."
        pred[:, j] = model.predict_proba(xv)[:, 1]
        if name.startswith("gpu"):
            model.save_model(WORK / f"{name}_{unit}.ubj")
        else:
            model.booster_.save_model(str(WORK / f"{name}_{unit}.txt"))
        print(name, j+1, "/14", unit, flush=True)
        del model
    predictions[name] = pred.astype(np.float64)
    np.save(WORK / f"predictions_{name}.npy", pred)
    del frame, xt, xv, pred
    gc.collect()
anchor = project_vinje(sum(w * predictions[name] for name, w in
    {"lp":.140625, "context":.421875, "joint":.1875, "context_joint":.25}.items()))
for name, weight in (("gpu608", .5), ("gpu664", .25), ("lgb664", .1)):
    anchor = project_vinje((1-weight)*anchor + weight*predictions[name])
np.save(WORK / "anchor.npy", anchor)
del predictions


## 6. Fit startup16 seeds 4173 and 1, then average

In [ ]:
frame = feature_table("lgb664")
# startup16 was trained from the float32 rebuilt table, preserving that conversion.
frame = frame.astype(np.float32)
spec = feature_specification(frame.columns.tolist(), UNITS)
topology = yaml.safe_load((PKG / "data/extended/Tokke_Vinje_topology.yaml").read_text())["model"]["generator"]
ref = max(float(topology[u]["p_max"]) for u in UNITS)
constants = {u:{"pmin":float(topology[u]["p_min"]), "pmax":float(topology[u]["p_max"]),
    "start":float(topology[u]["startcost_const"]), "stop":float(topology[u]["stopcost_const"]), "ref":ref} for u in UNITS}
xt = build_rows(frame, sample, spec, constants)
xv = build_rows(frame, np.arange(NTRAIN,len(frame)), spec, constants)
yt = Y[sample].ravel()
assert xt.shape == (2749012,131) and xv.shape == (1705200,131)
# Release the full feature blocks before fitting the large pooled table.
del frame, base, lp_frame, joint_frame, shop_frame, Y, sample
gc.collect()
startup_predictions = []
for seed in STARTUP_SEEDS:
    params = dict(PARAMS, random_state=seed, n_jobs=THREADS,
        subsample=1.0 if seed == 4173 else .8, subsample_freq=0 if seed == 4173 else 1)
    model = lgb.LGBMClassifier(**params)
    model.fit(xt, yt, categorical_feature=[111,112])
    model.booster_.save_model(str(WORK / f"startup16_{seed}.txt"))
    startup_predictions.append(model.booster_.predict(xv, num_threads=THREADS).reshape(-1,14))
    print("Completed startup16 seed",seed,flush=True)
    del model
probabilities = project_vinje(.75*anchor + .25*np.mean(startup_predictions,axis=0))
del xt, xv, yt, anchor, startup_predictions, spec, constants, topology
gc.collect()


## 7. Apply Tokke projection, export and check

In [ ]:
probabilities = project_tokke(probabilities)


In [ ]:
template = pd.read_csv(PKG / "sample_submission.csv")
expected = [f"result_committed_{u}_t{h}" for u in UNITS for h in range(168)]
assert template.columns.tolist() == ["Run No"] + expected
assert mapping["Run No"].tolist() == template["Run No"].tolist()
assert probabilities.shape == (725*168,14)
assert np.isfinite(probabilities).all() and probabilities.min() >= 0 and probabilities.max() <= 1
wide = probabilities.reshape(725,168,14).transpose(0,2,1).reshape(725,-1)
result = pd.DataFrame(wide,columns=expected)
result.insert(0,"Run No",mapping["Run No"].to_numpy())
OUTPUT = Path.cwd() / OUTPUT_NAME
result.to_csv(OUTPUT,index=False,float_format="%.9g")
readback = pd.read_csv(OUTPUT)
assert readback.shape == (725,2353) and readback.columns.tolist() == template.columns.tolist()
assert readback["Run No"].tolist() == template["Run No"].tolist()
assert np.max(np.abs(readback.iloc[:,1:].to_numpy()-wide)) <= 5.000001e-10
actual_sha = file_sha256(OUTPUT)
receipt = {"file": str(OUTPUT), "sha256": actual_sha, "historical_sha256": HISTORICAL_SHA,
    "historical_byte_match": actual_sha == HISTORICAL_SHA, "elapsed_seconds":time.monotonic()-STARTED,
    "packages":package_versions,"python":sys.version,"platform":sys.platform,"xgboost_build":xgb.build_info(),
    "raw_hashes":raw_hashes,"seeds":STARTUP_SEEDS,"device":XGB_DEVICE,
    "catalogue_number":CATALOGUE_NUMBER,"threads":THREADS,"peak_memory_bytes":peak_memory_bytes(),
    "legacy_members":legacy_receipts if USE_LEGACY else [],
    "validation_scores_computed":False}
RECEIPT = OUTPUT.with_suffix(".receipt.json")
RECEIPT.write_text(json.dumps(receipt,indent=2))
print(json.dumps(receipt,indent=2))
if actual_sha != HISTORICAL_SHA:
    print("Numerical reproduction differs from the historical CSV. Score equivalence is not established by this run.")
assert receipt["elapsed_seconds"] < 12*3600, "Runtime exceeded the course's 12-hour limit."
assert receipt["peak_memory_bytes"] <= 32*1024**3, "Peak process memory exceeded 32 GiB."
assert actual_sha == HISTORICAL_SHA, "Historical byte match failed; inspect the saved receipt before using this output."
